# web item

In [ ]:
from __future__ import annotations

#UI
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fasthtml.common import *
from fasthtml.components import Uk_input_tag
from fasthtml.svg import *
from monsterui.all import *
import random
import math
import pandas as pd
import httpx
import re
import heapq
import json
import logging
from enum import Enum

import io, base64, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import networkx as nx
from itertools import combinations
from collections import deque

from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
import matplotlib.pyplot as plt
import string
from typing import NamedTuple
import uuid




In [ ]:
from HexMagic.geology import Geology, DrainageBasins, Watershed
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.game.flag import CountryFlag , PieceType, GameContext, DiagramGlyphs
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.overlay import OverlaySpec, FlowOverlay, CreamOverlay, RiverOverlay, OceanWaveOverlay
from HexMagic.overlay import ClimateOverlay, WatershedOverlay, TemperatureOverlay, PrecipitationOverlay,SoilOverlay
from HexMagic.climate import  TerraDemo, TerrainFactory
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.primitives import MapRect, MapPath,MapCord,  MapSize, HexRegion
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns

In [ ]:


@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve:  dict[int, float] = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list[float] = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    
    def summary(self):
        """Print tier distribution."""
        land = self.tiers[self.terrain.elevations > 0]
        print(f"Land hexes: {len(land)}")
        for tier in range(self.n_tiers):
            count = np.sum(land == tier)
            bar = '█' * (count // 10)
            print(f"  Tier {tier}: {count:5d} {bar}")


In [ ]:
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)


In [ ]:
@patch
def __ft__(self: FoodYield):
    """Distribution charts for food yield tuning."""
    t = self.terrain
    land = t.elevations > 0

    tiers     = self.tiers[land]
    elevs     = t.elevations[land]
    climate   = t.fields.get('climate', t.compute_climate())[land]

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

    # ── 1. Tier distribution bar chart ──
    ax = axes[0]
    counts = [np.sum(tiers == i) for i in range(self.n_tiers)]
    colors = plt.cm.YlGn(np.linspace(0.2, 0.9, self.n_tiers))
    bars = ax.bar(range(self.n_tiers), counts, color=colors, edgecolor='#555', linewidth=0.5)
    ax.set_title('Tier Distribution', fontweight='bold', fontsize=10)
    ax.set_xlabel('Tier'); ax.set_ylabel('Land hexes')
    ax.set_xticks(range(self.n_tiers))
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    str(count), ha='center', va='bottom', fontsize=7, color='#444')
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 2. Tier by elevation band ──
    ax = axes[1]
    elev_max = np.percentile(elevs, 98)
    n_bands = 6
    band_edges = np.linspace(0, elev_max, n_bands + 1)
    band_data  = []
    band_labels = []
    for lo, hi in zip(band_edges, band_edges[1:]):
        mask = (elevs >= lo) & (elevs < hi)
        band_data.append(tiers[mask].tolist() if mask.any() else [0])
        band_labels.append(f"{lo:.0f}–{hi:.0f}")

    bp = ax.boxplot(band_data, tick_labels=band_labels, patch_artist=True,
                    medianprops=dict(color='#222', linewidth=1.5),
                    whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch in bp['boxes']:
        patch.set_facecolor('#8D6E63'); patch.set_alpha(0.55)
    ax.set_title('Tier by Elevation Band', fontweight='bold', fontsize=10)
    ax.set_xlabel('Elevation'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 3. Tier by climate zone ──
    ax = axes[2]
    climate_names = {0:'Ocean/Ice', 1:'Tundra', 2:'Boreal',
                     3:'Temperate', 4:'Subtropical', 5:'Tropical'}
    zones_present = sorted(set(int(c) for c in climate))
    zone_data   = [tiers[climate == z].tolist() for z in zones_present]
    zone_labels = [climate_names.get(z, str(z)) for z in zones_present]
    zone_colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(zones_present)))

    bp2 = ax.boxplot(zone_data, tick_labels=zone_labels, patch_artist=True,
                     medianprops=dict(color='#222', linewidth=1.5),
                     whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch, color in zip(bp2['boxes'], zone_colors):
        patch.set_facecolor(color); patch.set_alpha(0.65)
    ax.set_title('Tier by Climate Zone', fontweight='bold', fontsize=10)
    ax.set_xlabel('Climate'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    land_count = int(np.sum(land))
    fig.suptitle(
        f"FoodYield — {land_count} land hexes, {self.n_tiers} tiers",
        fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"temp_curve · soil_mult · {self.n_tiers} tiers",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


In [ ]:
class GameParts:

    def __init__(self, radius: int = 20, useKoreanMap: bool = False,size=300):
        if useKoreanMap:
            self.terr: TerraDemo = TerraDemo().japan_korea_map()
        else:
            world = TerrainFactory.create_world(
                bounds=MapRect(MapCord(0, 0), MapSize(size, size)),
                preset='temperate',
                name='Maiden Lane',
                radius=15,
                lon_span=10.0,
                num_plates=8,
                subdivisions=3,
                ocean_fraction=0.3,
                oceanic_sides=['N'],
                terrain_age='young',
                formation_type='ridge',
                elevation_scale=1.5,
                erosion_age=0.1,
                num_lakes=0,
                seed=23,
                debug=True
            )

            self.terr = world.terrain
        self.grid: HexGrid = self.terr.hexGrid
        self.referenceIndex: int = self.grid.middle
        self.grid.adjustRadius(radius)
        rivers = self.terr.carve_to_ocean(num_lakes=0)
        self.builder: SVGBuilder = self.grid.builder
        self.basins: DrainageBasins = world.basins
        for i in range(len(self.grid.hexes)):
            self.grid.hexes[i].label = str(i)

        fy: FoodYield = FoodYield(self.terr, self.basins)
        
        self.yields: np.ndarray = fy.compute()
        self.fy: FoodYield = fy


In [ ]:

@patch
def hp2i(self: GameParts, hexpos: HexPosition) -> int:
    return self.grid.hexposition_to_index(hexpos=hexpos, origin_index=self.referenceIndex)

@patch
def i2hp(self: GameParts, index: int) -> HexPosition:
   return self.grid.index_to_hexposition(index=index, origin_index=self.referenceIndex)
        

In [ ]:
@patch
def overlayContext(parts: GameParts, *, region=None, padding=1, radius=None,
                   corridors=None, extra_squads=None, extra_pieces=None):
    """Build a GameContext from a GameParts instance."""
    terrain = parts.terr
    if region is not None:
        terrain = terrain.zoom(region, padding=padding)

    grid = terrain.hexGrid
    if radius:
        grid.adjustRadius(radius)

    squads = list(getattr(parts, 'squads', []))
    if extra_squads:
        squads.extend(extra_squads)

    pieces = []
    for sq in squads:
        pieces.extend([p for p in sq.pieces if p.place is not None])
    if extra_pieces:
        pieces.extend(extra_pieces)

    return GameContext(
        terrain=terrain, grid=grid, builder=grid.builder,
        c2f=getattr(terrain, 'c2f', None),
        extras=dict(
            basins=parts.basins,
            coarse_basins=parts.basins,
            corridors=corridors or [],
            pieces=pieces,
            squads=squads,
        )
    )


In [ ]:
showDemo = True

In [ ]:
showDemo = False

In [ ]:
myStuff = GameParts(radius=30)

In [ ]:
ctx = myStuff.overlayContext()
if showDemo:
    TerrainDisplay(
        CreamOverlay(stylized=True),
        RiverOverlay(max_width=4),
        FoodOverlay(),
        OceanWaveOverlay(
        num_waves=5,
        spacing=8,
        amplitude_start=3.0,
        amplitude_decay=0.6,
        wavelength=40,
        stroke_color="#4a7fb5",
        opacity_start=0.5,
        opacity_decay=0.7,
        stroke_width=1.2),
        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
ctx = myStuff.overlayContext()
myStuff.builder.layers = []
if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        

        ctx=ctx,
        debug = not showDemo
    )

## MapPlace

In [ ]:
@dataclass
class MapPlace:
    location: HexPosition = HexPosition.origin()
    facing: HexPosition = HexPosition.W
    level: int = 0
    harvest: int = 0
    parent: GameParts | None = None
    tenure: int = 0  # turns spent at this location
    pantry: int = 0  # per-piece food storage


In [ ]:
@patch
def location(self:GameParts,location, facing = HexPosition.W):
    index = self.hp2i(location)
    retPlace=MapPlace(location,facing)
    retPlace.level = self.terr.elevationLevel(index)
    retPlace.harvest  = int(self.fy.tiers[index])
    retPlace.parent = self
    return retPlace

In [ ]:
@patch
def location_by_index(self:GameParts,index, facing = HexPosition.W):
    location = self.i2hp(index)
    return self.location(location,facing)

In [ ]:
myStuff.grid.middle, myStuff.grid.index_to_hexposition(myStuff.grid.middle), myStuff.i2hp(myStuff.grid.middle)

In [ ]:
place = myStuff.location_by_index(336)
place.level, place.harvest

In [ ]:
@patch
def sight(self:MapPlace) -> HexRegion:
    """All hex indices visible from this piece's location."""
  
    effective_sight = max(2,self.level)
      
    retRegion = HexRegion(
        hexes=set([self.parent.hp2i(hp) for hp in self.location.field_of_view( self.facing, effective_sight)]),
        hexGrid = self.parent.grid)

    return retRegion


In [ ]:
place.sight()

In [ ]:
place = myStuff.location_by_index(336)
place.level, place.sight()

In [ ]:
place = myStuff.location_by_index(273)
place.level, place.harvest

## Pieces

In [ ]:
LIFESPAN: dict[PieceType, int] = {
    PieceType.PAWN:   18,
    PieceType.BISHOP: 12,
    PieceType.KNIGHT: 3,
    PieceType.QUEEN:  24,
    PieceType.KING:   999,
}

# after the strategy
LIFESPAN[PieceType.PAWN]   = 36
LIFESPAN[PieceType.BISHOP] = 5
LIFESPAN[PieceType.KNIGHT] = 5
LIFESPAN[PieceType.QUEEN]  = 20
LIFESPAN[PieceType.KING]   = 999


In [ ]:
@dataclass
class Piece:
    id:str = ""
    flag:CountryFlag | None = None
    type:PieceType = PieceType.PAWN
    place:MapPlace | None = None
    timeout:int = 0
    name:str = ""
    year:int = 1900
    lantern = StyleCSS("blank",fill="#007",opacity=0.3)

In [ ]:
@patch
def __eq__(self: Piece, other) -> bool:
    if not isinstance(other, Piece): return False
    return self.id == other.id

@patch
def __hash__(self: Piece) -> int:
    return hash(self.id)


In [ ]:
@patch
def visible_allies(self: Piece, squad: 'Squad') -> list['Piece']:
    """Allies in this piece's sight cone, sorted by birth year then name."""
    if self.place is None:
        return []
    my_cone = self.sight().hexes
    parts = self.place.parent
    allies = []
    for p in squad.pieces:
        if p is self or p.place is None:
            continue
        idx = parts.hp2i(p.place.location)
        if idx in my_cone:
            allies.append(p)
    # Oldest first (lowest year), then alphabetically
    allies.sort(key=lambda p: (p.year, p.name))
    return allies


In [ ]:


@patch
def fetch_avatar(self: Piece, style: str = "avataaars") -> str:
    """Fetch and cache DiceBear SVG avatar using piece name as seed."""
    if not hasattr(self, '_avatar_svg') or self._avatar_svg is None:
        seed = self.name or f"piece_{id(self)}"
        resp = httpx.get(f"https://api.dicebear.com/9.x/{style}/svg?seed={seed}")
        resp.raise_for_status()
        self._avatar_svg = resp.text
    return self._avatar_svg

@patch
def avatar_id(self: Piece) -> str:
    return f"avatar_{self.name or id(self)}"

@patch
def register_avatar(self: Piece, builder: SVGBuilder, style: str = "avataaars"):
    """Register avatar as a <symbol> definition on the builder (idempotent)."""
    aid = self.avatar_id()
    # Skip if already registered
    if any(getattr(d, 'attributes', {}).get('id') == aid for d in builder.definitions):
        return
    svg_raw = self.fetch_avatar(style)
    # Extract viewBox from the SVG
    
    vb_match = re.search(r'viewBox="([^"]+)"', svg_raw)
    vb = vb_match.group(1) if vb_match else "0 0 280 280"
    # Strip outer <svg> tags to get inner content
    inner = re.sub(r'<svg[^>]*>', '', svg_raw)
    inner = re.sub(r'</svg>\s*$', '', inner).strip()
    symbol = SVGDef("symbol", aid, inner, viewBox=vb)
    builder.add_definition(symbol)

@patch
def draw_avatar(self: Piece, center: MapCord, builder: SVGBuilder,
                size: float = 40, layer: str = None,
                opacity: float = 1.0, attrs: dict = None) -> str:
    """Render avatar at center with given size. Registers symbol if needed."""
    self.register_avatar(builder)
    aid = self.avatar_id()
    x, y = center.x - size / 2, center.y - size / 2
    extra = CountryFlag._render_attrs(attrs) if attrs else ""
    svg = (f'<use href="#{aid}" x="{x:.1f}" y="{y:.1f}" '
           f'width="{size:.1f}" height="{size:.1f}"{extra}/>')
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer:
        builder.adjust(layer, svg)
    return svg


In [ ]:
class FoodStats(NamedTuple):
    """Per-piece-type food economics."""
    diet: int        # upkeep per turn while on board
    cost: int        # one-time cost to place
    harvest: float   # multiplier applied to tile's food yield tier
    drain: float     # cost imposed on EACH enemy piece in sight cone
    bonus_sight: int # extra sight range beyond elevation-based default

FOOD_TABLE: dict[PieceType, FoodStats] = {
    PieceType.PAWN:   FoodStats(diet=2, cost=7,  harvest=1.25, drain=0.25, bonus_sight=0),
    PieceType.BISHOP: FoodStats(diet=3, cost=14, harvest=0.75, drain=1.5,  bonus_sight=2),
    PieceType.KNIGHT: FoodStats(diet=2, cost=6,  harvest=0.0,  drain=3.0,  bonus_sight=0),
    PieceType.QUEEN:  FoodStats(diet=5, cost=27, harvest=0.0,  drain=4.25, bonus_sight=0),
    PieceType.KING:   FoodStats(diet=2, cost=0,  harvest=1.0,  drain=0.5,  bonus_sight=1),
}

#This is after the new strategies to even them
FOOD_TABLE[PieceType.PAWN]   = FoodStats(diet=1, cost=3,  harvest=1.25, drain=1.0,  bonus_sight=0)
FOOD_TABLE[PieceType.BISHOP] = FoodStats(diet=3, cost=9,  harvest=1.0,  drain=0.75, bonus_sight=3)
FOOD_TABLE[PieceType.KNIGHT] = FoodStats(diet=2, cost=10, harvest=0.0,  drain=3.0,  bonus_sight=0)
FOOD_TABLE[PieceType.QUEEN]  = FoodStats(diet=4, cost=18, harvest=0.0,  drain=4.5,  bonus_sight=2)
FOOD_TABLE[PieceType.KING]   = FoodStats(diet=2, cost=0,  harvest=1.0,  drain=0.5,  bonus_sight=1)


@patch(as_prop=True)
def food(self: Piece) -> FoodStats:
    return FOOD_TABLE[self.type]


In [ ]:
@patch
def sight(self: Piece) -> HexRegion:
    """All hex indices visible from this piece's location."""
    bonus = self.food.bonus_sight
    effective_sight = max(1, self.place.level) + bonus
    
    return HexRegion(
        hexes=set([self.place.parent.hp2i(hp) 
                   for hp in self.place.location.field_of_view(self.place.facing, effective_sight)]),
        hexGrid=self.place.parent.grid)


## Squad

In [ ]:
@dataclass
class Squad:
    pieces: list[Piece] = field(default_factory=list)
    name: str = ""
    year: int = 1900
    animal: str = ""
    flag: CountryFlag | None = None
    countryName: str = ""
    storage:int = 100
    target:int = -1
    #alive:bool = True

    @classmethod
    def squads(cls, count: int = 4, flag: CountryFlag | None = None) -> list['Squad']:
        if flag is None:
            flag = CountryFlag.seaborn("husl", 1)[0]
        return [Squad(name=name, animal=animal, flag=flag) 
                for animal, name in flag.country_squads(count)]

In [ ]:
@patch
def make_squad(self: CountryFlag, piece_types: list[PieceType], year: int, seed: int = 0) -> Squad:
    """Create a Squad with animal name and age-appropriate named Pieces."""
    animal = self.animal_name()
    squad_nm = self.squad_name(animal, seed)
    
    rng = random.Random(hash((self.name, year, seed)))
    
    pieces = []
    for pt in piece_types:
        birth_year = year + rng.randint(20, 38)
        names = self.commonNames(year=birth_year)
        name = rng.choice(names)
        player = Piece(flag=self, type=pt, name=name, year=birth_year)
        player.id = player.name + "_" + uuid.uuid4().hex[:10]
        player.lantern = StyleCSS(f"{player.id}_lantern", fill=self.primary, opacity=0.3)
        pieces.append(player)

    
    return Squad(pieces=pieces, name=squad_nm, year=year, animal=animal, flag=self)


In [ ]:
@patch
def loadSquads(self:GameParts, year: int=1880, seed: int = 0,levels=3):
    self.flags = CountryFlag.seaborn("bright",levels=levels)
    self.squads = []
    for i,flag in enumerate(self.flags):
        roster = {
            PieceType.PAWN: 3, 
            #PieceType.BISHOP: 2, 
            #PieceType.QUEEN: 2, 
            PieceType.KING: 1, 
            #PieceType.KNIGHT: 4
        }
        pieceKinds = [pt for pt, n in roster.items() for _ in range(n)]


        squad = flag.make_squad(pieceKinds, year=year, seed =seed) 
        self.squads.append(squad)

In [ ]:
@patch
def kill(self: Squad):
    """Remove all pieces from the board. Called when king dies."""
    for p in self.pieces:
        p.place = None


In [ ]:
myStuff.loadSquads()

In [ ]:
myStuff.squads[0].pieces[0]

from fasthtml.common import *
from fasthtml.components import Uk_input_tag
from fasthtml.svg import *
from monsterui.all import *

In [ ]:
#read_url("https://monsterui.answer.ai/cards/md")

In [ ]:

@patch
def __ft__(self: Piece):
    b = SVGBuilder()
    b.width, b.height = 60, 60
    self.draw_avatar(MapCord(30, 30), b, size=50, layer="avatar")

    return Card(
        DivCentered(
            NotStr(b.xml()),
            P(self.name, cls=(TextT.sm, TextT.medium)),
            P(f"{self.type.icon} · {self.year}", cls=TextPresets.muted_sm)),
        cls=CardT.hover)


In [ ]:
myStuff.loadSquads()
show(myStuff.squads[0].pieces[0])

In [ ]:
#show(myStuff.squads[0].pieces[0])

In [ ]:
@patch
def __ft__(self: Squad):
    # Header: flag banner + squad name + animal
    flag_builder = self.flag.flag_banner(width=210, height=120, wavy=True)

    # Animal icon
    ab = SVGBuilder()
    ab.width, ab.height = 250,250
    ab.adjust("animal", self.flag.animal_svg(
        size='large', animal=self.animal,
        center=MapCord(125, 125),
        piece_id=f"sq_{self.name}"))


    header = DivFullySpaced(
       # NotStr(flag_builder.xml()),
        H3(self.name),
        NotStr(ab.xml()),
        

        cls=TextPresets.muted_sm)

    # Group pieces by type
    from collections import defaultdict
    by_type = defaultdict(list)
    for p in self.pieces:
        by_type[p.type].append(p)

    sections = []
    for pt, pieces in by_type.items():
        # Section header with piece icon
        sec_header = DivLAligned(
            Span(pt.icon, cls=TextT.lg),
            H4(pt.value.title()),
            P(f"× {len(pieces)}", cls=TextPresets.muted_sm))
        # 4-wide grid of piece cards
        grid = Grid(*[p.__ft__() for p in pieces],
                    cols_sm=2, cols_md=4)
        sections.append(Div(sec_header, grid, cls='space-y-2'))

    return DivCentered(header,Card( *sections, cls='space-y-4'))


In [ ]:
@patch
def table(self: Squad):
    rows = []
    for p in self.pieces:
        loc = ""
        facing = ""
        if p.place and p.place.parent:
            loc = str(p.place.parent.hp2i(p.place.location))
            facing = str(p.place.facing.label)
        rows.append(Tr(
            Td(p.type.icon),
            Td(p.name, cls=TextT.medium),
            Td(str(p.year), cls=TextPresets.muted_sm),
            Td(loc, cls=TextPresets.muted_sm),
            Td(facing, cls=TextPresets.muted_sm),
        ))
    
    return Card(
        Table(
            Thead(Tr(Th(""), Th("Name"), Th("Born"), Th("Loc"), Th("Facing"))),
            Tbody(*rows),
        ),
        header=(H4(self.name), P(f"🐾 {self.animal} · {len(self.pieces)} pieces", cls=TextPresets.muted_sm)),
    )


In [ ]:
myStuff.loadSquads()
if showDemo:
    show(myStuff.squads[0])
else:
    show(myStuff.squads[0].table())

In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
show(unit)

In [ ]:
unit.place.sight()

In [ ]:
unit.sight()

In [ ]:
class TargetType(Enum):
    KING = "king"
    OASIS = "oasis"
    CONSOLIDATE = "consolidate"


## Overlays

In [ ]:
def FogOverlay( fill="#ffffff", stroke="#cccccc", **kw) -> OverlaySpec:
    """White out hexes with no terrain data."""
    def render(ctx: OverlayContext) -> str:
        c2f = getattr(ctx, 'c2f', None)
        if not c2f: return ""
        mapped = {ni for indices in c2f.values() for ni in (indices if isinstance(indices, list) else [indices])}
        fog = StyleCSS("fog", fill=fill, stroke=stroke, stroke_width=1)
        ctx.builder.add_style(fog)
        for i in range(len(ctx.grid.hexes)):
            if i not in mapped:
                ctx.grid.hexes[i].style = fog
        return ""
    return OverlaySpec("fog", render, priority=6)


In [ ]:
sight = unit.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
TerrainDisplay(
    TerrainOverlay(),
    RiverOverlay(max_width=4),
    FogOverlay(),
    
    

    ctx=ctx,
    debug = not showDemo
)

In [ ]:
sight = unit.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(fill="purple"),
        

        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
def LanternOverlay(**kw) -> OverlaySpec:
    """Render sight regions for placed pieces using their lantern style, plus facing bars."""
    def render(ctx) -> str:
        grid, builder = ctx.grid, ctx.builder
        N = len(grid.hexes)
        r = grid.radius
        parts = []

        for piece in ctx.pieces:
            if piece.place is None: continue
            builder.add_style(piece.lantern)

            # Sight region
            for idx in sorted(piece.sight().hexes):
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        h = grid.hexes[fi]
                        parts.append(Hex(h.radius, h.center, piece.lantern, v=h.v).svg())

            # Facing bar at piece's location
            loc_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(loc_idx):
                if 0 <= fi < N:
                    c = grid.hexes[fi].center
                    glyphs = DiagramGlyphs(piece.flag, size=r * 0.7)
                    glyphs.register_styles(builder)
                    dirs = HexPosition.directions()
                    facing_dir = dirs.index(piece.place.facing) if piece.place.facing in dirs else 0
                    parts.append(glyphs.facing_bar(c, facing_dir))

        return '\n'.join(parts)

    return OverlaySpec("lanterns", render, requires={'pieces'}, priority=62)


In [ ]:
def PieceOverlay(**kw) -> OverlaySpec:
    """Render all pieces using their flag's chess-piece SVG."""
    def render(ctx) -> str:
        if not ctx.pieces:
            return ""

        grid = ctx.grid
        N = len(grid.hexes)
        scale = grid.radius * 0.8 / 22.5
        parts = []

        for i, piece in enumerate(ctx.pieces):
            if piece.place is None or piece.flag is None:
                continue

            coarse_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(coarse_idx):
                if 0 <= fi < N:
                    center = grid.hexes[fi].center
                    pid = f"pc_{piece.id}_{i}"

                    svg_str, pat_def = piece.flag.piece_svg(
                        piece.type,
                        MapCord(center.x, center.y),
                        scale=scale,
                        size='board',
                        piece_id=pid,
                    )

                    if pat_def is not None:
                        ctx.builder.add_definition(pat_def)

                    parts.append(svg_str)

        return '\n'.join(parts)

    return OverlaySpec("pieces", render, requires={'pieces'}, priority=80)


In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
sight = unit.place.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []

if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(),
        LanternOverlay(),
        PieceOverlay(),
        pieces=[unit],

        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
unit.place.facing = HexPosition.SE
sight = unit.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(),
        LanternOverlay(),
        PieceOverlay(),
        pieces=[unit],
        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
if showDemo:
    show(myStuff.squads[0])
else:
    show(myStuff.squads[0].table())

In [ ]:
def SquadNamesOverlay(font_size: int = 14, font_family: str = "Cinzel", **kw) -> OverlaySpec:
    """Render squad names at the centroid of their placed pieces."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts = []
        
        for squad in ctx.squads:
            if not squad.name:
                continue
            
            # Collect fine-grid indices for all placed pieces
            placed_fine = []
            for piece in squad.pieces:
                if piece.place is None:
                    continue
                coarse_idx = piece.place.parent.hp2i(piece.place.location)
                for fi in ctx.fine_indices(coarse_idx):
                    if 0 <= fi < N:
                        placed_fine.append(fi)
            
            if not placed_fine:
                continue
            
            # Centroid of placed piece centers
            cx = sum(grid.hexes[fi].center.x for fi in placed_fine) / len(placed_fine)
            cy = sum(grid.hexes[fi].center.y for fi in placed_fine) / len(placed_fine)
            
            # Label style from flag
            label_style = squad.flag.labelStyle(f"sqname_{squad.name.replace(' ', '_')}")
            ctx.builder.add_style(label_style)
            ctx.builder.add_font(font_family)
            
            # Offset below centroid so it doesn't sit on top of pieces
            label_y = cy + grid.radius * 1.8
            
            parts.append(
                f'<text x="{cx:.1f}" y="{label_y:.1f}" text-anchor="middle" '
                f'font-size="{font_size}" font-family="\'{font_family}\', sans-serif" '
                f'dominant-baseline="middle" class="{label_style.name}">'
                f'{squad.name}</text>'
            )
        
        return '\n'.join(parts)
    
    return OverlaySpec("squad_names", render, requires={'squads'}, priority=90)


## Moves

In [ ]:
class MoveError(Exception):
    """Raised when a Move fails validation."""
    pass

In [ ]:
PURCHASE_COST: dict[PieceType, int] = {
    PieceType.PAWN:   3,
    PieceType.BISHOP: 9,
    PieceType.KNIGHT: 10,
    PieceType.QUEEN:  18,
}

class Purchase(NamedTuple):
    piece_type: PieceType
    location: MapPlace
    facing: HexPosition


In [ ]:
class Rotation(NamedTuple):
    piece: Piece
    facing: HexPosition
class Placement(NamedTuple):
    piece: Piece
    location: MapPlace
    
@dataclass
class Move:
    rotations: list[Rotation] = field(default_factory=list)
    placement: Placement | None = None
    purchase: Purchase | None = None

    # Add occupied-hex check to Move validation
    @classmethod
    def from_squad(cls, squad: Squad, rotations: list[Rotation] = None, 
                placement: Placement | None = None) -> 'Move':
        rotations = rotations or []
        errors = []
        
        for r in rotations:
            if r.piece not in squad.pieces:
                errors.append(f"{r.piece.name}: not in squad '{squad.name}'")
            elif r.piece.place is None:
                errors.append(f"{r.piece.name}: cannot rotate — not yet placed")
        
        if placement is not None:
            if placement.piece not in squad.pieces:
                errors.append(f"{placement.piece.name}: not in squad '{squad.name}'")
            elif placement.piece.place is not None:
                errors.append(f"{placement.piece.name}: cannot place — already on the board")
        
        if placement is not None:
            rotated_pieces = {r.piece for r in rotations}
            if placement.piece in rotated_pieces:
                errors.append(f"{placement.piece.name}: appears in both rotation and placement")
        
        placed_allies = [p for p in squad.pieces if p.place is not None]
        if placement is not None and not placed_allies:
            if placement.piece.type != PieceType.KING:
                errors.append(f"{placement.piece.name}: first piece placed must be the King")
        
        # Sight check — use POST-rotation facing for rotated pieces
        if placement is not None and placement.piece in squad.pieces:
            if placed_allies:
                target_idx = placement.location.parent.hp2i(placement.location.location)
                
                # Build rotation lookup: piece -> new_facing
                rot_map = {r.piece: r.facing for r in rotations}
                
                visible = set()
                for ally in placed_allies:
                    facing = rot_map.get(ally, ally.place.facing)
                    bonus = ally.food.bonus_sight
                    effective_sight = max(2, ally.place.level) + bonus
                    sight_hexes = set(
                        placement.location.parent.hp2i(hp)
                        for hp in ally.place.location.field_of_view(facing, effective_sight)
                    )
                    visible |= sight_hexes
                
                if target_idx not in visible:
                    errors.append(f"{placement.piece.name}: target hex {target_idx} not in any ally's sight")
        
        # Check hex not already occupied
        if placement is not None and placement.location.parent is not None:
            target_idx = placement.location.parent.hp2i(placement.location.location)
            parts = placement.location.parent
            for sq in parts.squads:
                for p in sq.pieces:
                    if p.place is not None:
                        if parts.hp2i(p.place.location) == target_idx:
                            errors.append(f"{placement.piece.name}: hex {target_idx} already occupied by {p.name}")
        
        if errors:
            raise MoveError("; ".join(errors))
        
        return Move(rotations=rotations, placement=placement)


    #Move.from_squad = from_squad




In [ ]:


@dataclass
class Move:
    rotations: list[Rotation] = field(default_factory=list)
    placement: Placement | None = None
    purchase: Purchase | None = None

    @classmethod
    def from_squad(cls, squad: Squad, rotations: list[Rotation] = None,
                   placement: Placement | None = None,
                   purchase: Purchase | None = None) -> 'Move':
        rotations = rotations or []
        errors = []

        # ── Rotation validation ──
        for r in rotations:
            if r.piece not in squad.pieces:
                errors.append(f"{r.piece.name}: not in squad '{squad.name}'")
            elif r.piece.place is None:
                errors.append(f"{r.piece.name}: cannot rotate — not yet placed")

        # ── Placement validation ──
        if placement is not None:
            if placement.piece not in squad.pieces:
                errors.append(f"{placement.piece.name}: not in squad '{squad.name}'")
            elif placement.piece.place is not None:
                errors.append(f"{placement.piece.name}: cannot place — already on the board")

        if placement is not None:
            rotated_pieces = {r.piece for r in rotations}
            if placement.piece in rotated_pieces:
                errors.append(f"{placement.piece.name}: appears in both rotation and placement")

        placed_allies = [p for p in squad.pieces if p.place is not None]

        # First piece must be King
        if placement is not None and not placed_allies:
            if placement.piece.type != PieceType.KING:
                errors.append(f"{placement.piece.name}: first piece placed must be the King")

        # ── Purchase validation ──
        if purchase is not None:
            if purchase.piece_type == PieceType.KING:
                errors.append("cannot purchase a King")
            elif purchase.piece_type not in PURCHASE_COST:
                errors.append(f"no purchase cost defined for {purchase.piece_type.value}")
            else:
                # Find the squad's king
                kings = [p for p in squad.pieces if p.type == PieceType.KING and p.place is not None]
                if not kings:
                    errors.append("no placed King — cannot purchase")
                else:
                    king = kings[0]
                    cost = PURCHASE_COST[purchase.piece_type]
                    if king.place.pantry < cost:
                        errors.append(
                            f"King's pantry ({king.place.pantry}) < cost ({cost}) "
                            f"for {purchase.piece_type.value}")

            # Purchase target: land, unoccupied, in sight
            if purchase.location.parent is not None:
                parts = purchase.location.parent
                tgt = parts.hp2i(purchase.location.location)

                if parts.terr.elevations[tgt] <= 0:
                    errors.append(f"purchase hex {tgt} is ocean")

                for sq in parts.squads:
                    for p in sq.pieces:
                        if p.place is not None and parts.hp2i(p.place.location) == tgt:
                            errors.append(f"purchase hex {tgt} already occupied by {p.name}")

                # Sight check (post-rotation)
                rot_map = {r.piece: r.facing for r in rotations}
                visible = set()
                # Include the piece being placed this turn too
                check_allies = list(placed_allies)
                if placement is not None and placement.piece in squad.pieces:
                    check_allies_for_purchase = check_allies  # placed piece isn't on board yet...
                else:
                    check_allies_for_purchase = check_allies

                for ally in check_allies_for_purchase:
                    facing = rot_map.get(ally, ally.place.facing)
                    bonus = ally.food.bonus_sight
                    eff = max(2, ally.place.level) + bonus
                    sight_hexes = set(
                        parts.hp2i(hp)
                        for hp in ally.place.location.field_of_view(facing, eff)
                    )
                    visible |= sight_hexes

                if tgt not in visible:
                    errors.append(f"purchase hex {tgt} not in any ally's sight")

        # ── Placement sight + occupied check ──
        if placement is not None and placement.piece in squad.pieces:
            if placed_allies:
                parts = placement.location.parent
                target_idx = parts.hp2i(placement.location.location)

                rot_map = {r.piece: r.facing for r in rotations}
                visible = set()
                for ally in placed_allies:
                    facing = rot_map.get(ally, ally.place.facing)
                    bonus = ally.food.bonus_sight
                    eff = max(2, ally.place.level) + bonus
                    sight_hexes = set(
                        parts.hp2i(hp)
                        for hp in ally.place.location.field_of_view(facing, eff)
                    )
                    visible |= sight_hexes

                if target_idx not in visible:
                    errors.append(f"{placement.piece.name}: target hex {target_idx} not in any ally's sight")

            if placement.location.parent is not None:
                parts = placement.location.parent
                target_idx = parts.hp2i(placement.location.location)
                for sq in parts.squads:
                    for p in sq.pieces:
                        if p.place is not None and parts.hp2i(p.place.location) == target_idx:
                            errors.append(f"{placement.piece.name}: hex {target_idx} already occupied by {p.name}")

        if errors:
            raise MoveError("; ".join(errors))

        return Move(rotations=rotations, placement=placement, purchase=purchase)


In [ ]:
@patch
def apply(self: Move, parts: GameParts, squad: Squad = None):
    """Apply this move to the board."""
    for r in self.rotations:
        if r.piece.place is None:
            raise MoveError(f"{r.piece.name}: can't rotate — not placed")
        r.piece.place.facing = r.facing
    
    if self.placement is not None:
        self.placement.piece.place = self.placement.location

    if self.purchase is not None and squad is not None:
        kings = [p for p in squad.pieces if p.type == PieceType.KING and p.place is not None]
        if not kings:
            raise MoveError("no placed King — cannot purchase")
        king = kings[0]
        cost = PURCHASE_COST[self.purchase.piece_type]
        king.place.pantry -= cost

        # Create the new piece
        rng = random.Random()
        names = squad.flag.commonNames(year=squad.year + 20)
        name = rng.choice(names)
        new_piece = Piece(
            id=name + "_" + uuid.uuid4().hex[:10],
            flag=squad.flag,
            type=self.purchase.piece_type,
            name=name,
            year=squad.year,
        )
        new_piece.lantern = StyleCSS(f"{new_piece.id}_lantern", fill=squad.flag.primary, opacity=0.3)
        new_piece.place = self.purchase.location
        new_piece.place.facing = self.purchase.facing
        squad.pieces.append(new_piece)


need to modify apply so that it does purchases

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the updated `apply`:

```python
@patch
def apply(self: Move, parts: GameParts, squad: Squad = None):
    """Apply this move to the board."""
    for r in self.rotations:
        if r.piece.place is None:
            raise MoveError(f"{r.piece.name}: can't rotate — not placed")
        r.piece.place.facing = r.facing
    
    if self.placement is not None:
        self.placement.piece.place = self.placement.location

    if self.purchase is not None and squad is not None:
        kings = [p for p in squad.pieces if p.type == PieceType.KING and p.place is not None]
        if not kings:
            raise MoveError("no placed King — cannot purchase")
        king = kings[0]
        cost = PURCHASE_COST[self.purchase.piece_type]
        king.place.pantry -= cost

        # Create the new piece
        rng = random.Random()
        names = squad.flag.commonNames(year=squad.year + 20)
        name = rng.choice(names)
        new_piece = Piece(
            id=''.join(random.choices(string.ascii_letters, k=16)),
            flag=squad.flag,
            type=self.purchase.piece_type,
            name=name,
            year=squad.year,
        )
        new_piece.lantern = StyleCSS(f"{new_piece.id}_lantern", fill=squad.flag.primary, opacity=0.3)
        new_piece.place = self.purchase.location
        new_piece.place.facing = self.purchase.facing
        squad.pieces.append(new_piece)
```

Key changes:
- Added `squad` parameter so we can find the king and append the new piece
- Deducts cost from king's pantry
- Creates a new `Piece` with a random name from the flag's name pool, assigns it the purchased type, places it, and adds it to the squad's roster

In [ ]:
@dataclass
class MoveLog:
    parts: GameParts
    _df: pd.DataFrame = field(default_factory=lambda: pd.DataFrame(
        columns=['turn', 'squad', 'piece', 'action', 'detail']))
    
    def record(self, squad: Squad, move: Move):
        """Validate, apply, and log."""
        validated = Move.from_squad(squad, move.rotations, move.placement, move.purchase)
        validated.apply(self.parts, squad)
        turn = len(self._df)
        
        rows = []
        for r in validated.rotations:
            rows.append(dict(turn=turn, squad=squad.name, piece=r.piece.name,
                             action='rotate', detail=r.facing.label))
        if validated.placement:
            p = validated.placement
            idx = self.parts.hp2i(p.location.location)
            rows.append(dict(turn=turn, squad=squad.name, piece=p.piece.name,
                             action='place', detail=str(idx)))
        if validated.purchase:
            rows.append(dict(turn=turn, squad=squad.name, 
                             piece=validated.purchase.piece_type.value,
                             action='purchase', 
                             detail=str(self.parts.hp2i(validated.purchase.location.location))))
        
        self._df = pd.concat([self._df, pd.DataFrame(rows)], ignore_index=True)
    
    @property
    def df(self) -> pd.DataFrame: return self._df
    
    def for_squad(self, name: str) -> pd.DataFrame:
        return self._df[self._df.squad == name]
    
    def turn_count(self) -> int:
        return int(self._df.turn.max() + 1) if len(self._df) else 0


@patch
def food_income(self: Squad, parts: GameParts) -> float:
    """Total food harvested by all placed pieces."""
    return sum(p.food.harvest * p.place.harvest
               for p in self.pieces if p.place is not None)

@patch
def food_upkeep(self: Squad) -> float:
    """Total diet cost for all placed pieces."""
    return sum(p.food.diet for p in self.pieces if p.place is not None)

@patch
def food_drain_suffered(self: Squad, all_squads: list['Squad']) -> float:
    """Total drain imposed on this squad by enemy sight cones."""
    my_placed = {p.place.parent.hp2i(p.place.location)
                 for p in self.pieces if p.place is not None}
    total = 0.0
    for enemy_squad in all_squads:
        if enemy_squad is self:
            continue
        for enemy in enemy_squad.pieces:
            if enemy.place is None:
                continue
            cone = enemy.sight().hexes
            caught = len(my_placed & cone)
            total += enemy.food.drain * caught
    return total

@patch
def food_net(self: Squad, parts: GameParts, all_squads: list['Squad']) -> float:
    """Net food change this turn (before placement costs)."""
    return self.food_income(parts) - self.food_upkeep() - self.food_drain_suffered(all_squads)


## Squad Network

In [ ]:
@patch
def to_nx(self: Squad) -> nx.DiGraph:
    """Build directed sight graph as NetworkX DiGraph with food attributes."""
    G = nx.DiGraph()
    placed = [p for p in self.pieces if p.place is not None]
    parts = placed[0].place.parent if placed else None
    if not parts:
        return G
    
    for p in placed:
        idx = parts.hp2i(p.place.location)
        G.add_node(p.id, 
                   piece=p,  # back-reference
                   piece_type=p.type.value,
                   harvest=int(p.food.harvest * p.place.harvest),
                   diet=p.food.diet,
                   drain=p.food.drain,
                   cost=p.food.cost,
                   pantry=p.place.pantry,
                   hex_idx=idx,
                   is_sink=p.food.harvest == 0.0 or p.type == PieceType.KING,
                   is_king=p.type == PieceType.KING)
    
    for p in placed:
        cone = p.sight().hexes
        for other in placed:
            if other is p:
                continue
            idx = parts.hp2i(other.place.location)
            if idx in cone:
                G.add_edge(p.id, other.id)
    
    return G


In [ ]:
@patch
def supply_bottlenecks(self: Squad) -> list[Piece]:
    """Pieces whose removal disconnects the sight network."""
    G = self.to_nx().to_undirected()
    return [G.nodes[n]['piece'] for n in nx.articulation_points(G)]

@patch
def isolated_pieces(self: Squad) -> list[Piece]:
    """Pieces not connected to the king's component."""
    G = self.to_nx().to_undirected()
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id: return []
    king_component = nx.node_connected_component(G, king_id)
    return [G.nodes[n]['piece'] for n in G.nodes if n not in king_component]


In [ ]:
@patch
def food_sinks(self: Squad) -> list[Piece]:
    """Pieces that consume but don't produce — need to be fed by the network."""
    G = self.to_nx()
    return [G.nodes[n]['piece'] for n in G.nodes 
            if G.nodes[n]['is_sink']]

@patch
def food_sources(self: Squad) -> list[Piece]:
    """Pieces that can produce surplus (harvest > diet)."""
    G = self.to_nx()
    return [G.nodes[n]['piece'] for n in G.nodes 
            if G.nodes[n]['harvest'] > G.nodes[n]['diet']]

@patch
def flow_network(self: Squad) -> dict:
    """Analyse food flow using the sight graph.
    
    Returns dict with:
        G: the DiGraph
        sinks: set of sink node ids
        reachable: dict[node_id, set[sink_id]] — which sinks each piece can reach
        disconnected: list[Piece] — pieces that can't reach ANY sink
        sink_layers: dict[sink_id, dict[node_id, int]] — BFS distance from each sink
    """
    G = self.to_nx()
    if not G.nodes:
        return {'G': G, 'sinks': set(), 'reachable': {}, 
                'disconnected': [], 'sink_layers': {}}
    
    sinks = {n for n, d in G.nodes(data=True) if d['is_sink']}
    
    # For each sink, find all nodes that can reach it (reverse BFS)
    R = G.reverse()
    sink_connected = {}
    for sink in sinks:
        sink_connected[sink] = nx.ancestors(R, sink) | {sink}
    
    # Per-node: which sinks can it reach?
    reachable = {n: set() for n in G.nodes}
    for sink, connected in sink_connected.items():
        for n in connected:
            reachable[n].add(sink)
    
    disconnected = [G.nodes[n]['piece'] for n in G.nodes 
                    if not reachable[n] and n not in sinks]
    
    # BFS layers from each sink (along reverse edges = upstream distance)
    sink_layers = {}
    for sink in sinks:
        sink_layers[sink] = nx.shortest_path_length(R, sink)
    
    return {
        'G': G,
        'sinks': sinks,
        'sink_connected': sink_connected,
        'sink_layers': sink_layers,
        'reachable': reachable,
        'disconnected': disconnected,
    }




In [ ]:
@patch
def resolve_flow(self: Squad):
    """Centralized treasury with local fallback for disconnected pieces."""
    G = self.to_nx()
    if len(G) < 2:
        return

    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id:
        return

    king_piece = G.nodes[king_id]['piece']
    U = G.to_undirected(as_view=True)
    king_comp = nx.node_connected_component(U, king_id)

    for nid, data in G.nodes(data=True):
        if nid == king_id:
            continue
        p = data['piece']
        if nid in king_comp:
            # Connected: everything flows to king
            king_piece.place.pantry += p.place.pantry
            p.place.pantry = 0
        # else: disconnected, keeps its own pantry (will likely starve)

In [ ]:
@patch
def king_alive(self: Squad) -> bool:
    """Is the king still on the board?"""
    return any(p.type == PieceType.KING and p.place is not None for p in self.pieces)

In [ ]:
@patch
def orient_facings(self: Squad, parts: GameParts):
    """Cheap target-aware facing: blend ally connectivity with target projection."""
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2: return
    
    target = getattr(self, '_target', None)
    target_hp = parts.i2hp(target.hex_idx) if target and target.hex_idx >= 0 else None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    allies = {parts.hp2i(p.place.location) for p in placed}
    
    for piece in placed:
        my_hp = piece.place.location
        bonus = piece.food.bonus_sight
        eff = max(2, piece.place.level) + bonus
        
        best_dir, best_score = piece.place.facing, -999
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in my_hp.field_of_view(d, eff))
            
            # Connectivity: always valuable
            score = len((allies - {parts.hp2i(my_hp)}) & cone) * 3.0
            
            # King visibility bonus
            if king and king is not piece:
                if parts.hp2i(king.place.location) in cone:
                    score += 4.0
            
            # Target projection (the new part)
            if target_hp is not None:
                target_idx = parts.hp2i(target_hp)
                if target_idx in cone:
                    score += 8.0
                else:
                    # Partial: how far does cone extend toward target?
                    if cone:
                        min_dist = min(parts.i2hp(h).distance(target_hp) for h in cone)
                        score += 3.0 / max(1, min_dist)
            
            if score > best_score:
                best_score = score
                best_dir = d
        
        piece.place.facing = best_dir


In [ ]:
@patch
def tick_lifespan(self: Squad) -> list[Piece]:
    """Increment tenure for placed pieces. Evict any that exceed lifespan."""
    evicted = []
    for p in self.pieces:
        if p.place is None or p.type == PieceType.KING:
            continue
        p.place.tenure += 1
        if p.place.tenure >= LIFESPAN.get(p.type, 4):
            p.place = None
            evicted.append(p)
    return evicted


#### SquadFlow

In [ ]:
@dataclass
class SquadFlow:
    """Computed food flow state for a squad. Shared by diagram + overlay renderers."""
    squad: Squad
    G: nx.DiGraph
    edge_flows: dict  # (u, v) -> float
    node_inflow: dict  # nid -> float
    node_outflow: dict  # nid -> float

    @classmethod
    def topology_only(cls, squad: Squad) -> 'SquadFlow':
        """Build graph topology without running max-flow. Fast."""
        G = squad.to_nx()
        empty = {n: 0 for n in G.nodes}
        return cls(squad, G, {}, empty, empty)
    
    @classmethod
    def from_squad(cls, squad: Squad) -> 'SquadFlow':
        G = squad.to_nx()
        if len(G) < 2:
            empty = {n: 0 for n in G.nodes}
            return cls(squad, G, {}, empty, empty)

        king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
        if not king_id:
            empty = {n: 0 for n in G.nodes}
            return cls(squad, G, {}, empty, empty)

        U = G.to_undirected(as_view=True)
        king_comp = nx.node_connected_component(U, king_id)

        edge_flows = {}
        node_inflow = {n: 0 for n in G.nodes}
        node_outflow = {n: 0 for n in G.nodes}

        for nid, data in G.nodes(data=True):
            if nid == king_id or nid not in king_comp:
                continue
            net = data['harvest'] - data['diet']
            if net > 0:
                # Surplus flows from this piece toward king
                # Find the shortest sight-path to king for the diagram
                try:
                    path = nx.shortest_path(U, nid, king_id)
                    for i in range(len(path) - 1):
                        u, v = path[i], path[i + 1]
                        edge_flows[(u, v)] = edge_flows.get((u, v), 0) + net
                        node_outflow[u] += net
                        node_inflow[v] += net
                except nx.NetworkXNoPath:
                    pass
            elif net < 0:
                # Deficit: king sends food to this piece
                try:
                    path = nx.shortest_path(U, king_id, nid)
                    for i in range(len(path) - 1):
                        u, v = path[i], path[i + 1]
                        edge_flows[(u, v)] = edge_flows.get((u, v), 0) + abs(net)
                        node_outflow[u] += abs(net)
                        node_inflow[v] += abs(net)
                except nx.NetworkXNoPath:
                    pass

        return cls(squad, G, edge_flows, node_inflow, node_outflow)
    
    def post_net(self, nid) -> float:
        """Net food for a node after flow redistribution."""
        data = self.G.nodes[nid]
        return data['harvest'] - data['diet'] + self.node_inflow[nid] - self.node_outflow[nid]
    
    def edge_flow(self, u, v) -> float:
        """Net flow on undirected edge (positive = u→v)."""
        return self.edge_flows.get((u, v), 0) - self.edge_flows.get((v, u), 0)
    
    @property
    def max_flow(self) -> float:
        return max(self.edge_flows.values(), default=1)
    
    @property
    def total_flow(self) -> float:
        return sum(self.edge_flows.values())
    
    @property
    def bottlenecks(self) -> set:
        if len(self.G) < 2: return set()
        return set(nx.articulation_points(self.G.to_undirected(as_view=True)))
    
    @property 
    def total_surplus(self) -> int:
        placed = [p for p in self.squad.pieces if p.place is not None]
        return sum(int(p.food.harvest * p.place.harvest) for p in placed) - sum(p.food.diet for p in placed)

    # Inside SquadFlow
    @property
    def king_component(self) -> set:
        if len(self.G) < 2: return set(self.G.nodes)
        king_id = next((n for n, d in self.G.nodes(data=True) if d['is_king']), None)
        if not king_id: return set()
        U = self.G.to_undirected(as_view=True)
        return nx.node_connected_component(U, king_id)


In [ ]:
@patch
def supply_bottlenecks(self: Squad) -> list[Piece]:
    """Pieces whose removal disconnects the sight network."""
    G = self.to_nx()
    U = G.to_undirected(as_view=True)
    return [G.nodes[n]['piece'] for n in nx.articulation_points(U)]

@patch
def isolated_pieces(self: Squad) -> list[Piece]:
    """Pieces not connected to the king's component."""
    G = self.to_nx()
    U = G.to_undirected(as_view=True)
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id: return []
    king_component = nx.node_connected_component(U, king_id)
    return [G.nodes[n]['piece'] for n in G.nodes if n not in king_component]

@patch
def flow_network(self: Squad) -> dict:
    """Analyse food flow using the sight graph."""
    G = self.to_nx()
    if not G.nodes:
        return {'G': G, 'sinks': set(), 'reachable': {},
                'disconnected': [], 'sink_layers': {}}

    sinks = {n for n, d in G.nodes(data=True) if d['is_sink']}

    R = G.reverse(copy=False)
    sink_connected = {}
    for sink in sinks:
        sink_connected[sink] = nx.ancestors(R, sink) | {sink}

    reachable = {n: set() for n in G.nodes}
    for sink, connected in sink_connected.items():
        for n in connected:
            reachable[n].add(sink)

    disconnected = [G.nodes[n]['piece'] for n in G.nodes
                    if not reachable[n] and n not in sinks]

    sink_layers = {}
    for sink in sinks:
        sink_layers[sink] = nx.shortest_path_length(R, sink)

    return {
        'G': G,
        'sinks': sinks,
        'sink_connected': sink_connected,
        'sink_layers': sink_layers,
        'reachable': reachable,
        'disconnected': disconnected,
    }


In [ ]:
# need to remove duplicate functions

In [ ]:
@patch
def sight_projection(self: SquadFlow, parts: GameParts) -> float:
    """How well do sight cones project toward the squad's target hex?"""
    target = getattr(self.squad, '_target', None)
    if target is None or target.hex_idx < 0:
        return 0.0
    target_hp = parts.i2hp(target.hex_idx)
    score = 0.0
    for n in self.G.nodes:
        cone = self.G.nodes[n]['piece'].sight().hexes
        if target.hex_idx in cone:
            score += 10.0
        elif cone:
            min_dist = min(parts.i2hp(h).distance(target_hp) for h in cone)
            score += 2.0 / max(1, min_dist)
    return score


In [ ]:
@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    node_scale: float = 0.9
    on_click: Callable[[Piece], dict] | None = None
    
    def __ft__(self):
        return self._render()
    
    def _render(self,flow:SquadFlow = None):
        
        if flow is None:
            flow = SquadFlow.from_squad(self.squad)  # ← use self.squad

        if flow is None:
            flow=GS.squad_flow() #flow = SquadFlow.from_squad(self.squad)
        G = flow.G
        
        if len(G) < 1:
            return Div(P(f"{self.squad.name}: no placed pieces"))
        
        flag = self.squad.flag
        node_r = 22.5 * self.node_scale
        pad = node_r * 2.5
        inner_w = self.width - pad * 2
        inner_h = self.height - pad * 2
        
        # --- Layout ---
        if len(G) == 1:
            pos = {n: np.array([0.0, 0.0]) for n in G.nodes}
        else:
            pos = nx.kamada_kawai_layout(G)
        
        xs = [p[0] for p in pos.values()]
        ys = [p[1] for p in pos.values()]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        x_range = x_max - x_min or 1.0
        y_range = y_max - y_min or 1.0
        
        def to_svg(nx_pos):
            return (pad + (nx_pos[0] - x_min) / x_range * inner_w,
                    pad + (nx_pos[1] - y_min) / y_range * inner_h)
        
        svg_pos = {nid: to_svg(p) for nid, p in pos.items()}
        
        # --- Build SVG ---
        builder = SVGBuilder()
        builder.width = self.width
        builder.height = self.height
        
        # Arrow marker
        arrow_id = f"arrow_{self.squad.name.replace(' ', '_')}"
        builder.adjust("defs",
            f'<defs><marker id="{arrow_id}" viewBox="0 0 10 10" '
            f'refX="10" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{flag.darkPrimary}" opacity="0.6"/>'
            f'</marker></defs>')
        
        # --- Edges ---
        edge_parts = []
        shrink = node_r + 8
        seen = set()
        
        for u, v in G.edges():
            key = tuple(sorted([u, v]))
            if key in seen: continue
            seen.add(key)
            
            net_flow = flow.edge_flow(u, v)
            total = abs(net_flow)
            
            # Arrow points in direction of net flow
            if net_flow >= 0:
                src, dst = u, v
            else:
                src, dst = v, u
            
            x1, y1 = svg_pos[src]
            x2, y2 = svg_pos[dst]
            dx, dy = x2 - x1, y2 - y1
            dist = math.sqrt(dx*dx + dy*dy)
            if dist < 1: continue
            ux, uy = dx / dist, dy / dist
            
            if total > 0:
                w = 1.5 + (total / flow.max_flow) * 4
                color = "#4CAF50"
                dash = ""
                opacity = 0.6
            else:
                w = 1.0
                color = flag.darkPrimary
                dash = ' stroke-dasharray="4,3"'
                opacity = 0.25
            
            # Bridge highlight
            is_bridge = (u in flow.bottlenecks or v in flow.bottlenecks)
            if is_bridge and total == 0:
                color = "#E53935"
                opacity = 0.4
                w = 1.5
            
            edge_parts.append(
                f'<line x1="{x1 + ux*shrink:.1f}" y1="{y1 + uy*shrink:.1f}" '
                f'x2="{x2 - ux*shrink:.1f}" y2="{y2 - uy*shrink:.1f}" '
                f'stroke="{color}" stroke-opacity="{opacity}" stroke-width="{w:.1f}" '
                f'marker-end="url(#{arrow_id})"{dash}/>')
            
            if total > 0:
                mx = (x1 + x2) / 2
                my = (y1 + y2) / 2
                px, py = -uy * 10, ux * 10
                edge_parts.append(
                    f'<text x="{mx + px:.1f}" y="{my + py:.1f}" '
                    f'text-anchor="middle" font-size="10" font-weight="bold" '
                    f'fill="#4CAF50" opacity="0.8">{total:.0f}🌾</text>')
        
        builder.adjust("edges", '\n'.join(edge_parts))
        
        # --- Nodes ---
        node_parts = []
        for nid, data in G.nodes(data=True):
            piece = data['piece']
            cx, cy = svg_pos[nid]
            
            post_net = flow.post_net(nid)
            tenure = piece.place.tenure if piece.place else 0
            life = LIFESPAN.get(piece.type, 999)
            turns_left = max(0, life - tenure) if life < 999 else -1
            pantry = piece.place.pantry if piece.place else 0
            
            # Ring color based on post-flow net
            if data['is_king']:
                ring_color, ring_width = "#FFD700", 3.5
            elif post_net >= 0:
                ring_color, ring_width = "#4CAF50", 2.5
            else:
                ring_color, ring_width = "#E53935", 2.5
            
            # Bottleneck warning
            is_bn = nid in flow.bottlenecks
            
            # HTMX attrs
            hx_attrs = ""
            if self.on_click:
                attrs = self.on_click(piece)
                hx_attrs = ' '.join(f'{k}="{v}"' for k, v in attrs.items())
            group_open = f'<g style="cursor:pointer" {hx_attrs}>' if hx_attrs else '<g>'
            
            ring_r = node_r + 6
            
            # Bottleneck outer ring
            bn_ring = ""
            if is_bn:
                bn_ring = (
                    f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{ring_r + 4:.1f}" '
                    f'fill="none" stroke="#E53935" stroke-width="2" '
                    f'stroke-dasharray="4,2" opacity="0.7"/>')
            
            # White background + colored ring
            ring = (
                f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{ring_r:.1f}" '
                f'fill="white" fill-opacity="0.9" '
                f'stroke="{ring_color}" stroke-width="{ring_width}"/>')
            
            # Chess piece
            piece_center = MapCord(cx, cy)
            svg_str, pat_def = flag.piece_svg(
                piece.type, piece_center,
                scale=self.node_scale,
                size='board',
                piece_id=f"flow_{nid}_{piece.type.value}")
            if pat_def is not None:
                builder.add_definition(pat_def)
            
            # Name
            name_y = cy + ring_r + 14
            name_label = (
                f'<text x="{cx:.1f}" y="{name_y:.1f}" text-anchor="middle" '
                f'font-size="11" font-weight="bold" fill="{flag.darkPrimary}">'
                f'{piece.name}</text>')
            
            # Post-flow net + pantry + turns
            info_y = name_y + 13
            net_color = "#4CAF50" if post_net >= 0 else "#E53935"
            turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
            info_label = (
                f'<text x="{cx:.1f}" y="{info_y:.1f}" text-anchor="middle" font-size="10">'
                f'<tspan fill="{net_color}">{post_net:+.0f}🌾</tspan>'
                f'  <tspan fill="#888">📦{pantry}</tspan>'
                f'  <tspan fill="#666">{turns_str}</tspan>'
                f'</text>')
            
            node_parts.append(
                f'{group_open}{bn_ring}{ring}{svg_str}{name_label}{info_label}</g>')
        
        builder.adjust("nodes", '\n'.join(node_parts))
        
        # --- Footer ---
        king = next((p for p in self.squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        king_pantry = king.place.pantry if king else 0
        placed_count = sum(1 for p in self.squad.pieces if p.place is not None)
        
        target = getattr(self.squad, '_target', None)
        target_str = target.label if target else "none"
        
        bn_count = len(flow.bottlenecks)
        
        footer = P(
            f"🌾 {flow.total_surplus:+d}/turn · "
            f"👑 {king_pantry} pantry · "
            f"🔄 {flow.total_flow:.0f} flow · "
            f"⚠️ {bn_count} bottleneck{'s' if bn_count != 1 else ''} · "
            f"🎯 {target_str}")
        
        return Card(
            NotStr(builder.xml()),
            footer=footer,
            header=DivFullySpaced(
                Div(H4(self.squad.name),
                    P(f"{flag.name} · {placed_count} placed")),
            ),
        )


The SquadFlowDiagram doesn't seem to putt all of the pieces on the board 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

There are two likely culprits:

**1. `GS.squad_flow()` hardcoded fallback** — In `_render`, when no `flow` is passed, it calls `GS.squad_flow()` (some global) instead of computing from `self.squad`:

```python
def _render(self, flow: SquadFlow = None):
    if flow is None:
        flow = GS.squad_flow()  # ← references undefined/stale global
```

Should be:

```python
def _render(self, flow: SquadFlow = None):
    if flow is None:
        flow = SquadFlow.from_squad(self.squad)
```

**2. Only placed pieces appear** — `to_nx()` filters to `p.place is not None`, so unplaced pieces are invisible in the diagram. If you want them shown (grayed out, off to the side), you'd need to add them as disconnected nodes.

Which of these is the issue you're hitting? The `GS` reference seems like the most likely one to cause missing pieces if it's returning flow for a different squad or a stale snapshot.

In [ ]:
def FoodNetworkOverlay(
    focus_squad: Squad | None = None,
    edge_opacity: float = 0.6,
    bottleneck_r_mult: float = 1.4,
    **kw
) -> OverlaySpec:
    """Food sharing network on the map, powered by SquadFlow."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts = []
        
        for squad in ctx.squads:
            flow=GS.squad_flow() #flow = SquadFlow.from_squad(squad) 
            G = flow.G
            if len(G) < 1: continue
            
            flag = squad.flag
            glyphs = DiagramGlyphs(flag, size=grid.radius * 0.5)
            glyphs.register_styles(ctx.builder)
            is_focused = focus_squad is None or squad is focus_squad
            
            # --- Edges ---
            if is_focused and len(G) > 1:
                U = G.to_undirected()
                bridges = set(nx.bridges(U))
                
                arrow_id = f"fna_{flag.name.replace(' ', '_')}"
                ctx.builder.add_definition(SVGDef("marker", arrow_id,
                    f'<path d="M 0 0 L 8 4 L 0 8 z" fill="{flag.darkPrimary}" opacity="0.6"/>',
                    viewBox="0 0 8 8", refX="8", refY="4",
                    markerWidth="5", markerHeight="5", orient="auto-start-reverse"))
                
                seen = set()
                for u, v in G.edges():
                    key = tuple(sorted([u, v]))
                    if key in seen: continue
                    seen.add(key)
                    
                    u_idx = G.nodes[u]['hex_idx']
                    v_idx = G.nodes[v]['hex_idx']
                    
                    # Flow direction
                    net_flow = flow.edge_flow(u, v)
                    total = abs(net_flow)
                    if net_flow >= 0:
                        src_idx, dst_idx = u_idx, v_idx
                    else:
                        src_idx, dst_idx = v_idx, u_idx
                    
                    # Edge style: flow vs sight-only, bridge vs redundant
                    is_bridge = (u, v) in bridges or (v, u) in bridges
                    
                    if total > 0:
                        sw = max(2, grid.radius * 0.08) + (total / flow.max_flow) * grid.radius * 0.15
                        color = "#4CAF50"
                        dash = ""
                        opacity = edge_opacity
                    elif is_bridge:
                        sw = max(2, grid.radius * 0.08)
                        color = "#E53935"
                        dash = 'stroke-dasharray="5,3"'
                        opacity = edge_opacity * 0.6
                    else:
                        sw = max(1.2, grid.radius * 0.05)
                        color = flag.darkPrimary
                        dash = 'stroke-dasharray="5,3"'
                        opacity = edge_opacity * 0.4
                    
                    for fi_s in ctx.fine_indices(src_idx):
                        for fi_d in ctx.fine_indices(dst_idx):
                            if 0 <= fi_s < N and 0 <= fi_d < N:
                                c1 = grid.hexes[fi_s].center
                                c2 = grid.hexes[fi_d].center
                                dx = c2.x - c1.x
                                dy = c2.y - c1.y
                                dist = math.sqrt(dx*dx + dy*dy)
                                if dist < 1: continue
                                ux, uy = dx / dist, dy / dist
                                shrink = grid.radius * 0.4
                                
                                parts.append(
                                    f'<line x1="{c1.x + ux*shrink:.1f}" y1="{c1.y + uy*shrink:.1f}" '
                                    f'x2="{c2.x - ux*shrink:.1f}" y2="{c2.y - uy*shrink:.1f}" '
                                    f'stroke="{color}" stroke-opacity="{opacity:.2f}" '
                                    f'stroke-width="{sw:.1f}" stroke-linecap="round" '
                                    f'marker-end="url(#{arrow_id})" {dash}/>')
                                
                                # Flow label at midpoint
                                if total > 0:
                                    mx = (c1.x + c2.x) / 2
                                    my = (c1.y + c2.y) / 2
                                    px, py = -uy * grid.radius * 0.4, ux * grid.radius * 0.4
                                    fs = max(8, grid.radius * 0.28)
                                    parts.append(
                                        f'<text x="{mx + px:.1f}" y="{my + py:.1f}" '
                                        f'text-anchor="middle" font-size="{fs:.0f}" '
                                        f'font-weight="bold" fill="#4CAF50" opacity="0.8">'
                                        f'{total:.0f}🌾</text>')
            
            # --- Target direction indicator ---
            if is_focused:
                target = getattr(squad, '_target', None)
                if target and target.hex_idx >= 0:
                    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
                    if king_id:
                        k_idx = G.nodes[king_id]['hex_idx']
                        t_idx = target.hex_idx
                        for fi_k in ctx.fine_indices(k_idx):
                            if 0 <= fi_k < N:
                                kc = grid.hexes[fi_k].center
                                for fi_t in ctx.fine_indices(t_idx):
                                    if 0 <= fi_t < N:
                                        tc = grid.hexes[fi_t].center
                                        dx = tc.x - kc.x
                                        dy = tc.y - kc.y
                                        dist = math.sqrt(dx*dx + dy*dy)
                                        if dist < 1: break
                                        ux, uy = dx / dist, dy / dist
                                        arrow_len = grid.radius * 2.5
                                        ax = kc.x + ux * arrow_len
                                        ay = kc.y + uy * arrow_len
                                        icon = "⚔️" if target.type == TargetType.KING else "🌾"
                                        
                                        parts.append(
                                            f'<line x1="{kc.x:.1f}" y1="{kc.y:.1f}" '
                                            f'x2="{ax:.1f}" y2="{ay:.1f}" '
                                            f'stroke="{flag.primary}" stroke-opacity="0.4" '
                                            f'stroke-width="{max(3, grid.radius * 0.15):.1f}" '
                                            f'stroke-linecap="round" stroke-dasharray="8,4"/>')
                                        parts.append(
                                            f'<text x="{ax:.1f}" y="{ay:.1f}" '
                                            f'text-anchor="middle" font-size="{max(12, grid.radius * 0.5):.0f}" '
                                            f'dominant-baseline="central">{icon}</text>')
                                        break
            
            # --- Nodes ---
            for nid, data in G.nodes(data=True):
                piece = data['piece']
                idx = data['hex_idx']
                post_net = flow.post_net(nid)
                is_bn = nid in flow.bottlenecks
                
                tenure = piece.place.tenure if piece.place else 0
                life = LIFESPAN.get(piece.type, 999)
                turns_left = max(0, life - tenure) if life < 999 else -1
                
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        c = grid.hexes[fi].center
                        r = grid.radius * 0.25
                        
                        # Bottleneck warning ring
                        if is_bn and is_focused:
                            parts.append(glyphs.ring(
                                c, r * bottleneck_r_mult,
                                glyphs.danger_style()))
                        
                        # Node glyph
                        if post_net > 0:
                            parts.append(glyphs.sheaf(c, size=r * 2.5))
                        elif post_net < 0:
                            parts.append(glyphs.shield(c, size=r * 2))
                        else:
                            parts.append(glyphs.ring(c, r))
                        
                        if is_focused:
                            # Pantry bar
                            pantry = piece.place.pantry if piece.place else 0
                            bar_w = grid.radius * 0.8
                            bar_h = max(3, grid.radius * 0.1)
                            bar_x = c.x - bar_w / 2
                            bar_y = c.y + r * 2.0
                            
                            max_pantry = max(piece.food.cost * 2, 20)
                            fill_frac = max(0, min(1, pantry / max_pantry))
                            fill_color = "#4CAF50" if pantry > 0 else "#E53935"
                            
                            parts.append(
                                f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" '
                                f'width="{bar_w:.1f}" height="{bar_h:.1f}" '
                                f'rx="1.5" fill="#ddd" opacity="0.6"/>')
                            parts.append(
                                f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" '
                                f'width="{bar_w * fill_frac:.1f}" height="{bar_h:.1f}" '
                                f'rx="1.5" fill="{fill_color}" opacity="0.8"/>')
                            
                            # Net + turns label
                            label_y = bar_y + bar_h + max(8, grid.radius * 0.3)
                            net_color = "#4CAF50" if post_net >= 0 else "#E53935"
                            turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
                            fs = max(8, grid.radius * 0.28)
                            
                            parts.append(
                                f'<text x="{c.x:.1f}" y="{label_y:.1f}" '
                                f'text-anchor="middle" font-size="{fs:.0f}">'
                                f'<tspan fill="{net_color}" font-weight="bold">{post_net:+.0f}</tspan>'
                                f' <tspan fill="#888">{turns_str}</tspan>'
                                f'</text>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("food_network", render, requires={'squads'}, priority=85)


In [ ]:
if showDemo:
    ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(),
        LanternOverlay(),
        PieceOverlay(),
        FoodNetworkOverlay(),
        pieces=myStuff.squads[0].pieces,
        ctx=ctx,
        debug = not showDemo
    )


In [ ]:
@dataclass 
class GameRecord:
    """Tidy per-turn telemetry for all squads."""
    _df: pd.DataFrame = field(default_factory=lambda: pd.DataFrame(columns=[
        'turn', 'squad', 'is_player', 'alive',
        'king_pantry', 'total_harvest', 'total_diet', 'net_food',
        'drain_dealt', 'drain_taken',
        'pieces_placed', 'pieces_available',
        'bottlenecks', 'king_connected', 'isolated',
        'sight_coverage',
        'target_type', 'stance',
    ]))
    
    def snapshot(self, turn: int, parts: GameParts, player_squad: Squad):
        rows = []
        for sq in parts.squads:
            placed = [p for p in sq.pieces if p.place is not None]
            king = next((p for p in placed if p.type == PieceType.KING), None)
            
            # Food
            harvest = sum(int(p.food.harvest * p.place.harvest) for p in placed)
            diet = sum(p.food.diet for p in placed)
            
            # Drain
            drain_dealt = sum(
                ep.food.drain
                for p in placed for osq in parts.squads if osq is not sq
                for ep in osq.pieces
                if ep.place and parts.hp2i(ep.place.location) in p.sight().hexes)
            drain_taken = sum(
                p.food.drain
                for osq in parts.squads if osq is not sq
                for p in osq.pieces if p.place
                for myp in placed
                if parts.hp2i(myp.place.location) in p.sight().hexes)
            
            # Network
            G = sq.to_nx()
            U = G.to_undirected(as_view=True) if len(G) > 1 else nx.Graph()
            bn = len(list(nx.articulation_points(U))) if len(G) > 1 else 0
            king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
            king_comp = len(nx.node_connected_component(U, king_id)) if king_id and len(G) > 1 else len(placed)
            iso = len(sq.isolated_pieces())
            
            # Sight
            visible = set()
            for p in placed: visible |= p.sight().hexes
            
            target = getattr(sq, '_target', None)
            
            rows.append(dict(
                turn=turn, squad=sq.name,
                is_player=sq is player_squad,
                alive=sq.king_alive(),
                king_pantry=king.place.pantry if king else 0,
                total_harvest=harvest,
                total_diet=diet,
                net_food=harvest - diet,
                drain_dealt=drain_dealt,
                drain_taken=drain_taken,
                pieces_placed=len(placed),
                pieces_available=sum(1 for p in sq.pieces if p.place is None),
                bottlenecks=bn,
                king_connected=king_comp,
                isolated=iso,
                sight_coverage=len(visible),
                target_type=target.type.value if target else 'none',
                stance=getattr(sq, '_stance', 'expand'),
            ))
        
        self._df = pd.concat([self._df, pd.DataFrame(rows)], ignore_index=True)
    
    @property
    def df(self) -> pd.DataFrame:
        return self._df
    
    def for_squad(self, name: str) -> pd.DataFrame:
        return self._df[self._df.squad == name]
    
    def latest(self) -> pd.DataFrame:
        if self._df.empty: return self._df
        return self._df[self._df.turn == self._df.turn.max()]
    
    def sparkline_data(self, col: str) -> dict[str, list]:
        """Dict of squad→values for any column, ready for plotting."""
        return {
            name: grp.sort_values('turn')[col].tolist()
            for name, grp in self._df.groupby('squad')
        }


## TurnProcessor

In [ ]:
@dataclass
class TurnProcessor:
    parts: GameParts
    squads: list[Squad]
    log: MoveLog = None
    record: GameRecord = None      # ← add this
    turn: int = 0
    game_over: bool = False
    winner: Squad | None = None
    player_squad: Squad | None = None  # optional — None for pure AI games

    def __post_init__(self):
        if self.log is None:
            self.log = MoveLog(self.parts)
        if self.record is None:
            self.record = GameRecord()
    
    def _king(self, squad: Squad) -> Piece | None:
        return next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    
    
    def play_round(self, moves: dict[Squad, Move]) -> list[dict]:
        summaries = []
        for squad in self.squads:
            if not squad.king_alive():
                continue
            if squad not in moves:
                continue
            s = self.resolve_move(squad, moves[squad])
            summaries.append(s)
            if self.game_over:
                break
        self.record.snapshot(self.turn, self.parts, player_squad=self.player_squad)
        self.turn += 1
        return summaries

    
    def _check_game_over(self):
        alive = [s for s in self.squads if s.king_alive()]
        if len(alive) <= 1:
            self.game_over = True
            self.winner = alive[0] if alive else None
    
    def status(self) -> pd.DataFrame:
        rows = []
        for s in self.squads:
            king = self._king(s)
            placed = [p for p in s.pieces if p.place is not None]
            G = s.to_nx()
            king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
            bottlenecks = len(list(nx.articulation_points(G.to_undirected()))) if len(G) > 1 else 0
            rows.append({
                'squad': s.name,
                'king_pantry': king.place.pantry if king else 0,
                'placed': len(placed),
                'bottlenecks': bottlenecks,
                'king': '👑' if s.king_alive() else '💀',
            })
        return pd.DataFrame(rows)


In [ ]:
@patch
def _placement_cost(self:TurnProcessor, squad: Squad, move: Move, summary: dict):
    if move.placement is None: return
    piece = move.placement.piece
    cost = piece.food.cost
    king = self._king(squad)
    if king and piece.type != PieceType.KING:
        king.place.pantry -= cost
    piece.place.pantry = int(cost * 0.75)
    summary['placement_cost'] = cost
    summary['placement_pantry'] = piece.place.pantry

@patch
def _food_cycle(self:TurnProcessor, squad: Squad, summary: dict):
    king = self._king(squad)
    placed = [p for p in squad.pieces if p.place is not None]
    for p in placed:
        earned = int(p.food.harvest * p.place.harvest)
        king.place.pantry += earned
        summary['food_harvested'] += earned
    for p in placed:
        king.place.pantry -= p.food.diet
        summary['food_diet'] += p.food.diet

@patch
def _starvation_priority(self:TurnProcessor, p: Piece, king_hp) -> tuple:
    RANK = {PieceType.PAWN: 0, PieceType.KNIGHT: 1,
            PieceType.BISHOP: 2, PieceType.QUEEN: 3}
    return (p.place.pantry,
            RANK.get(p.type, 0),
            -(p.place.location.distance(king_hp) if king_hp else 0))
@patch
def _apply_starvation(self:TurnProcessor, squad: Squad, summary: dict):
    king = self._king(squad)
    king_hp = king.place.location if king else None
    placed = [p for p in squad.pieces if p.place is not None]
    starving = sorted(
        [p for p in placed if p.place.pantry < 0 and p.type != PieceType.KING],
        key=lambda p: self._starvation_priority(p, king_hp)
    )
    for p in starving:
        p.place = None
        summary['starved'].append(p.name)

@patch
def _connectivity_snapshot(self:TurnProcessor, squad: Squad, summary: dict):
    G = squad.to_nx()
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if king_id:
        U = G.to_undirected()
        connected = sum(1 for n in G.nodes if nx.has_path(U, n, king_id))
        summary['connected'] = connected
        summary['disconnected'] = len(G) - connected

@patch
def resolve_move(self: TurnProcessor, squad: Squad, move: Move) -> dict:
    if self.game_over: raise MoveError("Game is over")
    if not squad.king_alive(): raise MoveError(f"{squad.name} is eliminated")

    king = self._king(squad)
    summary = {
        'squad': squad.name, 'turn': self.turn,
        'placement_cost': 0, 'placement_pantry': 0,
        'food_harvested': 0.0, 'food_diet': 0.0,
        'food_drain_dealt': 0.0, 'starved': [], 'expired': [],
        'connected': 0, 'disconnected': 0,
        'king_pantry_before': king.place.pantry if king else 0,
        'king_pantry_after': 0, 'pantries': {},
        'game_over': False, 'eliminated': None,
    }

    self.log.record(squad, move)
    if move.purchase is not None:
        summary['purchase_type'] = move.purchase.piece_type.value
        summary['purchase_cost'] = PURCHASE_COST[move.purchase.piece_type]

    self._placement_cost(squad, move, summary)
    summary['expired'] = [p.name for p in squad.tick_lifespan()]
    self._food_cycle(squad, summary)
    self._apply_drain(squad, summary)
    squad.resolve_flow()
    self._connectivity_snapshot(squad, summary)
    self._apply_starvation(squad, summary)

    summary['pantries'] = {p.name: p.place.pantry 
                           for p in squad.pieces if p.place is not None}
    self._check_elimination(squad, summary)
    summary['king_pantry_after'] = (king.place.pantry 
                                    if (king := self._king(squad)) and king.place else 0)
    self._check_game_over()
    return summary


In [ ]:
@patch
def _apply_drain(self: TurnProcessor, squad: Squad, summary: dict):
    placed = [p for p in squad.pieces if p.place is not None]
    drained_squads: dict[str, Squad] = {}  # name → squad
    
    for p in placed:
        cone = p.sight().hexes
        for enemy_sq in self.squads:
            if enemy_sq is squad: continue
            for enemy in enemy_sq.pieces:
                if enemy.place is None: continue
                if self.parts.hp2i(enemy.place.location) in cone:
                    drain = int(p.food.drain)
                    enemy_king = self._king(enemy_sq)
                    if enemy_king:
                        enemy_king.place.pantry -= drain
                        summary['food_drain_dealt'] += drain
                        drained_squads[enemy_sq.name] = enemy_sq

    for enemy_sq in drained_squads.values():
        enemy_king = self._king(enemy_sq)
        if enemy_king and enemy_king.place.pantry <= 0:
            enemy_king.place.pantry = 0
            enemy_sq.kill()
            summary.setdefault('kills', []).append(enemy_sq.name)


In [ ]:
@patch
def _check_elimination(self: TurnProcessor, squad: Squad, summary: dict):
    king = self._king(squad)
    if not squad.king_alive():
        squad.kill()
        summary['eliminated'] = squad.name
        return
    if king and king.place.pantry <= 0:
        king.place.pantry = 0  # clamp
        squad.kill()
        summary['eliminated'] = squad.name


In [ ]:
@patch(as_prop=True)
def land_components(self: GameParts) -> dict[int, int]:
    """Map each land hex to its connected component ID. Computed once."""
    if hasattr(self, '_land_comp'):
        return self._land_comp
    
    comp = {}
    cid = 0
    for i in range(len(self.terr.elevations)):
        if self.terr.elevations[i] <= 0 or i in comp:
            continue
        frontier = {i}
        while frontier:
            cur = frontier.pop()
            if cur in comp:
                continue
            comp[cur] = cid
            for nb in self.grid.neighborsOf(cur):
                if nb not in comp and self.terr.elevations[nb] > 0:
                    frontier.add(nb)
        cid += 1
    self._land_comp = comp
    return comp

@patch
def land_connected(self: GameParts, indices: list[int]) -> bool:
    comp = self.land_components
    cids = {comp.get(i) for i in indices}
    return len(cids) == 1 and None not in cids


In [ ]:
@patch
def find_starts(self: GameParts, n: int, ring_radius: int = 3, 
                min_distance: int = 6, balance_tolerance: float = 0.3,
                top_k: int = 40) -> list[int]:
    grid, terr, tiers = self.grid, self.terr, self.fy.tiers
    n_hexes = len(terr.elevations)
    
    # 1. Richness scores
    scores = np.zeros(n_hexes)
    for i in range(n_hexes):
        if terr.elevations[i] <= 0: continue
        neighbors = grid.indices_in_range(i, ring_radius)
        scores[i] = sum(tiers[j] for j in neighbors if terr.elevations[j] > 0)
    
    # 2. Top candidates sorted by score
    ranked = sorted([(i, scores[i]) for i in range(n_hexes) if scores[i] > 0],
                    key=lambda x: -x[1])[:top_k]
    
    # 3. Precompute hex distances between candidates
    def hex_dist(a, b):
        return grid.index_to_hexposition(a).distance(grid.index_to_hexposition(b))
    
    # 4. Try combinations, best balanced first
    best_set, best_spread = None, float('inf')
    
    for combo in combinations(range(len(ranked)), n):
        idxs = [ranked[c][0] for c in combo]
        scs = [ranked[c][1] for c in combo]
        
        # Distance check
        if any(hex_dist(a, b) < min_distance 
               for a, b in combinations(idxs, 2)):
            continue
        
        # Connectivity check
        if not self.land_connected(idxs):
            continue
        
        # Balance check
        median = np.median(scs)
        if median == 0: continue
        spread = (max(scs) - min(scs)) / median
        
        if spread < best_spread:
            best_spread = spread
            best_set = list(zip(idxs, scs))
            if spread <= balance_tolerance:
                break
    
    if best_set is None:
        raise ValueError(f"Could not find {n} balanced land-connected positions")
    
    result = [idx for idx, _ in best_set]
    print(f"Start positions (spread={best_spread:.2f}):")
    for idx, score in best_set:
        print(f"  Hex {idx}: richness={score:.0f}, level={terr.elevationLevel(idx)}, tier={tiers[idx]}")
    
    return result


In [ ]:
playerCount = 5
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads( year=1985, seed = 23,levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)
# Place each squad's king
for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    move = Move.from_squad(squad, placement=Placement(king, loc))
    move.apply(myStuff)

# Check it worked
if showDemo:
    for sq in myStuff.squads:
        show(sq.table())

## Targets

## Targets

In [ ]:
def _score_network(squad: Squad, parts: GameParts, flow: SquadFlow | None = None) -> float:
    """Score a squad's current facing configuration, weighted by active target."""
    if flow is None:
        flow = SquadFlow.from_squad(squad)
    G = flow.G
    if len(G) < 2: return 0.0
    
    king_comp = flow.king_component
    if not king_comp: return 0.0
    
    # --- Base: connectivity + structure ---
    connectivity = (len(king_comp) / len(G)) * 100
    bottleneck_penalty = len(flow.bottlenecks) * 5
    U = G.to_undirected(as_view=True)
    edge_bonus = U.number_of_edges() * 0.5
    base = connectivity - bottleneck_penalty + edge_bonus
    
    target = getattr(squad, '_target', None)
    
    # --- CONSOLIDATE or no target ---
    if target is None or target.type == TargetType.CONSOLIDATE:
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        return base + food_reach
    
    target_hp = parts.i2hp(target.hex_idx)
    projection = flow.sight_projection(parts)
    
    if target.type == TargetType.KING:
        # Maximize drain on enemies near target king
        drain_score = 0.0
        for n in G.nodes:
            p = G.nodes[n]['piece']
            if p.food.drain <= 0: continue
            cone = p.sight().hexes
            for enemy_sq in parts.squads:
                if enemy_sq is squad: continue
                for ep in enemy_sq.pieces:
                    if ep.place is None: continue
                    if parts.hp2i(ep.place.location) in cone:
                        dist_to_target = max(1, ep.place.location.distance(target_hp))
                        drain_score += p.food.drain * (3.0 / dist_to_target)
        
        return base + drain_score * 3.0 + projection * 2.0
    
    else:  # OASIS
        land_set = {i for i in range(len(parts.terr.elevations))
                    if parts.terr.elevations[i] > 0}
        
        all_visible = set()
        for n in G.nodes:
            all_visible |= G.nodes[n]['piece'].sight().hexes
        new_land = len(all_visible & land_set)
        
        oasis_bonus = 0.0
        for n in G.nodes:
            cone = G.nodes[n]['piece'].sight().hexes
            for h in cone:
                if h in land_set:
                    dist = parts.i2hp(h).distance(target_hp)
                    if dist <= 3:
                        oasis_bonus += parts.fy.tiers[h] * 0.5
        
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        return base + new_land * 1.5 + (projection + oasis_bonus) * 2.0 + food_reach


can we refactor _score_network to use asView and/or should we modify squadFlow and it work better?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question — there's clear overlap between `_score_network` and `SquadFlow`. Right now `_score_network` rebuilds the graph from scratch every call, while `SquadFlow` already holds `G`, `bottlenecks`, flow data, etc.

Here's my take on the cleanest path:

**1. `_score_network` should accept a `SquadFlow` instead of rebuilding everything**

```python
def _score_network(squad: Squad, parts: GameParts, flow: SquadFlow | None = None) -> float:
    """Score a squad's current facing configuration, weighted by active target."""
    if flow is None:
        flow = SquadFlow.from_squad(squad)
    G = flow.G
    if len(G) < 2: return 0.0
    
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id: return 0.0
    U = G.to_undirected(as_view=True)  # ← lightweight view
    
    # --- Base: connectivity ---
    king_comp = nx.node_connected_component(U, king_id)
    connectivity = (len(king_comp) / len(G)) * 100
    bottleneck_penalty = len(flow.bottlenecks) * 5  # ← reuse cached property
    edge_bonus = U.number_of_edges() * 0.5
    
    base = connectivity - bottleneck_penalty + edge_bonus
    
    target = getattr(squad, '_target', None)
    
    if target is None or target.type == TargetType.CONSOLIDATE:
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        return base + food_reach
    
    target_hp = parts.i2hp(target.hex_idx)
    
    if target.type == TargetType.KING:
        drain_score = 0.0
        for n in G.nodes:
            p = G.nodes[n]['piece']
            if p.food.drain <= 0: continue
            cone = p.sight().hexes
            for enemy_sq in parts.squads:
                if enemy_sq is squad: continue
                for ep in enemy_sq.pieces:
                    if ep.place is None: continue
                    ep_idx = parts.hp2i(ep.place.location)
                    if ep_idx in cone:
                        dist_to_target = max(1, ep.place.location.distance(target_hp))
                        drain_score += p.food.drain * (3.0 / dist_to_target)
        
        projection = _sight_projection(G, parts, target)
        return base + drain_score * 3.0 + projection * 2.0
    
    else:  # OASIS
        land_set = {i for i in range(len(parts.terr.elevations)) 
                    if parts.terr.elevations[i] > 0}
        all_visible = set()
        for n in G.nodes:
            all_visible |= G.nodes[n]['piece'].sight().hexes
        new_land = len(all_visible & land_set)
        
        projection = 0.0
        for n in G.nodes:
            p = G.nodes[n]['piece']
            cone = p.sight().hexes
            if target.hex_idx in cone:
                projection += 10.0
            for h in cone:
                if h in land_set:
                    dist = parts.i2hp(h).distance(target_hp)
                    if dist <= 3:
                        projection += parts.fy.tiers[h] * 0.5
        
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        return base + new_land * 1.5 + projection * 2.0 + food_reach


def _sight_projection(G: nx.DiGraph, parts: GameParts, target) -> float:
    """How well do sight cones project toward the target hex?"""
    target_hp = parts.i2hp(target.hex_idx)
    projection = 0.0
    for n in G.nodes:
        cone = G.nodes[n]['piece'].sight().hexes
        if target.hex_idx in cone:
            projection += 10.0
        elif cone:
            min_dist = min(parts.i2hp(h).distance(target_hp) for h in cone)
            projection += 2.0 / max(1, min_dist)
    return projection
```

**2. Optionally add a `king_component` cached property to `SquadFlow`**

Since both `_score_network` and other callers need the king's connected component:

```python
# Inside SquadFlow
@property
def king_component(self) -> set:
    if len(self.G) < 2: return set(self.G.nodes)
    king_id = next((n for n, d in self.G.nodes(data=True) if d['is_king']), None)
    if not king_id: return set()
    U = self.G.to_undirected(as_view=True)
    return nx.node_connected_component(U, king_id)
```

Then `_score_network` becomes `king_comp = flow.king_component`.

**Summary of what changes:**

| Before | After |
|---|---|
| `squad.to_nx()` rebuilt each call | Reuse `flow.G` |
| `G.to_undirected()` (copy) | `G.to_undirected(as_view=True)` |
| `nx.articulation_points(U)` recomputed | `flow.bottlenecks` (cached) |
| King component recomputed | `flow.king_component` (new property) |

The `as_view=True` matters most in `optimize_facings` where `_score_network` gets called in a tight loop over every piece × every direction. The view avoids allocating a full graph copy each time.

Want me to also wire `SquadFlow` into `orient_facings` so it can use the same flow instance?

I am starting to integrate this. Should _sight_projection be patched on an object?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

`SquadFlow` is the natural home — it already holds `G` and `squad` (which has `_target`). That way `_score_network` just calls `flow.sight_projection(parts)` instead of passing the graph around.

```python
@patch
def sight_projection(self: SquadFlow, parts: GameParts) -> float:
    """How well do sight cones project toward the squad's target hex?"""
    target = getattr(self.squad, '_target', None)
    if target is None or target.hex_idx < 0:
        return 0.0
    target_hp = parts.i2hp(target.hex_idx)
    score = 0.0
    for n in self.G.nodes:
        cone = self.G.nodes[n]['piece'].sight().hexes
        if target.hex_idx in cone:
            score += 10.0
        elif cone:
            min_dist = min(parts.i2hp(h).distance(target_hp) for h in cone)
            score += 2.0 / max(1, min_dist)
    return score
```

Then `_score_network` simplifies to:

```python
projection = flow.sight_projection(parts)
```

This also means if you later want projection scores for other things (e.g. overlay labels, AI move selection), they're available from any `SquadFlow` instance without reimporting a private function.

can you rewrite _score_network?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
def _score_network(squad: Squad, parts: GameParts, flow: SquadFlow | None = None) -> float:
    """Score a squad's current facing configuration, weighted by active target."""
    if flow is None:
        flow = SquadFlow.from_squad(squad)
    G = flow.G
    if len(G) < 2: return 0.0
    
    king_comp = flow.king_component
    if not king_comp: return 0.0
    
    # --- Base: connectivity + structure ---
    connectivity = (len(king_comp) / len(G)) * 100
    bottleneck_penalty = len(flow.bottlenecks) * 5
    U = G.to_undirected(as_view=True)
    edge_bonus = U.number_of_edges() * 0.5
    base = connectivity - bottleneck_penalty + edge_bonus
    
    target = getattr(squad, '_target', None)
    
    # --- CONSOLIDATE or no target ---
    if target is None or target.type == TargetType.CONSOLIDATE:
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        return base + food_reach
    
    target_hp = parts.i2hp(target.hex_idx)
    projection = flow.sight_projection(parts)
    
    if target.type == TargetType.KING:
        # Maximize drain on enemies near target king
        drain_score = 0.0
        for n in G.nodes:
            p = G.nodes[n]['piece']
            if p.food.drain <= 0: continue
            cone = p.sight().hexes
            for enemy_sq in parts.squads:
                if enemy_sq is squad: continue
                for ep in enemy_sq.pieces:
                    if ep.place is None: continue
                    if parts.hp2i(ep.place.location) in cone:
                        dist_to_target = max(1, ep.place.location.distance(target_hp))
                        drain_score += p.food.drain * (3.0 / dist_to_target)
        
        return base + drain_score * 3.0 + projection * 2.0
    
    else:  # OASIS
        land_set = {i for i in range(len(parts.terr.elevations))
                    if parts.terr.elevations[i] > 0}
        
        all_visible = set()
        for n in G.nodes:
            all_visible |= G.nodes[n]['piece'].sight().hexes
        new_land = len(all_visible & land_set)
        
        oasis_bonus = 0.0
        for n in G.nodes:
            cone = G.nodes[n]['piece'].sight().hexes
            for h in cone:
                if h in land_set:
                    dist = parts.i2hp(h).distance(target_hp)
                    if dist <= 3:
                        oasis_bonus += parts.fy.tiers[h] * 0.5
        
        food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
        return base + new_land * 1.5 + (projection + oasis_bonus) * 2.0 + food_reach
```

Key changes from the original:
- Takes optional `flow: SquadFlow` — avoids rebuilding the graph
- `flow.king_component` instead of manual component lookup
- `flow.bottlenecks` instead of recomputing articulation points
- `G.to_undirected(as_view=True)` for edge count (no copy)
- `flow.sight_projection(parts)` instead of inline projection loop
- OASIS branch still has its own per-hex bonus loop (separate concern from the directional projection)

In [ ]:
@patch
def optimize_facings(self: Squad, parts: GameParts, max_iters: int = 3) -> int:
    """Coordinate descent over facings using target-aware objective."""
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2:
        return 0
    
    # Seed with cheap heuristic
    self.orient_facings(parts)
    
    # Priority ordering (computed once)
    flow = SquadFlow.topology_only(self)
    U = flow.G.to_undirected(as_view=True)
    king_id = next((n for n, d in flow.G.nodes(data=True) if d['is_king']), None)
    bottlenecks = set(nx.articulation_points(U)) if len(flow.G) > 1 else set()
    king_neighbors = set(U.neighbors(king_id)) if king_id else set()
    
    target = getattr(self, '_target', None)
    target_hp = parts.i2hp(target.hex_idx) if target and target.hex_idx >= 0 else None
    
    def priority(p):
        nid = p.id
        is_bn = nid in bottlenecks
        is_kn = nid in king_neighbors
        dist = p.place.location.distance(target_hp) if target_hp else 999
        return (not is_bn, not is_kn, dist)
    
    total_changes = 0
    
    for _ in range(max_iters):
        changed = False
        ordered = sorted(placed, key=priority)
        
        for piece in ordered:
            if piece.place is None: continue
            original = piece.place.facing
            best_facing = original
            
            # Score current state
            flow = SquadFlow.topology_only(self)
            best_score = _score_network(self, parts, flow=flow)
            
            for d in dirs:
                if d == original: continue
                piece.place.facing = d
                flow = SquadFlow.topology_only(self)
                s = _score_network(self, parts, flow=flow)
                if s > best_score:
                    best_score = s
                    best_facing = d
            
            piece.place.facing = best_facing
            if best_facing != original:
                changed = True
                total_changes += 1
        
        if not changed: break
    
    return total_changes


In [ ]:
@dataclass
class SquadTarget:
    type: TargetType
    hex_idx: int = -1         # -1 for CONSOLIDATE
    value: float = 0.0
    enemy_squad: str = ""
    turns_remaining: int = 5  # countdown within strategy cycle
    
    @property
    def label(self) -> str:
        if self.type == TargetType.CONSOLIDATE:
            return f"🏰 consolidate ({self.turns_remaining}t left)"
        if self.type == TargetType.KING:
            return f"⚔️ {self.enemy_squad} king @ {self.hex_idx} ({self.turns_remaining}t)"
        return f"🌾 oasis @ {self.hex_idx} ({self.turns_remaining}t)"
    
    @property
    def is_expired(self) -> bool:
        return self.turns_remaining <= 0
    
    def tick(self):
        self.turns_remaining = max(0, self.turns_remaining - 1)


In [ ]:
@patch
def choose_target(self: Squad, parts: GameParts, 
                  strategy: PlacementStrategy,
                  oasis_radius: int = 3) -> SquadTarget:
    """Pick target based on strategy personality + board state."""
    king = next((p for p in self.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return SquadTarget(TargetType.CONSOLIDATE)
    
    stance = strategy.assess(self, parts)
    my_hp = king.place.location
    
    # --- Strategy × stance → preferred target type ---
    PREFERENCE = {
        "aggressive": {"expand": "king",  "defend": "king",        "attack": "king"},
        "expansive":  {"expand": "oasis", "defend": "consolidate", "attack": "oasis"},
        "balanced":   {"expand": "oasis", "defend": "consolidate", "attack": "king"},
        "turtle":     {"expand": "consolidate", "defend": "consolidate", "attack": "oasis"},
    }
    preferred = PREFERENCE.get(strategy.name, PREFERENCE["balanced"])[stance]
    
    # --- Score candidate kings ---
    king_candidates = []
    for sq in parts.squads:
        if sq is self: continue
        eking = next((p for p in sq.pieces 
                      if p.type == PieceType.KING and p.place is not None), None)
        if not eking: continue
        eidx = parts.hp2i(eking.place.location)
        dist = my_hp.distance(eking.place.location)
        if dist == 0: continue
        placed = sum(1 for p in sq.pieces if p.place is not None)
        weakness = 1.0 / max(1, eking.place.pantry) + 1.0 / max(1, placed)
        score = weakness / dist
        king_candidates.append(SquadTarget(
            TargetType.KING, eidx, value=score, enemy_squad=sq.name))
    
    # --- Score candidate oases ---
    oasis_candidates = []
    tiers = parts.fy.tiers
    terr = parts.terr
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    for i in range(len(terr.elevations)):
        if terr.elevations[i] <= 0: continue
        ring = parts.grid.indices_in_range(i, oasis_radius)
        richness = sum(tiers[j] for j in ring 
                       if terr.elevations[j] > 0 and j not in occupied)
        if richness < 15: continue
        dist = my_hp.distance(parts.i2hp(i))
        if dist == 0: continue
        score = richness / dist
        oasis_candidates.append(SquadTarget(TargetType.OASIS, i, value=score))
    
    # --- Pick based on preference, with fallback ---
    if preferred == "king" and king_candidates:
        return max(king_candidates, key=lambda t: t.value)
    if preferred == "oasis" and oasis_candidates:
        return max(oasis_candidates, key=lambda t: t.value)
    if preferred == "consolidate":
        return SquadTarget(TargetType.CONSOLIDATE)
    
    # Fallback chain: oasis → consolidate
    if oasis_candidates:
        return max(oasis_candidates, key=lambda t: t.value)
    return SquadTarget(TargetType.CONSOLIDATE)


## Best Moves

In [ ]:
@patch
def best_pawn_move(self: Squad, parts: GameParts) -> Move | None:
    """Find the best pawn placement: highest yield hex in or near ally sight.
    
    Returns a Move with optional rotation + pawn placement, or None if no pawns available.
    """
    # Find an unplaced pawn
    pawn = next((p for p in self.pieces 
                 if p.type == PieceType.PAWN and p.place is None), None)
    if pawn is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    grid = parts.grid
    terr = parts.terr
    tiers = parts.fy.tiers
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # First: check all hexes visible from current sight (no rotation needed)
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    # Filter to valid land hexes not already occupied
    candidates = {idx for idx in current_visible 
                  if 0 <= idx < len(terr.elevations) 
                  and terr.elevations[idx] > 0 
                  and idx not in occupied}
    
    best_idx = None
    best_yield = -1
    best_rotation = None  # (piece, new_facing) if rotation needed
    
    # Check current sight first
    for idx in candidates:
        if tiers[idx] > best_yield:
            best_yield = tiers[idx]
            best_idx = idx
            best_rotation = None
    
    # Now try each piece × each facing to find even better options
    dirs = HexPosition.directions()
    for ally in placed:
        original_facing = ally.place.facing
        for new_facing in dirs:
            if new_facing == original_facing:
                continue
            # Temporarily compute sight with new facing
            bonus = ally.food.bonus_sight
            effective_sight = max(2, ally.place.level) + bonus
            new_sight = set(
                parts.hp2i(hp) 
                for hp in ally.place.location.field_of_view(new_facing, effective_sight)
            )
            for idx in new_sight:
                if (0 <= idx < len(terr.elevations) 
                    and terr.elevations[idx] > 0 
                    and idx not in occupied
                    and tiers[idx] > best_yield):
                    best_yield = tiers[idx]
                    best_idx = idx
                    best_rotation = Rotation(ally, new_facing)
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    rotations = [best_rotation] if best_rotation else []
    
    return Move.from_squad(self, rotations=rotations, 
                           placement=Placement(pawn, loc))


In [ ]:
@patch
def best_queen_move(self: Squad, parts: GameParts, 
                    stance: str = "defend") -> Move | None:
    """Place a queen strategically based on stance.
    
    stance='defend': prioritize king protection + ally coverage
    stance='attack': prioritize draining enemy pieces
    """
    queen = next((p for p in self.pieces 
                  if p.type == PieceType.QUEEN and p.place is None), None)
    if queen is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < queen.food.cost + 10:
        return None
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # Ally + enemy positions
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    enemy_idxs = set()
    for sq in parts.squads:
        if sq is self: continue
        for p in sq.pieces:
            if p.place is not None:
                enemy_idxs.add(parts.hp2i(p.place.location))
    
    king_hp = king.place.location if king else None
    
    # All valid placement hexes
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    candidates = {idx for idx in current_visible
                  if 0 <= idx < len(terr.elevations)
                  and terr.elevations[idx] > 0
                  and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    
    for idx in candidates:
        hp = parts.i2hp(idx)
        loc = parts.location_by_index(idx)
        bonus = queen.food.bonus_sight
        eff_sight = max(2, loc.level) + bonus
        
        # Distance to king (lower = better for defense)
        king_dist = hp.distance(king_hp) if king_hp else 0
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
            caught_enemies = len(enemy_idxs & cone)
            allies_covered = len(ally_idxs & cone)
            
            if stance == "defend":
                # Fortress: protect the king, cover allies, still punish nearby enemies
                score = (
                    allies_covered * 3.0
                    - king_dist * 4.0
                    + caught_enemies * queen.food.drain * 1.0
                    + (5.0 if king_hp and parts.hp2i(king_hp) in cone else 0)  # bonus: can see king
                )
            else:  # "attack" — you miss 100% of the shots you don't take
                score = (
                    caught_enemies * queen.food.drain * 3.0
                    + allies_covered * 1.0
                    - king_dist * 1.0  # still prefer not to be stranded
                )
            
            if score > best_score:
                best_score = score
                best_idx = idx
                best_facing = d
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    # Rotation if target isn't currently visible
    rotations = []
    if best_idx not in current_visible:
        for ally in placed:
            b = ally.food.bonus_sight
            eff = max(2, ally.place.level) + b
            for d in dirs:
                cone = set(parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
                if best_idx in cone:
                    rotations = [Rotation(ally, d)]
                    break
            if rotations:
                break
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(queen, loc))


In [ ]:
@patch
def best_bishop_move(self: Squad, parts: GameParts) -> Move | None:
    """Place a bishop using auto-detected role: relay, harvester, or frontier."""
    bishop = next((p for p in self.pieces 
                   if p.type == PieceType.BISHOP and p.place is None), None)
    if bishop is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < bishop.food.cost + 10:
        return None
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    
    # Current collective sight
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    # Land hex set for frontier scoring
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    
    # --- Auto-detect role weights ---
    G = self.to_nx()
    U = G.to_undirected() if len(G) > 1 else nx.Graph()
    has_bottleneck = len(list(nx.articulation_points(U))) > 0 if len(G) > 1 else False
    isolated = self.isolated_pieces()
    king_pantry = king.place.pantry if king else 0
    food_tight = king_pantry < 30
    
    if has_bottleneck or len(isolated) > 0:
        # Network fragile — bridge it
        w_connect, w_drain, w_harvest, w_frontier = 15, 1, 1, 3
        role = "relay"
    elif food_tight:
        # Running low — earn food
        w_connect, w_drain, w_harvest, w_frontier = 5, 1, 8, 1
        role = "harvester"
    else:
        # Healthy — expand vision for future placements
        w_connect, w_drain, w_harvest, w_frontier = 5, 3, 2, 6
        role = "frontier"
    
    candidates = {idx for idx in current_visible
                  if idx in land_set and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    
    for idx in candidates:
        hp = parts.i2hp(idx)
        loc = parts.location_by_index(idx)
        bonus = bishop.food.bonus_sight
        eff_sight = max(2, loc.level) + bonus
        tier = int(parts.fy.tiers[idx])
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
            
            # Connectivity: allies this bishop can see
            allies_seen = len(ally_idxs & cone)
            
            # Drain: enemies caught
            enemies_caught = len(enemy_idxs & cone)
            
            # Harvest: net food over 4-turn lifespan
            harvest_per_turn = bishop.food.harvest * tier - bishop.food.diet
            lifetime_net = harvest_per_turn * LIFESPAN[PieceType.BISHOP] - bishop.food.cost
            
            # Frontier: new land hexes revealed that nobody currently sees
            new_land = len((cone & land_set) - current_visible)
            
            score = (allies_seen * w_connect
                     + enemies_caught * bishop.food.drain * w_drain
                     + lifetime_net * w_harvest
                     + new_land * w_frontier
                     + loc.level * 0.5)  # slight high-ground tiebreak
            
            if score > best_score:
                best_score = score
                best_idx = idx
                best_facing = d
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    # Rotation if target isn't in current sight
    rotations = []
    if best_idx not in current_visible:
        for ally in placed:
            b = ally.food.bonus_sight
            eff = max(2, ally.place.level) + b
            for d in dirs:
                cone = set(parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
                if best_idx in cone:
                    rotations = [Rotation(ally, d)]
                    break
            if rotations:
                break
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(bishop, loc))


In [ ]:
@patch
def best_knight_move(self: Squad, parts: GameParts) -> Move | None:
    """Place a knight as a one-turn drain missile targeting enemy clusters."""
    knight = next((p for p in self.pieces 
                   if p.type == PieceType.KNIGHT and p.place is None), None)
    if knight is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < knight.food.cost + 5:
        return None  # don't fire blanks when we're broke
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # Build enemy position map with piece value for targeting priority
    PIECE_VALUE = {
        PieceType.KING: 20, PieceType.QUEEN: 10,
        PieceType.BISHOP: 4, PieceType.PAWN: 2, PieceType.KNIGHT: 1,
    }
    enemy_map = {}  # idx -> total value of enemies at that hex
    for sq in parts.squads:
        if sq is self:
            continue
        for p in sq.pieces:
            if p.place is not None:
                idx = parts.hp2i(p.place.location)
                enemy_map[idx] = enemy_map.get(idx, 0) + PIECE_VALUE.get(p.type, 1)
    
    if not enemy_map:
        return None  # no targets, save the food
    
    # Current collective sight
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    
    candidates = {idx for idx in current_visible
                  if idx in land_set and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    best_rotation = None
    
    # Also try with one rotation to unlock new hexes
    sight_configs = [(None, current_visible)]  # (rotation, visible_set)
    for ally in placed:
        original = ally.place.facing
        b = ally.food.bonus_sight
        eff = max(2, ally.place.level) + b
        for d in dirs:
            if d == original:
                continue
            new_cone = set(
                parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
            # Recompute collective sight with this rotation
            expanded = (current_visible - ally.sight().hexes) | new_cone
            sight_configs.append((Rotation(ally, d), expanded))
    
    for rotation, visible in sight_configs:
        cands = {idx for idx in visible
                 if idx in land_set and idx not in occupied}
        
        for idx in cands:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            eff_sight = max(2, loc.level)  # knight has 0 bonus_sight
            
            for d in dirs:
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
                
                # Total drain value: enemies caught weighted by importance
                drain_value = sum(enemy_map.get(eidx, 0) 
                                  for eidx in (cone & set(enemy_map.keys())))
                enemies_hit = len(cone & set(enemy_map.keys()))
                
                # ROI: is draining worth the 6 food cost?
                total_drain = enemies_hit * knight.food.drain
                roi = total_drain - knight.food.cost - knight.food.diet
                
                if roi <= 0:
                    continue  # not worth deploying
                
                score = (drain_value * knight.food.drain * 2.0
                         + roi * 1.5
                         + loc.level * 0.3)  # high ground = bigger cone
                
                if score > best_score:
                    best_score = score
                    best_idx = idx
                    best_facing = d
                    best_rotation = rotation
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    rotations = [best_rotation] if best_rotation else []
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(knight, loc))


So I have been adding purchasing so the game design is shifting from one where you  have a full squad to one where you build one. So we need a best purchase function (which might be none if we don't have the room in the pantry)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good pivot! Here's a `best_purchase` that picks both *what* to buy and *where* to put it, based on squad needs:

```python
@patch
def best_purchase(self: Squad, parts: GameParts) -> Move | None:
    """Pick the best piece to purchase and where to place it. Returns None if broke."""
    king = next((p for p in self.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king:
        return None
    
    pantry = king.place.pantry
    
    # What can we afford? (keep a reserve so king doesn't starve)
    RESERVE = 8
    affordable = {pt: cost for pt, cost in PURCHASE_COST.items()
                  if cost <= pantry - RESERVE}
    if not affordable:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    terr = parts.terr
    tiers = parts.fy.tiers
    dirs = HexPosition.directions()
    
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    
    # Current collective sight
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    
    valid_hexes = {idx for idx in current_visible
                   if idx in land_set and idx not in occupied}
    
    if not valid_hexes:
        return None
    
    # --- Assess what we need ---
    pawn_count = sum(1 for p in placed if p.type == PieceType.PAWN)
    bishop_count = sum(1 for p in placed if p.type == PieceType.BISHOP)
    knight_count = sum(1 for p in placed if p.type == PieceType.KNIGHT)
    queen_count = sum(1 for p in placed if p.type == PieceType.QUEEN)
    
    flow = SquadFlow.topology_only(self)
    has_bottleneck = len(flow.bottlenecks) > 0
    food_tight = pantry < 30
    enemies_nearby = len(enemy_idxs & current_visible) > 0
    
    # Priority weights per piece type based on situation
    weights: dict[PieceType, float] = {}
    
    if PieceType.PAWN in affordable:
        # Always useful early; less so when saturated
        w = 10.0 if pawn_count < 3 else 4.0
        if food_tight: w += 5.0  # cheap + harvests
        weights[PieceType.PAWN] = w
    
    if PieceType.BISHOP in affordable:
        w = 6.0 if bishop_count < 2 else 2.0
        if has_bottleneck: w += 4.0  # relay potential
        weights[PieceType.BISHOP] = w
    
    if PieceType.KNIGHT in affordable:
        w = 2.0
        if enemies_nearby: w += 8.0  # drain missile
        else: w = 0.5  # waste of food with no targets
        weights[PieceType.KNIGHT] = w
    
    if PieceType.QUEEN in affordable:
        w = 3.0 if queen_count == 0 else 1.0
        if enemies_nearby and pantry > 40: w += 5.0
        weights[PieceType.QUEEN] = w
    
    if not weights:
        return None
    
    king_hp = king.place.location
    
    # --- Score every (piece_type, hex, facing) triple ---
    best_score = -999
    best_pt = None
    best_idx = None
    best_facing = None
    
    for pt, type_weight in weights.items():
        stats = FOOD_TABLE[pt]
        cost = PURCHASE_COST[pt]
        lifespan = LIFESPAN.get(pt, 10)
        
        for idx in valid_hexes:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            tier = int(tiers[idx])
            bonus = stats.bonus_sight
            eff_sight = max(2, loc.level) + bonus
            king_dist = hp.distance(king_hp)
            
            for d in dirs:
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
                
                # Harvest ROI over lifespan
                harvest_per_turn = stats.harvest * tier - stats.diet
                lifetime_roi = harvest_per_turn * lifespan - cost
                
                # Connectivity
                allies_seen = len(ally_idxs & cone)
                king_visible = 1 if parts.hp2i(king_hp) in cone else 0
                
                # Drain value
                enemies_hit = len(enemy_idxs & cone)
                drain_value = enemies_hit * stats.drain
                
                # Frontier (new unseen land)
                new_land = len((cone & land_set) - current_visible)
                
                score = (
                    type_weight * 2.0
                    + lifetime_roi * 1.5
                    + allies_seen * 2.0
                    + king_visible * 4.0
                    + drain_value * 2.0
                    + new_land * 1.0
                    - king_dist * 0.5
                    + loc.level * 0.3
                )
                
                if score > best_score:
                    best_score = score
                    best_pt = pt
                    best_idx = idx
                    best_facing = d
    
    if best_pt is None or best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    
    return Move.from_squad(
        self,
        purchase=Purchase(best_pt, loc, best_facing))
```

Key design choices:

- **Needs assessment** — counts existing piece types + board state (bottlenecks, food pressure, enemy proximity) to weight *what* to buy
- **ROI-aware** — computes lifetime harvest minus cost, so it won't buy a pawn on barren land or a knight with no targets
- **RESERVE** — keeps 8 food in the king's pantry so a purchase doesn't immediately bankrupt you
- **Same sight constraint** — only places in currently visible hexes (validated by `Move.from_squad`)
- **Returns `None`** freely — caller should treat "no purchase" as a valid outcome and just do rotations or nothing

You could also split this into two calls in a turn — `best_purchase` for the buy, then a separate rotation pass — since `Move` supports both `purchase` and `rotations` together.

I think best_purchase should be tied to our strategies
  # --- Strategy × stance → preferred target type ---
    PREFERENCE = {
        "aggressive": {"expand": "king",  "defend": "king",        "attack": "king"},
        "expansive":  {"expand": "oasis", "defend": "consolidate", "attack": "oasis"},
        "balanced":   {"expand": "oasis", "defend": "consolidate", "attack": "king"},
        "turtle":     {"expand": "consolidate", "defend": "consolidate", "attack": "oasis"},
    }
    pr

I don't know if the their is a linear program with an objective function that would solve this, but that feels like the right direction - sort of a diet problem where we figure out what to eat. These prefercnes would change the objective function

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great intuition — it *is* a diet problem. You have **needs** (food income, drain power, connectivity, frontier vision) and **piece types** that supply those needs at different costs. The strategy rotates the objective function.

Here's the structure I'd propose:

```python
# What the board currently needs (computed from state)
@dataclass
class BoardNeeds:
    food_gap: float      # how much more harvest/turn we need
    drain_need: float    # enemy value in sight range
    connect_gap: float   # bottlenecks or isolated pieces
    frontier_need: float # fraction of nearby land unseen
    king_threat: float   # enemy drain pressure on our king

    @classmethod
    def assess(cls, squad: Squad, parts: GameParts, flow: SquadFlow) -> 'BoardNeeds':
        king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        if not king: return cls(0,0,0,0,0)
        
        placed = [p for p in squad.pieces if p.place is not None]
        
        # Food gap: are we losing food each turn?
        income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
        upkeep = sum(p.food.diet for p in placed)
        food_gap = max(0, upkeep - income + 2)  # want +2 surplus minimum
        
        # Drain need: how many enemy pieces are near us?
        current_visible = set()
        for p in placed: current_visible |= p.sight().hexes
        enemy_in_sight = 0
        for sq in parts.squads:
            if sq is squad: continue
            for p in sq.pieces:
                if p.place and parts.hp2i(p.place.location) in current_visible:
                    enemy_in_sight += 1
        
        # Connectivity gap
        connect_gap = len(flow.bottlenecks) + len(squad.isolated_pieces())
        
        # Frontier: how much unseen land near king?
        king_ring = set(parts.grid.indices_in_range(
            parts.hp2i(king.place.location), 6))
        land_ring = {i for i in king_ring if parts.terr.elevations[i] > 0}
        seen = land_ring & current_visible
        frontier_need = 1.0 - (len(seen) / max(1, len(land_ring)))
        
        # King threat: enemy drain on our king's hex
        king_idx = parts.hp2i(king.place.location)
        king_threat = 0.0
        for sq in parts.squads:
            if sq is squad: continue
            for p in sq.pieces:
                if p.place and king_idx in p.sight().hexes:
                    king_threat += p.food.drain
        
        return cls(food_gap, enemy_in_sight, connect_gap, frontier_need, king_threat)
```

Then strategy × stance just defines the **objective coefficients** on those needs:

```python
# strategy_name -> stance -> weight vector matching BoardNeeds fields
PURCHASE_OBJ: dict[str, dict[str, tuple[float,...]]] = {
    #                        food  drain  connect  frontier  threat
    "aggressive": {
        "expand":           (0.5,  3.0,   1.0,     1.0,      0.5),
        "defend":           (1.0,  2.0,   3.0,     0.5,      3.0),
        "attack":           (0.3,  5.0,   1.0,     0.5,      0.5),
    },
    "expansive": {
        "expand":           (1.5,  0.5,   2.0,     4.0,      0.5),
        "defend":           (2.0,  0.5,   3.0,     1.0,      2.0),
        "attack":           (1.0,  1.5,   1.5,     3.0,      0.5),
    },
    "balanced": {
        "expand":           (2.0,  1.0,   2.0,     2.0,      1.0),
        "defend":           (2.0,  1.0,   3.0,     1.0,      3.0),
        "attack":           (1.0,  3.0,   1.5,     1.0,      1.0),
    },
    "turtle": {
        "expand":           (3.0,  0.3,   4.0,     1.0,      1.0),
        "defend":           (3.0,  0.5,   4.0,     0.5,      4.0),
        "attack":           (2.0,  1.0,   3.0,     0.5,      2.0),
    },
}
```

And each **piece type** gets scored by how well it *fills* those needs at a given hex:

```python
@patch
def best_purchase(self: Squad, parts: GameParts, 
                  strategy_name: str = "balanced",
                  stance: str = "expand") -> Move | None:
    """LP-style purchase: maximize objective = needs · weights · supply."""
    king = next((p for p in self.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None
    
    pantry = king.place.pantry
    RESERVE = 8
    affordable = {pt: cost for pt, cost in PURCHASE_COST.items()
                  if cost <= pantry - RESERVE}
    if not affordable: return None
    
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy_name, PURCHASE_OBJ["balanced"])[stance]
    
    # Needs as vector
    n_vec = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                      needs.frontier_need, needs.king_threat])
    w_vec = np.array(weights)
    
    placed = [p for p in self.pieces if p.place is not None]
    dirs = HexPosition.directions()
    terr, tiers = parts.terr, parts.fy.tiers
    
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    king_hp = king.place.location
    
    current_visible = set()
    for ally in placed: current_visible |= ally.sight().hexes
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    valid_hexes = {idx for idx in current_visible
                   if idx in land_set and idx not in occupied}
    
    if not valid_hexes: return None
    
    best_score = -999
    best_pt, best_idx, best_facing = None, None, None
    
    for pt in affordable:
        stats = FOOD_TABLE[pt]
        cost = PURCHASE_COST[pt]
        lifespan = LIFESPAN.get(pt, 10)
        
        for idx in valid_hexes:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            tier = int(tiers[idx])
            eff_sight = max(2, loc.level) + stats.bonus_sight
            
            for d in dirs:
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
                
                # Supply vector: what this placement provides toward each need
                s_food = (stats.harvest * tier - stats.diet) * lifespan - cost
                s_drain = len(enemy_idxs & cone) * stats.drain
                s_connect = (len(ally_idxs & cone) 
                             + (2 if parts.hp2i(king_hp) in cone else 0))
                s_frontier = len((cone & land_set) - current_visible)
                s_threat = (stats.drain if parts.hp2i(king_hp) in cone 
                            else 0)  # defends king by draining attackers
                
                supply = np.array([s_food, s_drain, s_connect, s_frontier, s_threat])
                
                # Objective: needs · weights · supply
                score = float(n_vec * w_vec @ supply)
                
                if score > best_score:
                    best_score = score
                    best_pt, best_idx, best_facing = pt, idx, d
    
    if best_pt is None: return None
    
    loc = parts.location_by_index(best_idx)
    return Move.from_squad(self, purchase=Purchase(best_pt, loc, best_facing))
```

**Why this works like the diet problem:**

| Diet Problem | This |
|---|---|
| Foods (bread, milk...) | Piece types (pawn, bishop...) |
| Nutrients (protein, iron...) | Needs (food, drain, connectivity...) |
| Min daily requirements | `BoardNeeds` from board state |
| Cost to minimize | `PURCHASE_COST` + reserve |
| Dietary preference | Strategy × stance → `w_vec` |

The score is literally `needs · weights · supply` — a bilinear form where strategy rotates the weight vector. An aggressive player under attack cranks up drain weight; a turtle in expand mode cranks up food and connectivity.

The one simplification vs a true LP: we're buying **one** piece per turn, so it's argmax over candidates rather than solving a system. If you ever want multi-purchase turns, *that's* when you'd reach for `scipy.optimize.linprog`.

## Board Needs

In [ ]:
# What the board currently needs (computed from state)
@dataclass
class BoardNeeds:
    food_gap: float      # how much more harvest/turn we need
    drain_need: float    # enemy value in sight range
    connect_gap: float   # bottlenecks or isolated pieces
    frontier_need: float # fraction of nearby land unseen
    king_threat: float   # enemy drain pressure on our king

    @classmethod
    def assess(cls, squad: Squad, parts: GameParts, flow: SquadFlow) -> 'BoardNeeds':
        king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        if not king: return cls(0,0,0,0,0)
        
        placed = [p for p in squad.pieces if p.place is not None]
        
        # Food gap: are we losing food each turn?
        income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
        upkeep = sum(p.food.diet for p in placed)
        food_gap = max(0, upkeep - income + 2)  # want +2 surplus minimum
        
        # Drain need: how many enemy pieces are near us?
        current_visible = set()
        for p in placed: current_visible |= p.sight().hexes
        enemy_in_sight = 0
        for sq in parts.squads:
            if sq is squad: continue
            for p in sq.pieces:
                if p.place and parts.hp2i(p.place.location) in current_visible:
                    enemy_in_sight += 1
        
        # Connectivity gap
        connect_gap = len(flow.bottlenecks) + len(squad.isolated_pieces())
        
        # Frontier: how much unseen land near king?
        king_ring = set(parts.grid.indices_in_range(
            parts.hp2i(king.place.location), 6))
        land_ring = {i for i in king_ring if parts.terr.elevations[i] > 0}
        seen = land_ring & current_visible
        frontier_need = 1.0 - (len(seen) / max(1, len(land_ring)))
        
        # King threat: enemy drain on our king's hex
        king_idx = parts.hp2i(king.place.location)
        king_threat = 0.0
        for sq in parts.squads:
            if sq is squad: continue
            for p in sq.pieces:
                if p.place and king_idx in p.sight().hexes:
                    king_threat += p.food.drain
        
        return cls(food_gap, enemy_in_sight, connect_gap, frontier_need, king_threat)


In [ ]:
# strategy_name -> stance -> weight vector matching BoardNeeds fields
PURCHASE_OBJ: dict[str, dict[str, tuple[float,...]]] = {
    #                        food  drain  connect  frontier  threat
    "aggressive": {
        "expand":           (0.5,  3.0,   1.0,     1.0,      0.5),
        "defend":           (1.0,  2.0,   3.0,     0.5,      3.0),
        "attack":           (0.3,  5.0,   1.0,     0.5,      0.5),
    },
    "expansive": {
        "expand":           (1.5,  0.5,   2.0,     4.0,      0.5),
        "defend":           (2.0,  0.5,   3.0,     1.0,      2.0),
        "attack":           (1.0,  1.5,   1.5,     3.0,      0.5),
    },
    "balanced": {
        "expand":           (2.0,  1.0,   2.0,     2.0,      1.0),
        "defend":           (2.0,  1.0,   3.0,     1.0,      3.0),
        "attack":           (1.0,  3.0,   1.5,     1.0,      1.0),
    },
    "turtle": {
        "expand":           (3.0,  0.3,   4.0,     1.0,      1.0),
        "defend":           (3.0,  0.5,   4.0,     0.5,      4.0),
        "attack":           (2.0,  1.0,   3.0,     0.5,      2.0),
    },
}


@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy_name: str = "balanced",
                  stance: str = "expand") -> Move | None:
    """Two-phase purchase: LP picks the type, per-type logic picks the hex."""
    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None

    pantry = king.place.pantry
    RESERVE = 8
    affordable = {pt: cost for pt, cost in PURCHASE_COST.items()
                  if cost <= pantry - RESERVE}
    if not affordable: return None

    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy_name, PURCHASE_OBJ["balanced"])[stance]

    n_vec = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                      needs.frontier_need, needs.king_threat])
    w_vec = np.array(weights)

    # --- Phase 1: score each piece TYPE by how well it addresses needs ---
    # Capability profile per type (how much of each need it can serve)
    CAPABILITY: dict[PieceType, tuple[float,...]] = {
        #                    food   drain  connect  frontier  threat
        PieceType.PAWN:     (3.0,   0.5,   1.0,     0.5,      0.3),
        PieceType.BISHOP:   (1.5,   0.5,   3.0,     3.0,      0.5),
        PieceType.KNIGHT:   (0.0,   4.0,   0.5,     0.5,      3.0),
        PieceType.QUEEN:    (0.0,   3.5,   1.5,     1.5,      3.5),
    }

    type_scores = {}
    for pt in affordable:
        cap = np.array(CAPABILITY[pt])
        type_scores[pt] = float(n_vec * w_vec @ cap)

    # Rank types by score, try best first
    ranked_types = sorted(type_scores, key=type_scores.get, reverse=True)

    # --- Phase 2: for top type, find best placement using per-type logic ---
    PLACEMENT_FN = {
        PieceType.PAWN:   Squad.best_pawn_move,
        PieceType.BISHOP: Squad.best_bishop_move,
        PieceType.KNIGHT: Squad.best_knight_move,
        PieceType.QUEEN:  Squad.best_queen_move,
    }

    for pt in ranked_types:
        if type_scores[pt] <= 0:
            continue

        # Temporarily add a ghost piece so placement logic can find a spot
        ghost = Piece(
            id='_ghost_', flag=self.flag, type=pt, name='_ghost_',
            year=self.year)
        ghost.lantern = StyleCSS("ghost_lantern", fill=self.flag.primary, opacity=0.1)
        self.pieces.append(ghost)

        try:
            fn = PLACEMENT_FN.get(pt)
            if fn is None:
                continue
            move = fn(self, parts)
            if move is None or move.placement is None:
                continue

            # Convert the placement into a purchase
            loc = move.placement.location
            facing = loc.facing
            return Move.from_squad(
                self,
                rotations=move.rotations,
                purchase=Purchase(pt, loc, facing))
        except MoveError:
            continue
        finally:
            self.pieces.remove(ghost)

    return None


# !pip install pyomo
# Uses GLPK solver (free) — or CBC
# !apt-get install -y glpk-utils  # if needed

from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint,
    Binary, maximize, SolverFactory, value
)

@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy: PlacementStrategy = None,
                  stance: str = "expand") -> Move | None:
    """MIP purchase: Pyomo model with explicit budget constraints."""
    strategy = strategy or PlacementStrategy.BALANCED
    strategy_name = strategy.name
    
    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None
    
    pantry = king.place.pantry
    placed = [p for p in self.pieces if p.place is not None]
    n_placed = len(placed)
    
    # Precompute board state
    terr, tiers = parts.terr, parts.fy.tiers
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    king_hp = king.place.location
    
    current_visible = set()
    for ally in placed: current_visible |= ally.sight().hexes
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    valid_hexes = [idx for idx in current_visible
                   if idx in land_set and idx not in occupied]
    
    if not valid_hexes: return None
    
    # Current economics
    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)
    
    # LP weights from strategy × stance
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy_name, PURCHASE_OBJ["balanced"])[stance]
    n_vec = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                      needs.frontier_need, needs.king_threat])
    w_vec = np.array(weights)
    
    # ── Precompute supply scores for every (type, hex, dir) ──
    affordable_types = [pt for pt, cost in PURCHASE_COST.items()
                        if cost <= pantry - 1]  # loose filter; Pyomo enforces budget
    if not affordable_types: return None
    
    options = {}  # (pt_val, hex_idx, dir_idx) -> {score, cost, diet, harvest_net, ...}
    
    for pt in affordable_types:
        fs = FOOD_TABLE[pt]
        cost = PURCHASE_COST[pt]
        lifespan = LIFESPAN.get(pt, 10)
        
        for idx in valid_hexes:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            tier = int(tiers[idx])
            eff_sight = max(2, loc.level) + fs.bonus_sight
            
            for di, d in enumerate(dirs):
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
                
                # Supply vector
                s_food = (fs.harvest * tier - fs.diet) * lifespan - cost
                s_drain = len(enemy_idxs & cone) * fs.drain
                s_connect = (len(ally_idxs & cone)
                             + (2 if parts.hp2i(king_hp) in cone else 0))
                s_frontier = len((cone & land_set) - current_visible)
                s_threat = fs.drain if parts.hp2i(king_hp) in cone else 0
                
                supply = np.array([s_food, s_drain, s_connect, s_frontier, s_threat])
                score = float(n_vec * w_vec @ supply)
                
                harvest_net = int(fs.harvest * tier) - fs.diet
                
                key = (pt.value, idx, di)
                options[key] = dict(
                    score=score,
                    cost=cost,
                    diet=fs.diet,
                    harvest_net=harvest_net,
                    pt=pt, idx=idx, di=di,
                )
    
    if not options: return None
    
    # ── Build Pyomo model ──
    m = ConcreteModel()
    
    m.OPTIONS = Set(initialize=list(options.keys()))
    m.score   = Param(m.OPTIONS, initialize={k: v['score'] for k, v in options.items()})
    m.cost    = Param(m.OPTIONS, initialize={k: v['cost'] for k, v in options.items()})
    m.diet    = Param(m.OPTIONS, initialize={k: v['diet'] for k, v in options.items()})
    m.h_net   = Param(m.OPTIONS, initialize={k: v['harvest_net'] for k, v in options.items()})
    
    # Decision: pick at most one option (or skip)
    m.x = Var(m.OPTIONS, domain=Binary)
    
    # ── Objective: maximize weighted score ──
    m.obj = Objective(
        expr=sum(m.x[k] * m.score[k] for k in m.OPTIONS),
        sense=maximize)
    
    # ── C1: At most one purchase ──
    m.one_purchase = Constraint(
        expr=sum(m.x[k] for k in m.OPTIONS) <= 1)
    
    # ── C2: Budget floor — keep dynamic reserve ──
    reserve = n_placed * strategy.min_reserve_per_piece
    m.budget = Constraint(
        expr=sum(m.x[k] * m.cost[k] for k in m.OPTIONS) <= pantry - reserve)
    
    # ── C3: Sustainability — post-purchase income/upkeep ratio ──
    # income + new_harvest >= min_ratio * (upkeep + new_diet)
    min_ratio = strategy.min_budget_ratio
    m.sustainability = Constraint(
        expr=(income + sum(m.x[k] * m.h_net[k] for k in m.OPTIONS)
              >= min_ratio * upkeep - income))
    # Rearranged: income + harvest_net >= min_ratio * upkeep - income
    # Actually cleaner as: (income + harvest_net) / (upkeep + diet) >= ratio
    # But that's nonlinear, so linearize:
    # income + sum(x*h_net) >= min_ratio * (upkeep + sum(x*diet))
    
    # Let me redo this properly:
    m.del_component(m.sustainability)
    m.sustainability = Constraint(
        expr=(income + sum(m.x[k] * m.h_net[k] for k in m.OPTIONS)
              + sum(m.x[k] * m.diet[k] for k in m.OPTIONS)
              >= min_ratio * (upkeep + sum(m.x[k] * m.diet[k] for k in m.OPTIONS))))
    
    # ── C4: Don't buy if already bleeding and low treasury ──
    if income < upkeep and pantry < reserve * 2:
        # Only allow net-positive placements
        for k in m.OPTIONS:
            if options[k]['harvest_net'] <= 0:
                m.x[k].fix(0)
    
    # ── Solve ──
    solver = SolverFactory('glpk')
    result = solver.solve(m, tee=False)
    
    # ── Extract solution ──
    chosen = None
    for k in m.OPTIONS:
        if value(m.x[k]) > 0.5:
            chosen = options[k]
            break
    
    if chosen is None:
        return None  # solver said skip (all constraints too tight)
    
    pt = chosen['pt']
    idx = chosen['idx']
    facing = dirs[chosen['di']]
    loc = parts.location_by_index(idx)
    
    return Move.from_squad(self, purchase=Purchase(pt, loc, facing))


@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy: 'PlacementStrategy' = None,
                  stance: str = "expand") -> Move | None:
    """Purchase: strategy-aware argmax over (type, hex, facing) options."""
    strategy = strategy or PlacementStrategy.BALANCED
    strategy_name = strategy.name

    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None

    placed = [p for p in self.pieces if p.place is not None]

    # ── Strategy-gated affordability check ──
    affordable_types = [pt for pt, cost in PURCHASE_COST.items()
                        if strategy.can_afford(self, parts, cost)]
    if not affordable_types: return None

    # ── Board state ──
    terr, tiers = parts.terr, parts.fy.tiers
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    king_hp = king.place.location
    king_idx = parts.hp2i(king_hp)

    current_visible = set()
    for ally in placed: current_visible |= ally.sight().hexes
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    valid_hexes = [idx for idx in current_visible
                   if idx in land_set and idx not in occupied]
    if not valid_hexes: return None

    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)

    # ── Objective weights from strategy × stance ──
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy_name, PURCHASE_OBJ["balanced"])[stance]
    n_vec = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                      needs.frontier_need, needs.king_threat])
    w_vec = np.array(weights)

    # ── Precompute supply scores ──
    options = {}  # (pt_val, hex_idx, dir_idx) -> dict

    for pt in affordable_types:
        fs = FOOD_TABLE[pt]
        cost = PURCHASE_COST[pt]
        lifespan = LIFESPAN.get(pt, 10)

        for idx in valid_hexes:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            tier = int(tiers[idx])
            eff_sight = max(2, loc.level) + fs.bonus_sight

            for di, d in enumerate(dirs):
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))

                s_food     = (fs.harvest * tier - fs.diet) * lifespan - cost
                s_drain    = len(enemy_idxs & cone) * fs.drain
                s_connect  = len(ally_idxs & cone) + (2 if king_idx in cone else 0)
                s_frontier = len((cone & land_set) - current_visible)
                s_threat   = fs.drain if king_idx in cone else 0.0

                supply = np.array([s_food, s_drain, s_connect, s_frontier, s_threat])
                score  = float(n_vec * w_vec @ supply)
                harvest_net = int(fs.harvest * tier) - fs.diet

                # Under threat, skip pieces that can't drain back
                if needs.king_threat > 2.0 and fs.drain == 0:
                    continue

                # Sustainability gate: post-purchase income must cover ratio
                min_ratio = strategy.min_budget_ratio
                if upkeep > 0:
                    post_income = income + harvest_net
                    post_upkeep = upkeep + fs.diet
                    if post_income < min_ratio * post_upkeep:
                        continue

                options[(pt.value, idx, di)] = dict(
                    score=score, cost=cost,
                    diet=fs.diet, harvest_net=harvest_net,
                    pt=pt, idx=idx, di=di,
                )

    if not options: return None

    # ── Argmax: pick highest-scoring affordable option ──
    _, chosen = max(
        ((k, v) for k, v in options.items()
         if strategy.can_afford(self, parts, v['cost'])),
        key=lambda kv: kv[1]['score'],
        default=(None, None)
    )
    if chosen is None: return None

    loc = parts.location_by_index(chosen['idx'])
    facing = dirs[chosen['di']]
    return Move.from_squad(self, purchase=Purchase(chosen['pt'], loc, facing))


@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy: 'PlacementStrategy' = None,
                  stance: str = "expand") -> Move | None:
    """MIP purchase: Pyomo model with strategy-aware budget constraints."""
    strategy = strategy or PlacementStrategy.BALANCED
    strategy_name = strategy.name

    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None

    pantry = king.place.pantry
    placed = [p for p in self.pieces if p.place is not None]

    # ── Strategy-gated affordability check ──
    affordable_types = [pt for pt, cost in PURCHASE_COST.items()
                        if strategy.can_afford(self, parts, cost)]
    if not affordable_types: return None

    # ── Board state ──
    terr, tiers = parts.terr, parts.fy.tiers
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    king_hp = king.place.location
    king_idx = parts.hp2i(king_hp)

    current_visible = set()
    for ally in placed: current_visible |= ally.sight().hexes
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    valid_hexes = [idx for idx in current_visible
                   if idx in land_set and idx not in occupied]
    if not valid_hexes: return None

    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)

    # ── Objective weights from strategy × stance ──
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy_name, PURCHASE_OBJ["balanced"])[stance]
    n_vec = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                      needs.frontier_need, needs.king_threat])
    w_vec = np.array(weights)

    # ── Precompute supply scores ──
    options = {}  # (pt_val, hex_idx, dir_idx) -> dict

    for pt in affordable_types:
        fs = FOOD_TABLE[pt]
        cost = PURCHASE_COST[pt]
        lifespan = LIFESPAN.get(pt, 10)

        for idx in valid_hexes:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            tier = int(tiers[idx])
            eff_sight = max(2, loc.level) + fs.bonus_sight

            for di, d in enumerate(dirs):
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))

                s_food     = (fs.harvest * tier - fs.diet) * lifespan - cost
                s_drain    = len(enemy_idxs & cone) * fs.drain
                s_connect  = len(ally_idxs & cone) + (2 if king_idx in cone else 0)
                s_frontier = len((cone & land_set) - current_visible)
                s_threat   = fs.drain if king_idx in cone else 0.0

                supply = np.array([s_food, s_drain, s_connect, s_frontier, s_threat])
                score  = float(n_vec * w_vec @ supply)
                harvest_net = int(fs.harvest * tier) - fs.diet

                options[(pt.value, idx, di)] = dict(
                    score=score, cost=cost,
                    diet=fs.diet, harvest_net=harvest_net,
                    pt=pt, idx=idx, di=di,
                )

    if not options: return None

    # ── Pyomo MIP ──
    m = ConcreteModel()
    m.OPTIONS = Set(initialize=list(options.keys()))
    m.score   = Param(m.OPTIONS, initialize={k: v['score']       for k, v in options.items()})
    m.cost    = Param(m.OPTIONS, initialize={k: v['cost']        for k, v in options.items()})
    m.diet    = Param(m.OPTIONS, initialize={k: v['diet']        for k, v in options.items()})
    m.h_net   = Param(m.OPTIONS, initialize={k: v['harvest_net'] for k, v in options.items()})
    m.x       = Var(m.OPTIONS, domain=Binary)

    # Objective
    m.obj = Objective(
        expr=sum(m.x[k] * m.score[k] for k in m.OPTIONS),
        sense=maximize)

    # C1: at most one purchase
    m.one_purchase = Constraint(
        expr=sum(m.x[k] for k in m.OPTIONS) <= 1)

    # C2: strategy-defined reserve floor
    reserve = len(placed) * strategy.min_reserve_per_piece
    m.budget = Constraint(
        expr=sum(m.x[k] * m.cost[k] for k in m.OPTIONS) <= pantry - reserve)

    # C3: post-purchase sustainability (linearised ratio)
    # (income + harvest_net) >= min_ratio * (upkeep + new_diet)
    min_ratio = strategy.min_budget_ratio
    m.sustainability = Constraint(
        expr=(income + sum(m.x[k] * m.h_net[k] for k in m.OPTIONS)
              >= min_ratio * (upkeep + sum(m.x[k] * m.diet[k] for k in m.OPTIONS))))

    # C4: under threat, only buy pieces that can drain in return
    if needs.king_threat > 2.0:
        for k, v in options.items():
            if FOOD_TABLE[v['pt']].drain == 0:
                m.x[k].fix(0)

    # ── Solve ──
    result = SolverFactory('glpk').solve(m, tee=False)

    chosen = next((options[k] for k in m.OPTIONS if value(m.x[k]) > 0.5), None)
    if chosen is None: return None

    loc = parts.location_by_index(chosen['idx'])
    facing = dirs[chosen['di']]
    return Move.from_squad(self, purchase=Purchase(chosen['pt'], loc, facing))


In [ ]:
@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy: 'PlacementStrategy' = None,
                  stance: str = "expand") -> Move | None:
    """Purchase: strategy-aware argmax over (type, hex, facing) options."""
    strategy = strategy or PlacementStrategy.BALANCED
    strategy_name = strategy.name

    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None

    placed = [p for p in self.pieces if p.place is not None]

    affordable_types = [pt for pt, cost in PURCHASE_COST.items()
                        if strategy.can_afford(self, parts, cost)]
    if not affordable_types: return None

    terr, tiers = parts.terr, parts.fy.tiers
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    king_hp = king.place.location
    king_idx = parts.hp2i(king_hp)

    current_visible = set()
    # Also track which allies can see each hex (for connectivity credit)
    ally_sight_map: dict[int, set] = {}  # hex_idx -> set of ally piece ids
    for ally in placed:
        cone = ally.sight().hexes
        current_visible |= cone
        for h in cone:
            ally_sight_map.setdefault(h, set()).add(ally.id)

    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    valid_hexes = [idx for idx in current_visible
                   if idx in land_set and idx not in occupied]
    if not valid_hexes: return None

    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)
    n_placed = len(placed)

    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy_name, PURCHASE_OBJ["balanced"])[stance]
    n_vec = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                      needs.frontier_need, needs.king_threat])
    w_vec = np.array(weights)

    # Relax sustainability early game — ratio only meaningful once we have income
    min_ratio = strategy.min_budget_ratio if income > 0 else 0.0

    options = {}

    for pt in affordable_types:
        fs = FOOD_TABLE[pt]
        cost = PURCHASE_COST[pt]
        lifespan = LIFESPAN.get(pt, 10)

        for idx in valid_hexes:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            tier = int(tiers[idx])
            eff_sight = max(2, loc.level) + fs.bonus_sight

            for di, d in enumerate(dirs):
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))

                s_food     = (fs.harvest * tier - fs.diet) * lifespan - cost
                s_drain    = len(enemy_idxs & cone) * fs.drain

                # Bidirectional connectivity:
                # - allies the new piece can see (it sees them)
                # - allies that can see the new hex (they see it)
                allies_seen_by_new = len(ally_idxs & cone)
                allies_that_see_new = len(ally_sight_map.get(idx, set()))
                king_visible = 2 if king_idx in cone else 0
                s_connect = allies_seen_by_new + allies_that_see_new + king_visible

                s_frontier = len((cone & land_set) - current_visible)
                s_threat   = fs.drain if king_idx in cone else 0.0

                supply = np.array([s_food, s_drain, s_connect, s_frontier, s_threat])
                score  = float(n_vec * w_vec @ supply)
                harvest_net = int(fs.harvest * tier) - fs.diet

                # Under threat, skip pieces that can't drain back
                if needs.king_threat > 2.0 and fs.drain == 0:
                    continue

                # Sustainability gate (relaxed early game)
                if min_ratio > 0 and upkeep > 0:
                    post_upkeep = upkeep + fs.diet
                    if income + harvest_net < min_ratio * post_upkeep:
                        continue

                options[(pt.value, idx, di)] = dict(
                    score=score, cost=cost,
                    diet=fs.diet, harvest_net=harvest_net,
                    pt=pt, idx=idx, di=di,
                )

    if not options: return None

    _, chosen = max(
        ((k, v) for k, v in options.items()
         if strategy.can_afford(self, parts, v['cost'])),
        key=lambda kv: kv[1]['score'],
        default=(None, None)
    )
    if chosen is None: return None

    loc = parts.location_by_index(chosen['idx'])
    facing = dirs[chosen['di']]
    return Move.from_squad(self, purchase=Purchase(chosen['pt'], loc, facing))


In [ ]:
@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy: 'PlacementStrategy' = None,
                  stance: str = "expand") -> Move | None:
    """Purchase: score all (type, hex, facing) options, with sane budget gates."""
    strategy = strategy or PlacementStrategy.BALANCED

    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None

    placed      = [p for p in self.pieces if p.place is not None]
    n_placed    = len(placed)
    pantry      = king.place.pantry
    income      = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep      = sum(p.food.diet for p in placed)
    net         = income - upkeep

    # ── Budget gate ──────────────────────────────────────────────────────────
    reserve = n_placed * strategy.min_reserve_per_piece
    if pantry <= reserve:
        return None  # can't afford anything safely

    # Cooldown gate
    last_buy = getattr(self, '_last_purchase_turn', -999)
    current_turn = getattr(parts, 'turn', 0)
    if current_turn - last_buy < strategy.buy_cooldown:
        return None

    # ── Board state ──────────────────────────────────────────────────────────
    terr, tiers = parts.terr, parts.fy.tiers
    dirs        = HexPosition.directions()
    occupied    = {parts.hp2i(p.place.location)
                   for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs  = {parts.hp2i(p.place.location)
                   for sq in parts.squads if sq is not self
                   for p in sq.pieces if p.place is not None}
    ally_idxs   = {parts.hp2i(p.place.location) for p in placed}
    king_idx    = parts.hp2i(king.place.location)

    current_visible: set[int] = set()
    ally_sight_map:  dict[int, int] = {}   # hex → count of allies that see it
    for ally in placed:
        for h in ally.sight().hexes:
            current_visible.add(h)
            ally_sight_map[h] = ally_sight_map.get(h, 0) + 1

    land_set    = {i for i, e in enumerate(terr.elevations) if e > 0}
    valid_hexes = [i for i in current_visible if i in land_set and i not in occupied]
    if not valid_hexes: return None

    # ── Needs vector ─────────────────────────────────────────────────────────
    flow   = SquadFlow.topology_only(self)
    needs  = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy.name, PURCHASE_OBJ["balanced"])[stance]
    n_vec  = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                       needs.frontier_need, needs.king_threat])
    w_vec  = np.array(weights)

    # ── Sustainability threshold ──────────────────────────────────────────────
    # Only enforce if we're currently sustainable — don't lock a losing squad out
    # of the one purchase that could save it
    enforce_sustainability = (income >= upkeep) and (n_placed > 1)
    min_ratio = strategy.min_budget_ratio if enforce_sustainability else 0.0

    # ── Score all options ─────────────────────────────────────────────────────
    best_score  = -1e9  # must beat this — ensures "buy nothing" wins if all < 0
    best_option = None

    for pt, fs in FOOD_TABLE.items():
        if pt == PieceType.KING: continue
        cost = PURCHASE_COST[pt]
        if cost > pantry - reserve: continue            # hard budget
        lifespan = LIFESPAN.get(pt, 10)

        for idx in valid_hexes:
            hp   = parts.i2hp(idx)
            loc  = parts.location_by_index(idx)
            tier = int(tiers[idx])
            eff_sight = max(2, loc.level) + fs.bonus_sight
            harvest_net = int(fs.harvest * tier) - fs.diet

            # Sustainability filter — skip if it makes economics worse
            if min_ratio > 0 and upkeep > 0:
                if income + harvest_net < min_ratio * (upkeep + fs.diet):
                    continue

            # Conservative strategies: only buy net-positive pieces
            if strategy.min_budget_ratio >= 1.0 and harvest_net <= 0:
                continue

            # Threat filter: only skip drain=0 if we already have defenders
            if needs.king_threat > 2.0 and fs.drain == 0 and n_placed > 3:
                continue

            for di, d in enumerate(dirs):
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))

                s_food     = (fs.harvest * tier - fs.diet) * lifespan - cost
                s_drain    = len(enemy_idxs & cone) * fs.drain
                s_connect  = (len(ally_idxs & cone)
                              + ally_sight_map.get(idx, 0)
                              + (2 if king_idx in cone else 0))
                s_frontier = len((cone & land_set) - current_visible)
                s_threat   = float(fs.drain if king_idx in cone else 0)

                supply = np.array([s_food, s_drain, s_connect, s_frontier, s_threat])
                score  = float(n_vec * w_vec @ supply)

                if score > best_score:
                    best_score  = score
                    best_option = (pt, idx, di, cost)

    # Require positive value — "buy nothing" always beats a bad purchase
    if best_option is None or best_score <= 0:
        return None

    pt, idx, di, _ = best_option
    self._last_purchase_turn = current_turn
    return Move.from_squad(
        self, purchase=Purchase(pt, parts.location_by_index(idx), dirs[di]))


So best purchase now doesn't recommend buying anything because it doesn't see how the pawns for instance will add to the network.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The sustainability gate is too aggressive early on, and `s_connect` only counts allies the new piece can *see*, not whether allies can see *it* (which is what actually grows the network). Here's the fix:

```python
@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy: 'PlacementStrategy' = None,
                  stance: str = "expand") -> Move | None:
    """Purchase: strategy-aware argmax over (type, hex, facing) options."""
    strategy = strategy or PlacementStrategy.BALANCED
    strategy_name = strategy.name

    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None

    placed = [p for p in self.pieces if p.place is not None]

    affordable_types = [pt for pt, cost in PURCHASE_COST.items()
                        if strategy.can_afford(self, parts, cost)]
    if not affordable_types: return None

    terr, tiers = parts.terr, parts.fy.tiers
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    king_hp = king.place.location
    king_idx = parts.hp2i(king_hp)

    current_visible = set()
    # Also track which allies can see each hex (for connectivity credit)
    ally_sight_map: dict[int, set] = {}  # hex_idx -> set of ally piece ids
    for ally in placed:
        cone = ally.sight().hexes
        current_visible |= cone
        for h in cone:
            ally_sight_map.setdefault(h, set()).add(ally.id)

    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    valid_hexes = [idx for idx in current_visible
                   if idx in land_set and idx not in occupied]
    if not valid_hexes: return None

    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)
    n_placed = len(placed)

    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy_name, PURCHASE_OBJ["balanced"])[stance]
    n_vec = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                      needs.frontier_need, needs.king_threat])
    w_vec = np.array(weights)

    # Relax sustainability early game — ratio only meaningful once we have income
    min_ratio = strategy.min_budget_ratio if income > 0 else 0.0

    options = {}

    for pt in affordable_types:
        fs = FOOD_TABLE[pt]
        cost = PURCHASE_COST[pt]
        lifespan = LIFESPAN.get(pt, 10)

        for idx in valid_hexes:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            tier = int(tiers[idx])
            eff_sight = max(2, loc.level) + fs.bonus_sight

            for di, d in enumerate(dirs):
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))

                s_food     = (fs.harvest * tier - fs.diet) * lifespan - cost
                s_drain    = len(enemy_idxs & cone) * fs.drain

                # Bidirectional connectivity:
                # - allies the new piece can see (it sees them)
                # - allies that can see the new hex (they see it)
                allies_seen_by_new = len(ally_idxs & cone)
                allies_that_see_new = len(ally_sight_map.get(idx, set()))
                king_visible = 2 if king_idx in cone else 0
                s_connect = allies_seen_by_new + allies_that_see_new + king_visible

                s_frontier = len((cone & land_set) - current_visible)
                s_threat   = fs.drain if king_idx in cone else 0.0

                supply = np.array([s_food, s_drain, s_connect, s_frontier, s_threat])
                score  = float(n_vec * w_vec @ supply)
                harvest_net = int(fs.harvest * tier) - fs.diet

                # Under threat, skip pieces that can't drain back
                if needs.king_threat > 2.0 and fs.drain == 0:
                    continue

                # Sustainability gate (relaxed early game)
                if min_ratio > 0 and upkeep > 0:
                    post_upkeep = upkeep + fs.diet
                    if income + harvest_net < min_ratio * post_upkeep:
                        continue

                options[(pt.value, idx, di)] = dict(
                    score=score, cost=cost,
                    diet=fs.diet, harvest_net=harvest_net,
                    pt=pt, idx=idx, di=di,
                )

    if not options: return None

    _, chosen = max(
        ((k, v) for k, v in options.items()
         if strategy.can_afford(self, parts, v['cost'])),
        key=lambda kv: kv[1]['score'],
        default=(None, None)
    )
    if chosen is None: return None

    loc = parts.location_by_index(chosen['idx'])
    facing = dirs[chosen['di']]
    return Move.from_squad(self, purchase=Purchase(chosen['pt'], loc, facing))
```

The two key changes:

1. **Bidirectional `s_connect`** — `ally_sight_map` precomputes which allies can see each hex, so placing a pawn *within the king's cone* gets credit for the connection, not just for what the pawn itself can see.

2. **`min_ratio = 0.0` when `income == 0`** — early game with only the king placed, `income=0` means the sustainability ratio is always violated. We skip the gate entirely until there's meaningful income to protect.

can you rewrite the best_purchase with can_afford? Add any improvements you think it could use.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy: 'PlacementStrategy' = None,
                  stance: str = "expand") -> Move | None:
    """MIP purchase: Pyomo model with strategy-aware budget constraints."""
    strategy = strategy or PlacementStrategy.BALANCED
    strategy_name = strategy.name

    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None

    pantry = king.place.pantry
    placed = [p for p in self.pieces if p.place is not None]

    # ── Strategy-gated affordability check ──
    affordable_types = [pt for pt, cost in PURCHASE_COST.items()
                        if strategy.can_afford(self, parts, cost)]
    if not affordable_types: return None

    # ── Board state ──
    terr, tiers = parts.terr, parts.fy.tiers
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    king_hp = king.place.location
    king_idx = parts.hp2i(king_hp)

    current_visible = set()
    for ally in placed: current_visible |= ally.sight().hexes
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    valid_hexes = [idx for idx in current_visible
                   if idx in land_set and idx not in occupied]
    if not valid_hexes: return None

    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)

    # ── Objective weights from strategy × stance ──
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy_name, PURCHASE_OBJ["balanced"])[stance]
    n_vec = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                      needs.frontier_need, needs.king_threat])
    w_vec = np.array(weights)

    # ── Precompute supply scores ──
    options = {}  # (pt_val, hex_idx, dir_idx) -> dict

    for pt in affordable_types:
        fs = FOOD_TABLE[pt]
        cost = PURCHASE_COST[pt]
        lifespan = LIFESPAN.get(pt, 10)

        for idx in valid_hexes:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            tier = int(tiers[idx])
            eff_sight = max(2, loc.level) + fs.bonus_sight

            for di, d in enumerate(dirs):
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))

                s_food     = (fs.harvest * tier - fs.diet) * lifespan - cost
                s_drain    = len(enemy_idxs & cone) * fs.drain
                s_connect  = len(ally_idxs & cone) + (2 if king_idx in cone else 0)
                s_frontier = len((cone & land_set) - current_visible)
                s_threat   = fs.drain if king_idx in cone else 0.0

                supply = np.array([s_food, s_drain, s_connect, s_frontier, s_threat])
                score  = float(n_vec * w_vec @ supply)
                harvest_net = int(fs.harvest * tier) - fs.diet

                options[(pt.value, idx, di)] = dict(
                    score=score, cost=cost,
                    diet=fs.diet, harvest_net=harvest_net,
                    pt=pt, idx=idx, di=di,
                )

    if not options: return None

    # ── Pyomo MIP ──
    m = ConcreteModel()
    m.OPTIONS = Set(initialize=list(options.keys()))
    m.score   = Param(m.OPTIONS, initialize={k: v['score']       for k, v in options.items()})
    m.cost    = Param(m.OPTIONS, initialize={k: v['cost']        for k, v in options.items()})
    m.diet    = Param(m.OPTIONS, initialize={k: v['diet']        for k, v in options.items()})
    m.h_net   = Param(m.OPTIONS, initialize={k: v['harvest_net'] for k, v in options.items()})
    m.x       = Var(m.OPTIONS, domain=Binary)

    # Objective
    m.obj = Objective(
        expr=sum(m.x[k] * m.score[k] for k in m.OPTIONS),
        sense=maximize)

    # C1: at most one purchase
    m.one_purchase = Constraint(
        expr=sum(m.x[k] for k in m.OPTIONS) <= 1)

    # C2: strategy-defined reserve floor
    reserve = len(placed) * strategy.min_reserve_per_piece
    m.budget = Constraint(
        expr=sum(m.x[k] * m.cost[k] for k in m.OPTIONS) <= pantry - reserve)

    # C3: post-purchase sustainability (linearised ratio)
    # (income + harvest_net) >= min_ratio * (upkeep + new_diet)
    min_ratio = strategy.min_budget_ratio
    m.sustainability = Constraint(
        expr=(income + sum(m.x[k] * m.h_net[k] for k in m.OPTIONS)
              >= min_ratio * (upkeep + sum(m.x[k] * m.diet[k] for k in m.OPTIONS))))

    # C4: under threat, only buy pieces that can drain in return
    if needs.king_threat > 2.0:
        for k, v in options.items():
            if FOOD_TABLE[v['pt']].drain == 0:
                m.x[k].fix(0)

    # ── Solve ──
    result = SolverFactory('glpk').solve(m, tee=False)

    chosen = next((options[k] for k in m.OPTIONS if value(m.x[k]) > 0.5), None)
    if chosen is None: return None

    loc = parts.location_by_index(chosen['idx'])
    facing = dirs[chosen['di']]
    return Move.from_squad(self, purchase=Purchase(chosen['pt'], loc, facing))
```

Key improvements over the previous version:

| Change | Why |
|---|---|
| `strategy.can_afford(...)` as the gate | Replaces ad-hoc `pantry - 1`; strategy owns affordability |
| `king_idx` computed once | Was recomputed inside every hex×dir loop |
| Removed duplicate `m.del_component / re-add` | The sustainability constraint was rebuilt incorrectly inline |
| C4 threat gate uses `needs.king_threat` | Wired to `BoardNeeds` instead of a separate income check |
| Removed the manual bleeding fix-to-zero block | `can_afford` + C3 sustainability make it redundant |
| `next(..., None)` for solution extraction | Cleaner than a for-break |

In [ ]:
@patch
def loadSquads(self: GameParts, year: int = 1880, seed: int = 0, levels=3):
    self.flags = CountryFlag.seaborn("bright", levels=levels)
    self.squads = []
    for i, flag in enumerate(self.flags):
        roster = {PieceType.KING: 1}  # everything else is purchased
        pieceKinds = [pt for pt, n in roster.items() for _ in range(n)]
        squad = flag.make_squad(pieceKinds, year=year, seed=seed)
        self.squads.append(squad)


In [ ]:
@patch
def setup_game(self: GameParts, n_players: int = 5, year: int = 1950, 
               seed: int = 23, starting_pantry: int = 100) -> TurnProcessor:
    """Place kings at balanced starts, seed pantries, return TurnProcessor."""
    self.loadSquads(year=year, seed=seed, levels=n_players)
    starts = self.find_starts(n=n_players, ring_radius=3, min_distance=6)
    
    for squad, start_idx in zip(self.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = self.location_by_index(start_idx)
        move = Move.from_squad(squad, placement=Placement(king, loc))
        move.apply(self)
        king.place.pantry = starting_pantry
    
    self.player_squad = self.squads[0]
    return TurnProcessor(self, self.squads)


In [ ]:
# Reset and replay
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    move = Move.from_squad(squad, placement=Placement(king, loc))
    move.apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)

for round_num in range(3):
    print(f"\n=== Purchase round {round_num + 1} ===")
    for squad in myStuff.squads:
        move = squad.best_purchase(myStuff)
        if move:
            summary = tp.resolve_move(squad, move)
            pt = move.purchase.piece_type.value if move.purchase else "?"
            idx = myStuff.hp2i(move.purchase.location.location) if move.purchase else "?"
            print(f"  {squad.name}: bought {pt} → hex {idx}, "
                  f"king_pantry={summary['king_pantry_after']}, "
                  f"connected={summary['connected']}")
        else:
            print(f"  {squad.name}: no purchase (broke or no valid hex)")

print(f"\n{tp.status()}")

I am part way through a refactor of 
```python
def best_purchase(self: Squad, parts: GameParts,
                  strategy: PlacementStrategy = None,
                  stance: str = "expand") -> Move | None:

```
what should I do next?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The error is straightforward — `PlacementStrategy` needs `min_reserve_per_piece` and `min_budget_ratio` attributes. You have two options:

**Option A: Add them to `PlacementStrategy`** (cleanest)

```python
@dataclass
class PlacementStrategy:
    name: str
    min_reserve_per_piece: float = 2.0
    min_budget_ratio: float = 0.8
    # ... existing fields
```

With per-strategy tuning:
```python
STRATEGIES = {
    "aggressive": PlacementStrategy("aggressive", min_reserve_per_piece=1.0, min_budget_ratio=0.5),
    "expansive":  PlacementStrategy("expansive",  min_reserve_per_piece=2.0, min_budget_ratio=0.7),
    "balanced":   PlacementStrategy("balanced",   min_reserve_per_piece=2.0, min_budget_ratio=0.8),
    "turtle":     PlacementStrategy("turtle",     min_reserve_per_piece=4.0, min_budget_ratio=1.0),
}
```

**Option B: Fall back to defaults in `best_purchase`** (quick fix)

```python
reserve = n_placed * getattr(strategy, 'min_reserve_per_piece', 2.0)
min_ratio = getattr(strategy, 'min_budget_ratio', 0.8)
```

**Option A is the right call** since these values are genuinely strategy-specific — a turtle should hoard more reserves than an aggressor. Also check if `strategy.name` works correctly (the error is at `strategy.min_reserve_per_piece` but `strategy_name = strategy.name` earlier in the function also needs that attribute to exist).

What does `PlacementStrategy` currently look like?

We need to refactor this so that is adding/purchasing three addional pawns. Is the purchase infracture ok to do so?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Almost — there are two gaps in `MoveLog.record` that will break purchases:

**1. It drops the `purchase` field:**
```python
validated = Move.from_squad(squad, move.rotations, move.placement)
#                                                   ^ no purchase!
```

**2. It doesn't pass `squad` to `apply`:**
```python
validated.apply(self.parts)  # apply needs squad for purchases
```

Here's the fix:

```python
@dataclass
class MoveLog:
    parts: GameParts
    _df: pd.DataFrame = field(default_factory=lambda: pd.DataFrame(
        columns=['turn', 'squad', 'piece', 'action', 'detail']))
    
    def record(self, squad: Squad, move: Move):
        """Validate, apply, and log."""
        validated = Move.from_squad(squad, move.rotations, move.placement, move.purchase)
        validated.apply(self.parts, squad)
        turn = len(self._df)
        
        rows = []
        for r in validated.rotations:
            rows.append(dict(turn=turn, squad=squad.name, piece=r.piece.name,
                             action='rotate', detail=r.facing.label))
        if validated.placement:
            p = validated.placement
            idx = self.parts.hp2i(p.location.location)
            rows.append(dict(turn=turn, squad=squad.name, piece=p.piece.name,
                             action='place', detail=str(idx)))
        if validated.purchase:
            rows.append(dict(turn=turn, squad=squad.name, 
                             piece=validated.purchase.piece_type.value,
                             action='purchase', 
                             detail=str(self.parts.hp2i(validated.purchase.location.location))))
        
        self._df = pd.concat([self._df, pd.DataFrame(rows)], ignore_index=True)
    
    @property
    def df(self) -> pd.DataFrame: return self._df
    
    def for_squad(self, name: str) -> pd.DataFrame:
        return self._df[self._df.squad == name]
    
    def turn_count(self) -> int:
        return int(self._df.turn.max() + 1) if len(self._df) else 0
```

Then the replay loop becomes:

```python
# Reset and replay
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    move = Move.from_squad(squad, placement=Placement(king, loc))
    move.apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)

for round_num in range(3):
    print(f"\n=== Purchase round {round_num + 1} ===")
    for squad in myStuff.squads:
        move = squad.best_purchase(myStuff)
        if move:
            summary = tp.resolve_move(squad, move)
            pt = move.purchase.piece_type.value if move.purchase else "?"
            idx = myStuff.hp2i(move.purchase.location.location) if move.purchase else "?"
            print(f"  {squad.name}: bought {pt} → hex {idx}, "
                  f"king_pantry={summary['king_pantry_after']}, "
                  f"connected={summary['connected']}")
        else:
            print(f"  {squad.name}: no purchase (broke or no valid hex)")

print(f"\n{tp.status()}")
```

The `best_purchase` LP should naturally favor pawns early — they're cheapest (cost=3), harvest well, and `food_gap` will be high when you've only got a king eating 2/turn with no income.

In [ ]:
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
tp = myStuff.setup_game(n_players=playerCount)

for round_num in range(3):
    print(f"\n=== Round {round_num + 1} ===")
    for squad in myStuff.squads:
        move = squad.best_purchase(myStuff)
        if move and move.purchase:
            summary = tp.resolve_move(squad, move)
            pt = move.purchase.piece_type.value
            idx = myStuff.hp2i(move.purchase.location.location)
            print(f"  {squad.name}: bought {pt} → hex {idx}, "
                  f"pantry={summary['king_pantry_after']}, "
                  f"connected={summary['connected']}")
        else:
            print(f"  {squad.name}: skipped")
    
    for squad in myStuff.squads:
        squad.optimize_facings(myStuff)

print(f"\n{tp.status()}")


Should the cell right above be refactored


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yeah, a few things stick out:

**1. `loadSquads` still creates 3 pawns + 1 king** — but the new design is "start with just a king, buy the rest." The roster in `loadSquads` should shrink:

```python
@patch
def loadSquads(self: GameParts, year: int = 1880, seed: int = 0, levels=3):
    self.flags = CountryFlag.seaborn("bright", levels=levels)
    self.squads = []
    for i, flag in enumerate(self.flags):
        roster = {PieceType.KING: 1}  # everything else is purchased
        pieceKinds = [pt for pt, n in roster.items() for _ in range(n)]
        squad = flag.make_squad(pieceKinds, year=year, seed=seed)
        self.squads.append(squad)
```

**2. The setup + loop is a pattern you'll repeat** — extract it:

```python
@patch
def setup_game(self: GameParts, n_players: int = 5, year: int = 1950, 
               seed: int = 23, starting_pantry: int = 100) -> TurnProcessor:
    """Place kings at balanced starts, seed pantries, return TurnProcessor."""
    self.loadSquads(year=year, seed=seed, levels=n_players)
    starts = self.find_starts(n=n_players, ring_radius=3, min_distance=6)
    
    for squad, start_idx in zip(self.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = self.location_by_index(start_idx)
        move = Move.from_squad(squad, placement=Placement(king, loc))
        move.apply(self)
        king.place.pantry = starting_pantry
    
    return TurnProcessor(self, self.squads)
```

**3. The round loop** should use `best_purchase` (not `best_pawn_move`) and becomes a one-liner per squad:

```python
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
tp = myStuff.setup_game(n_players=playerCount)

for round_num in range(3):
    print(f"\n=== Round {round_num + 1} ===")
    for squad in myStuff.squads:
        move = squad.best_purchase(myStuff)
        if move and move.purchase:
            summary = tp.resolve_move(squad, move)
            pt = move.purchase.piece_type.value
            idx = myStuff.hp2i(move.purchase.location.location)
            print(f"  {squad.name}: bought {pt} → hex {idx}, "
                  f"pantry={summary['king_pantry_after']}, "
                  f"connected={summary['connected']}")
        else:
            print(f"  {squad.name}: skipped")
    
    for squad in myStuff.squads:
        squad.optimize_facings(myStuff)

print(f"\n{tp.status()}")
```

The LP will naturally buy pawns first (cheapest, best food ROI when `food_gap` is high), then branch into bishops/queens as the board develops — no need to hardcode the pawn-first loop.

In [ ]:
# Module-level scorer functions (shared by overlay + panel)
def _score_pawn(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    return int(parts.fy.tiers[idx])

def _score_bishop(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    bonus = FOOD_TABLE[PieceType.BISHOP].bonus_sight
    eff = max(2, loc.level) + bonus
    tier = int(parts.fy.tiers[idx])
    dirs = HexPosition.directions()
    ally_idxs = {parts.hp2i(p.place.location) for p in squad.pieces if p.place is not None}
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff))
        allies_seen = len(ally_idxs & cone)
        new_land = len((cone & land_set) - current_visible)
        harvest_net = FOOD_TABLE[PieceType.BISHOP].harvest * tier - FOOD_TABLE[PieceType.BISHOP].diet
        score = allies_seen * 5 + new_land * 4 + harvest_net * 3 + loc.level * 0.5
        best = max(best, score)
    return best

def _score_knight(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    eff = max(2, loc.level)
    dirs = HexPosition.directions()
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not squad
                  for p in sq.pieces if p.place is not None}
    if not enemy_idxs: return -999
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff))
        enemies_hit = len(cone & enemy_idxs)
        roi = enemies_hit * FOOD_TABLE[PieceType.KNIGHT].drain - FOOD_TABLE[PieceType.KNIGHT].cost
        if roi <= 0: continue
        best = max(best, roi + loc.level * 0.3)
    return best

def _score_queen(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    bonus = FOOD_TABLE[PieceType.QUEEN].bonus_sight
    eff = max(2, loc.level) + bonus
    dirs = HexPosition.directions()
    king = next((p for p in squad.pieces if p.type == PieceType.KING and p.place is not None), None)
    king_hp = king.place.location if king else None
    ally_idxs = {parts.hp2i(p.place.location) for p in squad.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not squad
                  for p in sq.pieces if p.place is not None}
    king_dist = hp.distance(king_hp) if king_hp else 0
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff))
        allies_covered = len(ally_idxs & cone)
        enemies_caught = len(enemy_idxs & cone)
        sees_king = 1 if king_hp and parts.hp2i(king_hp) in cone else 0
        score = (allies_covered * 3.0 - king_dist * 4.0
                 + enemies_caught * FOOD_TABLE[PieceType.QUEEN].drain
                 + sees_king * 5.0)
        best = max(best, score)
    return best

PLACEMENT_SCORERS = {
    PieceType.PAWN:   _score_pawn,
    PieceType.BISHOP: _score_bishop,
    PieceType.KNIGHT: _score_knight,
    PieceType.QUEEN:  _score_queen,
}


In [ ]:
def OptimalPlacementOverlay(top_n: int = 3, **kw) -> OverlaySpec:
    """Show top-N candidate hexes per piece type using flag-colored chess pieces."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts_list = []
        
        for squad in ctx.squads:
            placed = [p for p in squad.pieces if p.place is not None]
            if not placed: continue
            parts_obj = placed[0].place.parent
            terr = parts_obj.terr
            flag = squad.flag
            
            current_visible = set()
            for ally in placed:
                current_visible |= ally.sight().hexes
            land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
            occupied = {parts_obj.hp2i(p.place.location)
                        for sq in parts_obj.squads for p in sq.pieces if p.place is not None}
            
            for pt, scorer in PLACEMENT_SCORERS.items():
                unplaced = [p for p in squad.pieces if p.type == pt and p.place is None]
                if not unplaced: continue
                
                scored = []
                for idx in current_visible:
                    s = scorer(idx, parts_obj, squad, current_visible, occupied, land_set)
                    if s > -999:
                        scored.append((idx, s))
                if not scored: continue
                
                scored.sort(key=lambda x: -x[1])
                top = scored[:top_n]
                
                scale = grid.radius * 0.7 / 22.5
                
                for rank, (idx, score) in enumerate(top):
                    opacity = 0.8 - rank * 0.2
                    
                    for fi in ctx.fine_indices(idx):
                        if 0 <= fi < N:
                            c = grid.hexes[fi].center
                            
                            # Ghost ring background
                            r = grid.radius * 0.6
                            parts_list.append(
                                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" r="{r:.1f}" '
                                f'fill="{flag.lightPrimary}" fill-opacity="{opacity * 0.5:.2f}" '
                                f'stroke="{flag.darkPrimary}" stroke-opacity="{opacity:.2f}" '
                                f'stroke-width="1.5" stroke-dasharray="3,2"/>')
                            
                            # Chess piece silhouette
                            svg_str, pat_def = flag.piece_svg(
                                pt, MapCord(c.x, c.y),
                                scale=scale, size='board',
                                piece_id=f"opt_{squad.name}_{pt.value}_{rank}_{fi}")
                            if pat_def is not None:
                                ctx.builder.add_definition(pat_def)
                            parts_list.append(
                                f'<g opacity="{opacity:.2f}">{svg_str}</g>')
                            
                            # Rank number (top-right)
                            parts_list.append(
                                f'<text x="{c.x + r * 0.7:.1f}" y="{c.y - r * 0.7:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(9, grid.radius * 0.35):.0f}" '
                                f'fill="{flag.darkPrimary}" font-weight="bold" '
                                f'opacity="{opacity:.2f}">{rank + 1}</text>')
                            
                            # Hex index (bottom)
                            parts_list.append(
                                f'<text x="{c.x:.1f}" y="{c.y + r * 1.1:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(7, grid.radius * 0.28):.0f}" '
                                f'fill="{flag.darkPrimary}" '
                                f'opacity="{opacity * 0.7:.2f}">#{idx}</text>')
        
        return '\n'.join(parts_list)
    
    return OverlaySpec("optimal_placement", render, requires={'squads'}, priority=75)


In [ ]:
@patch
def place_opening(self: Squad, parts: GameParts, start_idx: int, pantry: int = 100):
    """Place king at start_idx and 3 pawns on the best adjacent hexes."""
    king = next(p for p in self.pieces if p.type == PieceType.KING)
    pawns = [p for p in self.pieces if p.type == PieceType.PAWN and p.place is None][:3]
    
    # Place king first
    loc = parts.location_by_index(start_idx)
    Move.from_squad(self, placement=Placement(king, loc)).apply(parts)
    king.place.pantry = pantry
    
    # Score neighbors for pawn placement
    neighbors = parts.grid.neighborsOf(start_idx)
    occupied = {start_idx}
    scored = []
    for nb in neighbors:
        if nb < 0 or nb >= len(parts.terr.elevations): continue
        if parts.terr.elevations[nb] <= 0: continue
        tier = int(parts.fy.tiers[nb])
        level = parts.terr.elevationLevel(nb)
        scored.append((nb, tier * 3 + level))
    scored.sort(key=lambda x: -x[1])
    
    for pawn, (nb_idx, _) in zip(pawns, scored):
        if nb_idx in occupied: continue
        loc = parts.location_by_index(nb_idx)
        Move.from_squad(self, placement=Placement(pawn, loc)).apply(parts)
        pawn.place.pantry = int(pawn.food.cost * 0.75)
        occupied.add(nb_idx)
    
    # Orient everyone to face outward from king
    self.orient_facings(parts)


I want to add to moves that you can purchase a unit for something like twice its diet.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice — this gives squads a way to grow beyond their starting roster. A few design questions before I write it:

**Key choice: purchase = create + place in one move, or two steps?**

I'd lean toward **one step** (purchase puts the piece directly on the board), since an unplaced purchased piece sitting in the roster is awkward — you already paid for it but it's doing nothing. That keeps it analogous to the current `Placement` flow.

Here's a minimal sketch:

```python
PURCHASE_COST: dict[PieceType, int] = {
    PieceType.PAWN:   2,  # 2 × diet=1
    PieceType.BISHOP: 6,  # 2 × diet=3
    PieceType.KNIGHT: 4,  # 2 × diet=2
    PieceType.QUEEN: 8,   # 2 × diet=4
}
```

Then a `Purchase` named tuple alongside `Rotation` and `Placement`:

```python
class Purchase(NamedTuple):
    piece_type: PieceType
    location: MapPlace
    facing: HexPosition
```

And `Move` gets an optional `purchase` field:

```python
@dataclass
class Move:
    rotations: list[Rotation] = field(default_factory=list)
    placement: Placement | None = None
    purchase: Purchase | None = None
```

Validation in `from_squad` would check:
- King has enough pantry for `PURCHASE_COST[piece_type]`
- Target hex is in ally sight, on land, unoccupied
- Can't do both `placement` AND `purchase` in the same move (or can they?)
- Can't purchase a KING

Then `resolve_move` creates the `Piece`, adds it to `squad.pieces`, places it, and deducts cost from king.

**Questions for you:**
1. Can a move have BOTH a placement (of an existing piece) and a purchase (of a new one), or one-per-turn?
2. Should purchased pieces use the flag's `commonNames` for naming, same as `make_squad`?
3. Is 2× diet the right multiplier, or do you want it tunable per type like `PURCHASE_COST`?

In [ ]:
# After pawn rounds...
for squad in myStuff.squads:
    move = squad.best_queen_move(myStuff)
    if move:
        summary = tp.resolve_move(squad, move)
        print(f"  {squad.name}: 👑→♛ draining {summary.get('food_drain_dealt', 0)} food")


In [ ]:
for squad in myStuff.squads:
    placed = [p for p in squad.pieces if p.place is not None]
    if not placed: continue
    print(f"\n{squad.name}:")
    for p in placed:
        idx = myStuff.hp2i(p.place.location)
        tier = int(myStuff.fy.tiers[idx])
        in_enemy_cones = 0
        for esq in myStuff.squads:
            if esq is squad: continue
            for ep in esq.pieces:
                if ep.place is not None and idx in ep.sight().hexes:
                    in_enemy_cones += 1
        drain_taken = sum(ep.food.drain for esq in myStuff.squads if esq is not squad
                         for ep in esq.pieces if ep.place is not None 
                         and idx in ep.sight().hexes)
        net = int(p.food.harvest * tier) - p.food.diet - int(drain_taken)
        print(f"  {p.type.icon} {p.name}: pantry={p.place.pantry}, "
              f"tier={tier}, drain_taken={drain_taken:.0f}, net/turn={net:+d}"
              f"{' ⚠️' if p.place.pantry + net < 0 else ''}")


## Placement strategy

In [ ]:
@dataclass
class PlacementStrategy:
    """Squad personality that shapes piece priority and stance thresholds."""
    name: str
    # Stance thresholds (how easily this personality shifts)
    defend_threat: int = 2       # enemy pieces near king to trigger defend
    attack_pantry: int = 60      # min king pantry to go offensive
    attack_placed: int = 4       # min pieces on board before attacking
    # Knight eagerness: lower = fires more readily
    knight_roi_mult: float = 1.0  # multiplied into knight ROI gate
    # NEW: spending discipline
    min_reserve_per_piece: int = 5    # pantry reserve per placed piece
    min_budget_ratio: float = 1.0     # won't buy unless income/upkeep >= this
    max_deficit_turns: int = 2        # skip buying if net negative for N+ turns
    buy_cooldown: int = 0             # min turns between purchases (0=every turn)
    afforable_min = 3                 # min number of pieces before we check on afford
    
    def can_afford(self, squad: Squad, parts: GameParts, cost: int) -> bool:
    king = next((p for p in squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return False
    
    placed = [p for p in squad.pieces if p.place is not None]
    pantry = king.place.pantry
    
    # Dynamic reserve: scales with army size
    reserve = len(placed) * self.min_reserve_per_piece
    if pantry - cost < reserve:
        return False
    
    # Budget ratio check — but skip during bootstrap (only king placed)
    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)
    if len(placed) > 1 and upkeep > 0 and income / upkeep < self.min_budget_ratio:
        return False
    
    # Net food check: don't buy while bleeding (but allow if plenty of reserves)
    net = income - upkeep
    if net < 0 and pantry < reserve * 2 and len(placed) > afforable_min:
        return False
    
    return True

    
    def assess(self, squad: Squad, parts: GameParts) -> str:
        king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        if not king: return "defend"
        
        pantry = king.place.pantry
        placed = [p for p in squad.pieces if p.place is not None]
        G = squad.to_nx()
        U = G.to_undirected(as_view=True) if len(G) > 1 else nx.Graph()
        bottlenecks = len(list(nx.articulation_points(U))) if len(G) > 1 else 0
        
        king_hp = king.place.location
        threat_count = sum(
            1 for sq in parts.squads if sq is not squad
            for p in sq.pieces if p.place is not None
            and parts.i2hp(parts.hp2i(p.place.location)).distance(king_hp) <= 4
        )
        
        if threat_count >= self.defend_threat or bottlenecks >= 2:
            return "defend"
        elif pantry > self.attack_pantry and len(placed) >= self.attack_placed:
            return "attack"
        else:
            return "expand"
    
    
    

# Updated presets with spending discipline
PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
    min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0)

PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3,
    knight_roi_mult=0.7,
    min_reserve_per_piece=3, min_budget_ratio=0.6, buy_cooldown=0)

PlacementStrategy.BALANCED = PlacementStrategy(
    "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
    min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=1)

PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle", defend_threat=1, attack_pantry=999, attack_placed=99,
    min_reserve_per_piece=8, min_budget_ratio=1.2, buy_cooldown=2)


In [ ]:
@patch
def next_move(self: PlacementStrategy, squad: Squad, parts: GameParts) -> Move | None:
    """Assess stance, manage target, delegate to best_purchase."""
    target = getattr(squad, '_target', None)
    if target is not None: target.tick()
    if target is None or target.is_expired:
        squad._target = squad.choose_target(parts, self)
        squad.optimize_facings(parts, max_iters=3)
    else:
        squad.orient_facings(parts)
    
    stance = self.assess(squad, parts)
    return squad.best_purchase(parts, strategy=self, stance=stance)

### Target ideas

In [ ]:
def run_game(parts: GameParts, strategies: dict[str, PlacementStrategy], rounds: int = 15) -> TurnProcessor:
    tp = TurnProcessor(parts, parts.squads)
    
    for r in range(rounds):
        if tp.game_over: break
        
        for squad in parts.squads:
            if not squad.king_alive(): continue
            strat = strategies[squad.name]
            
            # next_move handles target management, facing optimisation, and purchase
            move = strat.next_move(squad, parts)
            
            # Always resolve the turn (harvest/drain/lifespan run even if no purchase)
            tp.resolve_move(squad, move or Move())
        
        if r % 5 == 0:
            print(f"\n--- Round {r} ---")
            print(tp.status())
    
    print(f"\n=== FINAL (round {tp.turn}) ===")
    print(tp.status())
    return tp



In [ ]:
# Fresh game
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    Move.from_squad(squad, placement=Placement(king, loc)).apply(myStuff)
    king.place.pantry = 100

# Assign different personalities
personality_rotation = [
    PlacementStrategy.EXPANSIVE,
    PlacementStrategy.AGGRESSIVE,
    PlacementStrategy.BALANCED,
    PlacementStrategy.TURTLE,
    PlacementStrategy.AGGRESSIVE,
]
strategies = {}
for i, sq in enumerate(myStuff.squads):
    strategies[sq.name] = personality_rotation[i % len(personality_rotation)]
    print(f"{sq.name}: {strategies[sq.name].name}")

tp = run_game(myStuff, strategies, rounds=15)

how do I install?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

In a solveit code cell:

```bash
!apt-get install -y glpk-utils
```

That installs `glpsol` which is what Pyomo's `'glpk'` solver looks for. Alternatively, CBC is another free option:

```bash
!apt-get install -y coinor-cbc
```

and then use `SolverFactory('cbc')` instead.

If you want to avoid external solvers entirely, since your problem is small (binary, one purchase per turn), you could also just drop Pyomo and do a plain argmax over the precomputed scores — which is effectively what the MIP reduces to given the `<= 1` purchase constraint. That would be faster and dependency-free:

```python
# Replace the Pyomo block with:
best = max(
    ((k, v) for k, v in options.items() if strategy.can_afford(self, parts, v['cost'])),
    key=lambda kv: kv[1]['score'],
    default=(None, None)
)
chosen = best[1]
```

Since you're only ever buying one piece, the MIP is overkill — the linear relaxation is already integral. The Pyomo version only starts earning its keep if you add multi-purchase turns or more complex constraints.

so what would the full function look like with 

```python
# Replace the Pyomo block with:
best = max(
    ((k, v) for k, v in options.items() if strategy.can_afford(self, parts, v['cost'])),
    key=lambda kv: kv[1]['score'],
    default=(None, None)
)
chosen = best[1]
```
in it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy: 'PlacementStrategy' = None,
                  stance: str = "expand") -> Move | None:
    """Purchase: strategy-aware argmax over (type, hex, facing) options."""
    strategy = strategy or PlacementStrategy.BALANCED
    strategy_name = strategy.name

    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None

    placed = [p for p in self.pieces if p.place is not None]

    # ── Strategy-gated affordability check ──
    affordable_types = [pt for pt, cost in PURCHASE_COST.items()
                        if strategy.can_afford(self, parts, cost)]
    if not affordable_types: return None

    # ── Board state ──
    terr, tiers = parts.terr, parts.fy.tiers
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    king_hp = king.place.location
    king_idx = parts.hp2i(king_hp)

    current_visible = set()
    for ally in placed: current_visible |= ally.sight().hexes
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    valid_hexes = [idx for idx in current_visible
                   if idx in land_set and idx not in occupied]
    if not valid_hexes: return None

    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)

    # ── Objective weights from strategy × stance ──
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy_name, PURCHASE_OBJ["balanced"])[stance]
    n_vec = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                      needs.frontier_need, needs.king_threat])
    w_vec = np.array(weights)

    # ── Precompute supply scores ──
    options = {}  # (pt_val, hex_idx, dir_idx) -> dict

    for pt in affordable_types:
        fs = FOOD_TABLE[pt]
        cost = PURCHASE_COST[pt]
        lifespan = LIFESPAN.get(pt, 10)

        for idx in valid_hexes:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            tier = int(tiers[idx])
            eff_sight = max(2, loc.level) + fs.bonus_sight

            for di, d in enumerate(dirs):
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))

                s_food     = (fs.harvest * tier - fs.diet) * lifespan - cost
                s_drain    = len(enemy_idxs & cone) * fs.drain
                s_connect  = len(ally_idxs & cone) + (2 if king_idx in cone else 0)
                s_frontier = len((cone & land_set) - current_visible)
                s_threat   = fs.drain if king_idx in cone else 0.0

                supply = np.array([s_food, s_drain, s_connect, s_frontier, s_threat])
                score  = float(n_vec * w_vec @ supply)
                harvest_net = int(fs.harvest * tier) - fs.diet

                # Under threat, skip pieces that can't drain back
                if needs.king_threat > 2.0 and fs.drain == 0:
                    continue

                # Sustainability gate: post-purchase income must cover ratio
                min_ratio = strategy.min_budget_ratio
                if upkeep > 0:
                    post_income = income + harvest_net
                    post_upkeep = upkeep + fs.diet
                    if post_income < min_ratio * post_upkeep:
                        continue

                options[(pt.value, idx, di)] = dict(
                    score=score, cost=cost,
                    diet=fs.diet, harvest_net=harvest_net,
                    pt=pt, idx=idx, di=di,
                )

    if not options: return None

    # ── Argmax: pick highest-scoring affordable option ──
    _, chosen = max(
        ((k, v) for k, v in options.items()
         if strategy.can_afford(self, parts, v['cost'])),
        key=lambda kv: kv[1]['score'],
        default=(None, None)
    )
    if chosen is None: return None

    loc = parts.location_by_index(chosen['idx'])
    facing = dirs[chosen['di']]
    return Move.from_squad(self, purchase=Purchase(chosen['pt'], loc, facing))
```

Two improvements vs the drop-in replacement:

1. **Sustainability and threat filters moved into the option-building loop** — options that fail these checks are excluded upfront rather than being scored and then discarded. This means `can_afford` in the argmax is the only remaining gate, which is clean.

2. **`default=(None, None)` destructuring** — safe even if the generator is empty.

## Web

In [ ]:
def appRoutes():
    global app, rt, hexserver
    #if 'hexserver' not in dir() or hexserver is None:
    if 'hexserver' not in globals() or hexserver is None:

        app = FastHTML(hdrs=Theme.violet.headers())
        rt = app.route
        hexserver = JupyUvi(app)
    return app, rt, hexserver

In [ ]:
app, rt,  hexserver = appRoutes()

In [ ]:

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
log = logging.getLogger("hexgame")

# In-memory log ring buffer
game_log = deque(maxlen=50)

class GameLogHandler(logging.Handler):
    def emit(self, record):
        game_log.append(self.format(record))

log.addHandler(GameLogHandler())
log.setLevel(logging.INFO)

# Option 1: Attach to uvicorn loggers directly
for name in ('uvicorn', 'uvicorn.access', 'uvicorn.error'):
    logging.getLogger(name).addHandler(GameLogHandler())


In [ ]:

def debug_log():
    return Card(
        Pre('\n'.join(game_log), cls="text-xs whitespace-pre-wrap max-h-[60vh] overflow-y-auto"),
        header=H4(f"Game Log ({len(game_log)} entries)"),
        footer=Button("Refresh", hx_get="/debug/log", hx_target="#debug-panel"),
        id="debug-panel",
    )


In [ ]:
def webMe(*c): return HTMX(*c, host='', app=app)

In [ ]:
@rt
def hello(name: str): return P(f"Hello {name}!")

webMe(Div(
    H3("Say Hello"),
    Form(
        Input(placeholder="Your name...", id='name'),
        Button("Send"),
        hx_get=hello, hx_target="#result"
    ),
    Div(id='result'),
))

In [ ]:
from HexMagic.primitives import HexButtonGroup, HexLegend
from starlette.responses import RedirectResponse

So I want to have an interative game as a fasthtml app. For the main game part there should be 3 panels
1. left side panel - this is a list of your pieces by rank. if a piece is on the board this rotates it clockwise. if a piece is not on the board it selects it for placement
2. the main map. if you have a selected piece and you click it places it on the map. if you click a selected piece that you are thinking about moving that is on the board it removes it from the board. below the map will be radio buttons (either a hexlegend or button group) that we can toggle overlays. there should be a 'commit move' button which says that this is your final move.
3. the history right side panel - both the moves on top and then a detail one on bottom that shows/explains the log from a particular round.

We do need a landing page that lets you start a new map and you can pick the size, number of countries and perhaps seed

## GlobalGameState

In [ ]:
@dataclass
class GlobalGameState:
    parts: GameParts | None = None
    tp: TurnProcessor | None = None
    player_squad: Squad | None = None
    selected_piece: Piece | None = None
    pending_rotations: list[Rotation] = field(default_factory=list)
    pending_placement: Placement | None = None
    pending_purchase: Purchase | None = None
    pending_purchase_type: PieceType | None = None  # add this
    active_overlays: set = field(default_factory=lambda: { 'rivers','lanterns','network','ocean'})
    turn: int = 0
    radius:float = 25
    debug_overlays: bool = False
    rotationClickId: int = None
    ai_strategies: dict[str, PlacementStrategy] = field(default_factory=dict)
    focused_piece: Piece | None = None
    # Add to GlobalGameState
    active_chart_cols: set = field(default_factory=lambda: {'king_pantry', 'pieces_placed'})
    active_chart_squads: set = field(default_factory=lambda: set())  # empty = all
    
    def select_piece(self, piece: Piece):
        """Select an unplaced piece for placement."""
        if piece.place is not None:
            return  # already on board — rotation handled separately
        self.selected_piece = piece
    
    def place_piece(self, hex_idx: int):
        """Place the selected piece at a hex."""
        if self.selected_piece is None:
            return False
        loc = self.parts.location_by_index(hex_idx)
        self.pending_placement = Placement(self.selected_piece, loc)
        self.selected_piece.place = loc  # preview immediately
        self.selected_piece = None
        return True
    
    def unplace_piece(self, piece: Piece):
        """Remove a pending placement (before committing)."""
        if self.pending_placement and self.pending_placement.piece is piece:
            piece.place = None
            self.pending_placement = None
    
    
    @property
    def is_active(self) -> bool:
        return self.parts is not None and self.tp is not None
    
    @property
    def game_over(self) -> bool:
        return self.tp.game_over if self.tp else False

# Global singleton
GS = GlobalGameState()


In [ ]:
@patch
def start_game(self:GlobalGameState, size=300, player_count=3, seed=23, radius=20):
    random.seed(seed)
    np.random.seed(seed)
    self.parts = GameParts(radius=radius, size=size)
    self.parts.grid.adjustRadius(radius)
    self.tp = self.parts.setup_game(n_players=player_count, year=1950,
                                     seed=seed, starting_pantry=100)
    self.player_squad = self.parts.squads[0]
    
    ai_pool = [PlacementStrategy.AGGRESSIVE, PlacementStrategy.EXPANSIVE,
                PlacementStrategy.TURTLE, PlacementStrategy.BALANCED]
    self.ai_strategies = {sq.name: ai_pool[i % len(ai_pool)]
                          for i, sq in enumerate(self.parts.squads[1:])}
    self.selected_piece = None
    self.pending_rotations = []
    self.pending_placement = None
    self.pending_purchase = None
    self.active_chart_cols = {'king_pantry', 'pieces_placed'}
    self.active_chart_squads = set()   # ← add these two lines
    self.turn = 0


Should start_game be refactored since we have

In [ ]:
@patch
def compute_optimal(self: GlobalGameState) -> dict[PieceType, list[tuple[int, float]]]:
    """Compute top-3 placement candidates per piece type. Cached per turn."""
    if not self.is_active:
        return {}
    
    squad = self.player_squad
    placed = [p for p in squad.pieces if p.place is not None]
    if not placed:
        return {}
    
    parts = self.parts
    terr = parts.terr
    
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # Reuse the same scorers from OptimalPlacementOverlay
    from types import SimpleNamespace
    SCORERS = {
        PieceType.PAWN:   _score_pawn,
        PieceType.BISHOP: _score_bishop,
        PieceType.KNIGHT: _score_knight,
        PieceType.QUEEN:  _score_queen,
    }
    
    results = {}
    for pt, scorer in SCORERS.items():
        unplaced = [p for p in squad.pieces if p.type == pt and p.place is None]
        if not unplaced:
            continue
        scored = []
        for idx in current_visible:
            s = scorer(idx, parts, squad, current_visible, occupied, land_set)
            if s > -999:
                scored.append((idx, s))
        scored.sort(key=lambda x: -x[1])
        results[pt] = scored[:3]
    
    return results


In [ ]:
@patch
def squad_flow(self: GlobalGameState, squad: Squad = None) -> SquadFlow:
    """Cached SquadFlow, invalidated on commit."""
    squad = squad or self.player_squad
    cache = getattr(self, '_flow_cache', {})
    key = (squad.name, self.turn)
    if key not in cache:
        cache[key] = SquadFlow.from_squad(squad)
        self._flow_cache = {key: cache[key]}  # keep only current turn
    return cache[key]


In [ ]:
@patch
def rotate_piece(self: GlobalGameState, piece: Piece):
    """Rotate a placed piece clockwise by one step."""
    if piece.place is None:
        return
    dirs = HexPosition.directions()
    cur_idx = dirs.index(piece.place.facing) if piece.place.facing in dirs else 0
    new_facing = dirs[(cur_idx + 1) % len(dirs)]
    
    # If this is the pending placement piece, just update its facing
    # (not a rotation — it hasn't been committed yet)
    if self.pending_placement and self.pending_placement.piece is piece:
        piece.place.facing = new_facing
        # Update the placement's location facing too
        self.pending_placement.location.facing = new_facing
        return
    
    # Otherwise it's a real rotation of an already-placed piece
    self.pending_rotations = [r for r in self.pending_rotations if r.piece is not piece]
    self.pending_rotations.append(Rotation(piece, new_facing))
    piece.place.facing = new_facing  # preview immediately


@patch
def commit_move(self: GlobalGameState) -> dict | None:
    if self.parts is None:
        return None
    
    # Undo placement preview before validation
    if self.pending_placement is not None:
        self.pending_placement.piece.place = None
    
    try:
        move = Move.from_squad(
            self.player_squad,
            rotations=self.pending_rotations,
            placement=self.pending_placement,
            purchase=self.pending_purchase)
    except MoveError as e:
        if self.pending_placement is not None:
            self.pending_placement.piece.place = self.pending_placement.location
        return {'error': str(e)}
    
    summary = self.tp.resolve_move(self.player_squad, move)
    
    # Run AI turns
    ai_summaries = []
    for squad in self.parts.squads:
        if squad is self.player_squad or not squad.king_alive():
            continue
        strategy = self.ai_strategies.get(squad.name, PlacementStrategy.BALANCED)
        ai_move = strategy.next_move(squad, self.parts)
        if ai_move:
            try:
                ai_summaries.append(self.tp.resolve_move(squad, ai_move))
            except MoveError:
                pass
    
    self.turn += 1
    if self.turn % 3 == 0:
        for squad in self.parts.squads:
            squad.optimize_facings(self.parts)
    
    self.pending_rotations = []
    self.pending_placement = None
    self.pending_purchase = None
    self.selected_piece = None
    
    return {'player': summary, 'ai': ai_summaries, 'turn': self.turn}


In [ ]:
@patch
def commit_move(self: GlobalGameState) -> dict | None:
    if self.parts is None:
        return None
    
    if self.pending_placement is not None:
        self.pending_placement.piece.place = None
    
    try:
        move = Move.from_squad(
            self.player_squad,
            rotations=self.pending_rotations,
            placement=self.pending_placement,
            purchase=self.pending_purchase)
    except MoveError as e:
        if self.pending_placement is not None:
            self.pending_placement.piece.place = self.pending_placement.location
        return {'error': str(e)}
    
    summary = self.tp.resolve_move(self.player_squad, move)
    
    # Run AI turns
    ai_summaries = []
    for squad in self.parts.squads:
        if squad is self.player_squad or not squad.king_alive():
            continue
        strategy = self.ai_strategies.get(squad.name, PlacementStrategy.BALANCED)
        ai_move = strategy.next_move(squad, self.parts)
        if ai_move:
            try:
                ai_summaries.append(self.tp.resolve_move(squad, ai_move))
            except MoveError:
                pass
    
    # ← snapshot after all moves this turn are done
    self.tp.record.snapshot(self.turn, self.parts, player_squad=self.player_squad)
    
    self.turn += 1
    if self.turn % 3 == 0:
        for squad in self.parts.squads:
            squad.optimize_facings(self.parts)
    
    self.pending_rotations = []
    self.pending_placement = None
    self.pending_purchase = None
    self.selected_piece = None
    
    return {'player': summary, 'ai': ai_summaries, 'turn': self.turn}


In [ ]:
# ── Landing page ──
@rt
def index():
    return Title("HexMagic"), Container(
        DivCentered(
            H1("⬡ HexMagic", cls="uk-h1"),
            P("Hex-based strategy with food economics", cls=TextPresets.muted_lg),
            Card(
                Form(
                    Grid(
                        LabelInput("Map Size", id="size", type="number", 
                                   value="300", placeholder="200-600"),
                        LabelInput("Players", id="player_count", type="number",
                                   value="3", placeholder="2-5"),
                        LabelInput("Seed", id="seed", type="number",
                                   value="23", placeholder="Any integer"),
                        LabelInput("Hex Radius", id="radius", type="number",
                                   value="20", placeholder="15-30"),
                        cols_sm=2,
                    ),
                    Button("⬡ Start Game", cls=(ButtonT.primary, "w-full mt-4"),
                           hx_post="/start_game", hx_target="body"),
                ),
                header=(H3("New Game"), Subtitle("Configure your world")),
            ),
            cls="min-h-screen py-12",
        )
    )

# ── Start game ──
@rt
def start_game(size: int = 300, player_count: int = 3, 
               seed: int = 23, radius: int = 20):
    player_count = max(2, min(5, player_count))
    size = max(200, min(600, size))
    radius = max(10, min(40, radius))
    
    GS.start_game(size=size, player_count=player_count, 
                   seed=seed, radius=radius)
    
    return RedirectResponse("/game", status_code=303)




read_url("https://fastht.ml/docs/llms-ctx.txt")

### History Panel

In [ ]:
@rt("/panel/history")
def panel_history():
    if GS.tp is None:
        return Div(P("No game active", cls=TextPresets.muted_sm))
    
    status_df = GS.tp.status()
    
    status_rows = [Tr(
        Td(row['king']),
        Td(row['squad'], cls=TextT.medium),
        Td(str(row['king_pantry'])),
        Td(str(row['placed'])),
        Td(GS.ai_strategies.get(row['squad'], 
           PlacementStrategy("you")).name if row['squad'] != GS.player_squad.name 
           else "🎮 you",
           cls=TextPresets.muted_sm),
    ) for _, row in status_df.iterrows()]
    
    status_table = Card(
        Table(
            Thead(Tr(Th(""), Th("Squad"), Th("🍞"), Th("#"), Th("AI"))),
            Tbody(*status_rows),
        ),
        header=H4(f"Turn {GS.turn}"),
    )
    
    # ... log table same as before
    # Move log
    log_rows = []
    if len(GS.tp.log.df) > 0:
        recent = GS.tp.log.df.tail(20)
        for _, row in recent.iterrows():
            log_rows.append(Tr(
                Td(str(int(row['turn'])), cls=TextPresets.muted_sm),
                Td(row['squad'], cls=TextT.medium),
                Td(row['piece']),
                Td(row['action']),
                Td(str(row['detail']), cls=TextPresets.muted_sm),
            ))
    
    log_table = Card(
        Table(
            Thead(Tr(Th("T"), Th("Squad"), Th("Piece"), Th("Act"), Th("Detail"))),
            Tbody(*log_rows),
        ) if log_rows else P("No moves yet", cls=TextPresets.muted_sm),
        header=H4("Move Log"),
    )
    
    return Div(status_table, log_table, cls="space-y-3")


In [ ]:
@rt("/hex_click", methods=["POST"])
def hex_click(hex_id: int):
    log.info(f"HEX_CLICK hex_id={hex_id} rotationMode={GS.rotationClickId}")
    
    # ── Rotation mode: pick facing ──
    if GS.rotationClickId is not None:
        src_idx = GS.rotationClickId
        GS.rotationClickId = None

        src_hp = GS.parts.i2hp(src_idx)
        dst_hp = GS.parts.i2hp(hex_id)

        facing = None
        for d in HexPosition.directions():
            if src_hp + d == dst_hp:
                facing = d
                break

        if facing is not None:
            piece = next((p for sq in GS.parts.squads
                         for p in sq.pieces
                         if p.place is not None
                         and GS.parts.hp2i(p.place.location) == src_idx
                         and p in GS.player_squad.pieces), None)
            if piece:
                GS.rotate_piece(piece)
                piece.place.facing = facing
                GS.pending_rotations = [r for r in GS.pending_rotations if r.piece is not piece]
                GS.pending_rotations.append(Rotation(piece, facing))
                log.info(f"  → rotated {piece.name} to face {facing.label}")

        pieces_update = Div(panel_pieces(), hx_swap_oob="innerHTML:#piece-panel")
        return panel_center(), pieces_update

    # ── Purchase placement flow ──
    if GS.pending_purchase_type is not None and GS.pending_purchase is None:
        loc = GS.parts.location_by_index(hex_id)

        # Compute facing toward king if possible
        king = next((p for p in GS.player_squad.pieces
                     if p.type == PieceType.KING and p.place is not None), None)
        facing = HexPosition.directions()[0]
        if king is not None:
            king_hp = king.place.location
            dst_hp  = GS.parts.i2hp(hex_id)
            best_d, best_dist = facing, float('inf')
            for d in HexPosition.directions():
                neighbor = dst_hp + d
                dist = neighbor.distance(king_hp)
                if dist < best_dist:
                    best_dist, best_d = dist, d
            facing = best_d

        GS.pending_purchase = Purchase(GS.pending_purchase_type, loc, facing)
        GS.pending_purchase_type = None
        log.info(f"  → purchase queued: {GS.pending_purchase.piece_type.value} @ hex {hex_id}")

        pieces_update = Div(panel_pieces(), hx_swap_oob="innerHTML:#piece-panel")
        return panel_center(), pieces_update

    # ── Check if clicking a placed player piece → enter rotation mode ──
    clicked_piece = next((p for p in GS.player_squad.pieces
                         if p.place is not None
                         and GS.parts.hp2i(p.place.location) == hex_id), None)

    if clicked_piece and GS.selected_piece is None and GS.pending_placement is None:
        GS.rotationClickId = hex_id
        log.info(f"  → rotation mode for {clicked_piece.name} at hex {hex_id}")
        return panel_center()

    # ── Normal placement flow ──
    if GS.selected_piece is not None:
        GS.place_piece(hex_id)
        log.info(f"  → placed {GS.pending_placement.piece.name if GS.pending_placement else '?'} at {hex_id}")
    elif GS.pending_placement:
        idx = GS.parts.hp2i(GS.pending_placement.piece.place.location)
        if idx == hex_id:
            log.info(f"  → unplaced {GS.pending_placement.piece.name}")
            GS.unplace_piece(GS.pending_placement.piece)

    pieces_update = Div(panel_pieces(), hx_swap_oob="innerHTML:#piece-panel")
    return panel_center(), pieces_update





@rt("/commit", methods=["POST"])
def commit():
    log.info(f"COMMIT turn={GS.turn} "
             f"rotations={len(GS.pending_rotations)} "
             f"placement={GS.pending_placement.piece.name if GS.pending_placement else None}")
    result = GS.commit_move()
    if result and 'error' in result:
        log.warning(f"  → ERROR: {result['error']}")
        return Div(
            P(f"⚠️ {result['error']}", cls="text-destructive font-medium text-center py-4"),
            Button("Back", hx_get="/game", hx_target="body"),
        )
    log.info(f"  → turn={GS.turn} king_pantry={result['player']['king_pantry_after']} "
             f"starved={result['player']['starved']}")
    return RedirectResponse("/game", status_code=303)


In [ ]:
@rt("/commit", methods=["POST"])
def commit():
    result = GS.commit_move()
    if result and 'error' in result:
        return Div(
            P(f"⚠️ {result['error']}", cls="text-destructive font-medium text-center py-4"),
            Button("Back", hx_get="/game", hx_target="body"),
        )
    
    # Check if player was eliminated
    if result and result['player'].get('eliminated') == GS.player_squad.name:
        return Title("Eliminated"), Container(
            DivCentered(
                H2("☠️ Your Kingdom Falls", cls="text-destructive"),
                P(f"The treasury ran dry on turn {GS.turn}.", cls=TextPresets.muted_lg),
                _game_over_summary(),
                Button("🏠 New Game", cls=(ButtonT.primary, "mt-4"),
                       hx_get="/", hx_target="body"),
            )
        )
    
    # Check if game over (player won)
    if GS.tp.game_over:
        winner = GS.tp.winner
        if winner and winner is GS.player_squad:
            return Title("Victory"), Container(
                DivCentered(
                    H2("👑 Victory!", cls="text-yellow-500"),
                    P(f"{winner.name} is the last kingdom standing!", cls=TextPresets.muted_lg),
                    _game_over_summary(),
                    Button("🏠 New Game", cls=(ButtonT.primary, "mt-4"),
                           hx_get="/", hx_target="body"),
                )
            )
    
    return RedirectResponse("/game", status_code=303)


def _game_over_summary():
    """Final status table for end-of-game screens."""
    if not GS.tp: return ""
    return Card(
        Table(
            Thead(Tr(Th(""), Th("Kingdom"), Th("🍞 Pantry"), Th("Pieces"), Th("Turns"))),
            Tbody(*[Tr(
                Td("👑" if sq.king_alive() else "💀"),
                Td(sq.name, cls="font-medium"),
                Td(str(next((p.place.pantry for p in sq.pieces 
                             if p.type == PieceType.KING and p.place), 0))),
                Td(str(sum(1 for p in sq.pieces if p.place is not None))),
                Td(str(GS.turn)),
            ) for sq in GS.parts.squads]),
        ),
        header=H4(f"Final Standings — Turn {GS.turn}"),
    )


In [ ]:
@rt("/piece/rotate", methods=["POST"])
def piece_rotate(piece_id: str):
    piece = next((p for p in GS.player_squad.pieces if p.id == piece_id), None)
    old_facing = piece.place.facing.label if piece and piece.place else "?"
    if piece:
        GS.rotate_piece(piece)
    new_facing = piece.place.facing.label if piece and piece.place else "?"
    log.info(f"ROTATE piece_id={piece_id} name={piece.name if piece else '?'} "
             f"{old_facing} → {new_facing}")
    
    # Return pieces panel as main swap + map as OOB swap
    map_update = Div(panel_center(), hx_swap_oob="innerHTML:#center-panel")
    return panel_pieces(), map_update


In [ ]:
@rt("/piece/select", methods=["POST"])
def piece_select(piece_id: str):
    piece = next((p for p in GS.player_squad.pieces if p.id == piece_id), None)
    log.info(f"SELECT piece_id={piece_id} found={piece is not None} "
             f"name={piece.name if piece else '?'} "
             f"on_board={piece.place is not None if piece else '?'}")
    if piece:
        GS.select_piece(piece)
    map_update = Div(panel_center(), hx_swap_oob="innerHTML:#center-panel")
    return panel_pieces(), map_update

@rt("/piece/deselect", methods=["POST"])
def piece_deselect(piece_id: str):
    log.info(f"DESELECT piece_id={piece_id}")
    GS.selected_piece = None
    map_update = Div(panel_center(), hx_swap_oob="innerHTML:#center-panel")
    return panel_pieces(), map_update


In [ ]:
@rt("/panel/center")
def panel_center(view: str = "map"):
    tabs = DivLAligned(
        Button("🗺️ Map",
               cls=(ButtonT.primary if view == "map" else ButtonT.ghost, "text-sm"),
               hx_get="/panel/center?view=map", hx_target="#center-panel",
               hx_swap="outerHTML"),
        Button("👥 Squad",
               cls=(ButtonT.primary if view == "squad" else ButtonT.ghost, "text-sm"),
               hx_get="/panel/center?view=squad", hx_target="#center-panel",
               hx_swap="outerHTML"),
        Button("🔗 Flow",
               cls=(ButtonT.primary if view == "flow" else ButtonT.ghost, "text-sm"),
               hx_get="/panel/center?view=flow", hx_target="#center-panel",
               hx_swap="outerHTML"),
        Button("📊 Charts",
               cls=(ButtonT.primary if view == "charts" else ButtonT.ghost, "text-sm"),
               hx_get="/panel/center?view=charts", hx_target="#center-panel",
               hx_swap="outerHTML"),
        cls="gap-1 pb-2",
    )
    
    if view == "squad":
        content = panel_squad_detail()
    elif view == "flow":
        content = panel_flow()
    elif view == "charts":
        content = panel_charts()
    else:
        content = panel_map_content()
    
    return Div(tabs, content, id="center-panel")

In [ ]:
def panel_squad_detail():
    """Squad detail view — reuses squad table + network info."""
    squad = GS.player_squad
    
    # Squad table (reuse existing .table() method)
    table_card = squad.table()
    
    # Network health summary
    G = squad.to_nx()
    U = G.to_undirected(as_view=True) if len(G) > 1 else None
    bottlenecks = list(nx.articulation_points(U)) if U else []
    isolated = squad.isolated_pieces()
    sources = squad.food_sources()
    sinks = squad.food_sinks()
    
    king = next((p for p in squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    
    network_card = Card(
        Grid(
            Div(
                DivLAligned(Span("🔗", cls=TextT.lg), Strong("Connected")),
                P(f"{len(G.nodes)} pieces", cls=TextPresets.muted_sm),
            ),
            Div(
                DivLAligned(Span("⚠️", cls=TextT.lg), Strong("Bottlenecks")),
                P(f"{len(bottlenecks)}", cls=TextPresets.muted_sm),
            ),
            Div(
                DivLAligned(Span("🌾", cls=TextT.lg), Strong("Sources")),
                P(f"{len(sources)} pieces", cls=TextPresets.muted_sm),
            ),
            Div(
                DivLAligned(Span("🍽️", cls=TextT.lg), Strong("Sinks")),
                P(f"{len(sinks)} pieces", cls=TextPresets.muted_sm),
            ),
            cols_sm=2, cols_md=4,
        ),
        header=DivFullySpaced(
            H4("Network Health"),
            Label(f"🍞 {king.place.pantry if king else 0}", cls=LabelT.primary),
        ),
    )
    
    # Isolated pieces warning
    warnings = []
    if isolated:
        names = ", ".join(f"{p.type.icon} {p.name}" for p in isolated)
        warnings.append(P(f"⚠️ Isolated: {names}", cls="text-destructive text-sm"))
    if bottlenecks:
        names = ", ".join(f"{G.nodes[n]['piece'].type.icon} {G.nodes[n]['piece'].name}" 
                          for n in bottlenecks)
        warnings.append(P(f"🔗 Bottlenecks: {names}", cls="text-warning text-sm"))
    
    return Div(network_card, *warnings, table_card, cls="space-y-3")


#### Squads and left panel

I think I have some duplicated functions


In [ ]:
def optimal_hex_info(pt, picks):
    """Render a compact list of top hex picks for a piece type."""
    if not picks:
        return ""
    
    parts = GS.parts
    rows = []
    for rank, (idx, score) in enumerate(picks):
        loc = parts.location_by_index(idx)
        tier = int(parts.fy.tiers[idx])
        level = loc.level
        
        # Quick terrain label
        terrain = parts.terr.elevationLevel(idx) if hasattr(parts.terr, 'elevationLevel') else f"L{level}"
        
        # ROI estimate
        fs = FOOD_TABLE[pt]
        if fs.harvest > 0:
            net = int(fs.harvest * tier) - fs.diet
            roi_str = f"{net:+d}/t"
        else:
            roi_str = f"⚔️{fs.drain}"
        
        rows.append(
            Div(
                DivFullySpaced(
                    DivLAligned(
                        Small(f"#{rank+1}", cls="font-bold w-5"),
                        Small(f"Hex {idx}", cls=TextT.medium),
                    ),
                    DivLAligned(
                        Small(f"🌾{tier}", cls="text-green-600"),
                        Small(f"⛰️{level}"),
                        Small(roi_str, cls="text-amber-600"),
                        cls="gap-1",
                    ),
                ),
                cls="px-2 py-0.5 text-xs bg-muted/30 rounded cursor-pointer hover:bg-muted/60",
                hx_post="/hex_focus",
                hx_vals=json.dumps({"hex_id": idx}),
                hx_target="#center-panel",
            )
        )
    
    return Div(*rows, cls="space-y-0.5 pl-6 pb-1")


In [ ]:
def piece_icon(piece, size=24):
    """Render a tiny inline SVG of a chess piece using flag colors."""
    b = SVGBuilder()
    b.width, b.height = size, size
    scale = size / 45.0
    piece.flag.draw_piece(
        piece.type, MapCord(size/2, size/2), b,
        scale=scale, size='board', layer="icon",
        piece_id=f"ico_{piece.id[:8]}")
    return Div(NotStr(b.xml()), 
               style=f"width:{size}px; height:{size}px; flex-shrink:0; line-height:0;")



In [ ]:
def _commit_label() -> tuple[str, str]:
    """Build commit button label + cls from pending actions."""
    parts = []
    for r in GS.pending_rotations:
        parts.append(f"⟳ {r.piece.type.icon}{r.piece.name}→{r.facing.label}")
    if GS.pending_placement:
        p = GS.pending_placement.piece
        idx = GS.parts.hp2i(GS.pending_placement.location.location)
        tier = int(GS.parts.fy.tiers[idx])
        parts.append(f"{p.type.icon}{p.name} → ⛰️{GS.pending_placement.location.level} 🌾{tier} (−{p.food.cost}🍞)")
    if GS.pending_purchase:
        pp = GS.pending_purchase
        idx = GS.parts.hp2i(pp.location.location)
        tier = int(GS.parts.fy.tiers[idx])
        cost = PURCHASE_COST[pp.piece_type]
        parts.append(f"🛒 {pp.piece_type.icon} → hex {idx} (−{cost}🍞)")
    if parts:
        return " · ".join(parts), ButtonT.primary
    return "— no move —", ButtonT.ghost


def _purchase_preview():
    """Render pending purchase info bar, or empty string."""
    if not GS.pending_purchase:
        return ""
    pp = GS.pending_purchase
    idx = GS.parts.hp2i(pp.location.location)
    tier = int(GS.parts.fy.tiers[idx])
    cost = PURCHASE_COST[pp.piece_type]
    return Div(
        DivFullySpaced(
            DivLAligned(
                Span(pp.piece_type.icon, cls=TextT.lg),
                Small(f"{pp.piece_type.value.title()} → hex {idx}", cls="text-xs font-medium"),
                Small(f"🌾{tier} ⛰️{pp.location.level}", cls=TextPresets.muted_sm),
            ),
            DivLAligned(
                Label(f"−{cost}🍞", cls=LabelT.destructive),
                Button("✕", cls=(ButtonT.ghost, "text-xs px-1"),
                       hx_post="/purchase/cancel", hx_target="#piece-panel"),
            ),
        ),
        cls="px-2 py-1 bg-warning/10 rounded ring-1 ring-warning/30",
    )


def _status_line():
    """Current selection/pending status indicator."""
    if GS.selected_piece:
        p = GS.selected_piece
        return P(f"📍 Place {p.type.icon} {p.name}",
                 cls="text-primary text-xs text-center py-1")
    if GS.pending_placement:
        p = GS.pending_placement.piece
        return P(f"Pending: {p.type.icon} {p.name}",
                 cls="text-warning text-xs text-center py-1")
    return ""


def _piece_row(p: Piece):
    """Single piece row with click action."""
    on_board = p.place is not None
    is_selected = GS.selected_piece is p
    is_pending = GS.pending_placement and GS.pending_placement.piece is p

    # Life remaining
    if on_board and p.type != PieceType.KING:
        remaining = LIFESPAN[p.type] - p.place.tenure
        life_str, life_cls = f"{remaining}t", ("text-destructive" if remaining <= 1 else "text-muted-foreground")
    elif on_board:
        life_str, life_cls = "∞", "text-muted-foreground"
    else:
        life_str, life_cls = "", ""

    facing_str = p.place.facing.label if on_board else ""
    food_str = ""
    if on_board:
        net = int(p.food.harvest * p.place.harvest) - p.food.diet
        food_str = f"{net:+d}"

    # Row style
    if is_selected:     row_cls = "bg-primary/15 ring-1 ring-primary"
    elif is_pending:    row_cls = "bg-warning/15 ring-1 ring-warning"
    elif not on_board:  row_cls = "opacity-60"
    else:               row_cls = "hover:bg-muted/50"

    # Click action
    if on_board:
        attrs = dict(hx_post="/piece/rotate", hx_vals=json.dumps({"piece_id": p.id}))
    elif is_selected:
        attrs = dict(hx_post="/piece/deselect", hx_vals=json.dumps({"piece_id": p.id}))
    else:
        attrs = dict(hx_post="/piece/select", hx_vals=json.dumps({"piece_id": p.id}))

    return Div(
        DivFullySpaced(
            DivLAligned(piece_icon(p, size=22), Span(p.name, cls="text-xs font-medium")),
            DivLAligned(
                Small(food_str, cls=f"w-6 text-right text-xs "
                      f"{'text-green-600' if food_str.startswith('+') else 'text-destructive' if food_str.startswith('-') else ''}"),
                Small(facing_str, cls="w-6 text-center text-xs text-muted-foreground"),
                Small(life_str, cls=f"w-5 text-right text-xs {life_cls}"),
                Small(f"{p.place.pantry}" if on_board else "", cls="w-7 text-right text-xs text-amber-600"),
                cls="gap-1",
            ),
        ),
        cls=f"px-2 py-1 rounded cursor-pointer transition-all {row_cls}",
        **attrs,
    )


def _piece_type_section(pt: PieceType, pieces: list[Piece], optimal: dict):
    """One collapsible section per piece type: rows + optional hex picks."""
    show_optimal = 'optimal' in GS.active_overlays
    has_unplaced = any(p.place is None for p in pieces)

    rows = [_piece_row(p) for p in pieces]
    picks = optimal_hex_info(pt, optimal.get(pt, [])) if show_optimal and has_unplaced else ""

    return Div(
        DivLAligned(
            piece_icon(pieces[0], size=18),
            Small(pt.value.title(), cls="font-medium text-xs"),
            Small(f"({sum(1 for p in pieces if p.place is None)} avail)", cls=TextPresets.muted_sm),
        ),
        *rows,
        picks,
        cls="space-y-0.5",
    )


In [ ]:
def _pantry_bar(pantry: int, max_pantry: int = 150) -> FT:
    """Color-coded pantry bar with danger thresholds."""
    pct = max(0, min(1, pantry / max_pantry))
    if pantry <= 0:
        color, label_cls = "#ef4444", "text-destructive font-bold animate-pulse"
        status = "☠️ ELIMINATED"
    elif pantry < 15:
        color, label_cls = "#ef4444", "text-destructive font-bold"
        status = f"🍞 {pantry} ⚠️ CRITICAL"
    elif pantry < 35:
        color, label_cls = "#f97316", "text-orange-500 font-medium"
        status = f"🍞 {pantry} ⚠️ LOW"
    else:
        color, label_cls = "#22c55e", "text-green-600"
        status = f"🍞 {pantry}"

    return Div(
        DivFullySpaced(
            Strong(status, cls=f"text-sm {label_cls}"),
            Small(f"{int(pct*100)}%", cls="text-muted-foreground text-xs"),
        ),
        Div(
            Div(style=f"width:{pct*100:.0f}%; height:100%; background:{color}; "
                      f"border-radius:4px; transition:width 0.3s ease;"),
            style="width:100%; height:8px; background:#e5e7eb; border-radius:4px; overflow:hidden;",
        ),
        cls="space-y-0.5",
    )


In [ ]:

def _panel_squad(squad: Squad = None, refresh: bool = False):
    if squad is None:
        squad = GS.player_squad
    king = next((p for p in squad.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    pantry = king.place.pantry if king else 0

    optimal = GS.compute_optimal() if 'optimal' in GS.active_overlays else {}
    commit_label, commit_cls = _commit_label()

    by_type = {}
    for p in squad.pieces:
        by_type.setdefault(p.type, []).append(p)

    TYPE_ORDER = [PieceType.KING, PieceType.QUEEN, PieceType.BISHOP,
                  PieceType.KNIGHT, PieceType.PAWN]
    sections = [_piece_type_section(pt, by_type[pt], optimal)
                for pt in TYPE_ORDER if pt in by_type]

    header_row = DivFullySpaced(
        DivLAligned(Small("", cls="w-6"), Small("Name", cls=TextPresets.muted_sm)),
        DivLAligned(
            Small("±", cls="w-6 text-right text-muted-foreground text-xs"),
            Small("Dir", cls="w-6 text-center text-muted-foreground text-xs"),
            Small("T", cls="w-5 text-right text-muted-foreground text-xs"),
            Small("🍞", cls="w-7 text-right text-xs"),
            cls="gap-1",
        ),
        cls="px-2 pb-1 border-b",
    )

    action_row = DivFullySpaced(
        Button("🛒 Buy", cls=(ButtonT.secondary, "text-sm flex-1"),
               hx_post="/purchase", hx_target="#piece-panel",
               disabled=pantry <= 8 or GS.pending_purchase is not None),
        Button(f"✅ {commit_label}", cls=(commit_cls, "text-sm flex-1"),
               hx_post="/commit", hx_target="body",
               disabled=commit_cls == ButtonT.ghost),
        cls="gap-2",
    )

    return Div(
        Div(
            DivFullySpaced(
                Strong(squad.name, cls="text-sm"),
                Button("◀", cls=(ButtonT.ghost, "text-xs px-1"),
                    hx_post="/toggle_panel", hx_target="#left-col"),
            ),
            _pantry_bar(pantry),
            cls="space-y-1",
        ),
        _purchase_preview(),
        action_row,
        _status_line(),
        header_row,
        Div(*sections, cls="space-y-2"),
        id="piece-panel",
        hx_target="#piece-panel",
        cls="space-y-1",
    )

@rt("/panel/pieces")
def panel_pieces():
    return _panel_squad()
    

## Shopping cart

In [ ]:
@rt("/purchase")
def purchase_panel():
    king = next((p for p in GS.player_squad.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    pantry = king.place.pantry if king else 0

    rows = []
    for pt, cost in PURCHASE_COST.items():
        if pt == PieceType.KING: continue
        affordable = pantry - cost >= 8  # keep reserve
        fs = FOOD_TABLE[pt]
        net_str = f"{int(fs.harvest * 5) - fs.diet:+d}/t" if fs.harvest > 0 else f"⚔️{fs.drain}"
        is_selected = GS.pending_purchase_type is pt

        rows.append(Div(
            DivFullySpaced(
                DivLAligned(
                    Span(pt.icon, cls=TextT.lg),
                    Div(
                        Small(pt.value.title(), cls="font-medium text-xs"),
                        Small(f"diet {fs.diet} · {net_str}", cls=TextPresets.muted_sm),
                    ),
                ),
                DivLAligned(
                    Label(f"−{cost}🍞", cls=LabelT.destructive if affordable else LabelT.secondary),
                    Button("Select" if not is_selected else "✓ Selected",
                           cls=(ButtonT.primary if is_selected else ButtonT.secondary, "text-xs"),
                           disabled=not affordable,
                           hx_post="/purchase/select",
                           hx_vals=json.dumps({"piece_type": pt.value}),
                           hx_target="#piece-panel"),
                ),
            ),
            cls=f"px-2 py-1.5 rounded {'bg-primary/10 ring-1 ring-primary' if is_selected else 'hover:bg-muted/50'}",
        ))

    return Div(
        DivFullySpaced(
            H4("Buy a Piece"),
            Button("✕", cls=(ButtonT.ghost, "text-xs px-1"),
                   hx_post="/purchase/cancel", hx_target="#piece-panel"),
        ),
        *rows,
        P("Then click a hex on the map to place it.", cls=TextPresets.muted_sm),
        id="piece-panel",
        cls="space-y-1",
    )


@rt("/purchase/select", methods=["POST"])
def purchase_select(piece_type: str):
    GS.pending_purchase_type = PieceType(piece_type)
    GS.pending_purchase = None  # clear any previous hex pick
    log.info(f"PURCHASE_SELECT type={piece_type}")
    map_update = Div(panel_center(), hx_swap_oob="innerHTML:#center-panel")
    return purchase_panel(), map_update


@rt("/purchase/cancel", methods=["POST"])
def purchase_cancel():
    GS.pending_purchase_type = None
    GS.pending_purchase = None
    return panel_pieces()


In [ ]:
@rt("/toggle_panel", methods=["POST"])
def toggle_panel():
    GS.panel_collapsed = not getattr(GS, 'panel_collapsed', False)
    if GS.panel_collapsed:
        return Div(
            Button("▶", cls=(ButtonT.ghost, "text-xs"),
                   hx_post="/toggle_panel", hx_target="#left-col"),
            id="left-col", cls="w-8 flex-shrink-0",
        )
    return Div(
        Div(hx_get="/panel/pieces", hx_trigger="load",
            hx_target="#piece-panel", id="piece-panel"),
        id="left-col", cls="w-[250px] flex-shrink-0 overflow-y-auto max-h-[80vh]",
    )


In [ ]:
@rt("/piece/focus", methods=["POST"])
def piece_focus(piece_id: str):
    """Switch to map view centered on a specific piece."""
    piece = next((p for p in GS.player_squad.pieces if p.id == piece_id), None)
    if piece and piece.place is not None:
        # Could set a focused_piece on GS to highlight on map
        GS.focused_piece = piece
    return panel_center(view="map")


!cat ../../HexMagic/game/flag.py

In [ ]:
def panel_squad_detail():
    """Rich squad view for the center panel — card grid + network health."""
    squad = GS.player_squad
    king = next((p for p in squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    
    # ── Network health banner ──
    G = squad.to_nx()
    U = G.to_undirected(as_view=True) if len(G) > 1 else None
    bottlenecks = list(nx.articulation_points(U)) if U else []
    isolated = squad.isolated_pieces()
    sources = squad.food_sources()
    sinks = squad.food_sinks()
    placed = [p for p in squad.pieces if p.place is not None]
    
    health_items = [
        ("🔗", "Connected", f"{len(placed)}/{len(squad.pieces)}"),
        ("⚠️", "Bottlenecks", str(len(bottlenecks))),
        ("🌾", "Sources", str(len(sources))),
        ("🍽️", "Sinks", str(len(sinks))),
        ("👑", "Treasury", f"🍞{king.place.pantry}" if king else "—"),
    ]
    health_bar = DivFullySpaced(
        *[DivCentered(
            Span(icon, cls=TextT.lg),
            Small(label, cls=TextPresets.muted_sm),
            Strong(val, cls="text-sm"),
            cls="space-y-0.5",
        ) for icon, label, val in health_items],
        cls="py-3 px-4 bg-muted/30 rounded-lg",
    )
    
    # ── Warnings ──
    warnings = []
    if isolated:
        names = ", ".join(f"{p.type.icon} {p.name}" for p in isolated)
        warnings.append(
            Div(DivLAligned(Span("🚨"), P(f"Isolated: {names}")),
                cls="text-destructive text-sm bg-destructive/10 rounded px-3 py-2"))
    if bottlenecks:
        names = ", ".join(f"{G.nodes[n]['piece'].type.icon} {G.nodes[n]['piece'].name}" 
                          for n in bottlenecks)
        warnings.append(
            Div(DivLAligned(Span("⚠️"), P(f"Bottlenecks: {names}")),
                cls="text-warning text-sm bg-warning/10 rounded px-3 py-2"))
    
    # ── Flow diagram ──
    flow_diagram = SquadFlowDiagram(
        squad=squad, parts=GS.parts,
        width=580, height=350,
        on_click=lambda p: {
            "hx-post": "/piece/focus",
            "hx-vals": json.dumps({"piece_id": p.id}),
            "hx-target": "#center-panel",
        },
    )
    
    # ── Piece cards grid ──
    cards = []
    for p in squad.pieces:
        on_board = p.place is not None
        fs = p.food
        
        # Avatar
        b = SVGBuilder()
        b.width, b.height = 60, 60
        p.draw_avatar(MapCord(30, 30), b, size=56, layer="avatar")
        avatar = Div(NotStr(b.xml()), style="flex-shrink:0; line-height:0;")
        
        # Stats
        if on_board:
            tier = p.place.harvest
            income = int(fs.harvest * tier)
            net = income - fs.diet
            remaining = LIFESPAN[p.type] - p.place.tenure if p.type != PieceType.KING else "∞"
            facing = p.place.facing.label
            
            stats = Div(
                DivFullySpaced(
                    Small("Income"), Small(f"+{income}", cls="text-green-600"),
                ),
                DivFullySpaced(
                    Small("Diet"), Small(f"-{fs.diet}", cls="text-destructive"),
                ),
                DivFullySpaced(
                    Small("Net"), 
                    Small(f"{net:+d}", cls="text-green-600 font-bold" if net >= 0 else "text-destructive font-bold"),
                ),
                DivFullySpaced(
                    Small("Drain"), Small(f"⚔️{fs.drain}"),
                ),
                DivFullySpaced(
                    Small("Pantry"), Small(f"🍞{p.place.pantry}", cls="text-amber-600"),
                ),
                DivFullySpaced(
                    Small("Facing"), Small(facing),
                ),
                DivFullySpaced(
                    Small("Turns left"), Small(str(remaining)),
                ),
                cls="text-xs space-y-0.5",
            )
            
            # Sight coverage count
            sight_count = len(p.sight().hexes)
            location_badge = Label(f"Hex {GS.parts.hp2i(p.place.location)}", cls=LabelT.secondary)
        else:
            stats = Div(
                DivFullySpaced(Small("Cost"), Small(f"🍞{fs.cost}")),
                DivFullySpaced(Small("Diet"), Small(f"{fs.diet}/turn")),
                DivFullySpaced(Small("Harvest"), Small(f"×{fs.harvest}")),
                DivFullySpaced(Small("Drain"), Small(f"⚔️{fs.drain}")),
                DivFullySpaced(Small("Lifespan"), Small(f"{LIFESPAN[p.type]} turns")),
                DivFullySpaced(Small("Sight"), Small(f"+{fs.bonus_sight}" if fs.bonus_sight else "base")),
                cls="text-xs space-y-0.5",
            )
            sight_count = 0
            location_badge = Label("Reserve", cls=LabelT.destructive)
        
        card = Card(
            DivLAligned(avatar, Div(
                DivLAligned(Span(p.type.icon), Strong(p.name)),
                P(f"b. {p.year}", cls=TextPresets.muted_sm),
            )),
            Divider(cls="my-2"),
            stats,
            header=DivFullySpaced(
                Small(p.type.value.title(), cls=TextPresets.muted_sm),
                location_badge,
            ),
            footer=DivFullySpaced(
                Small(f"👁️ {sight_count} hexes" if sight_count else "", cls=TextPresets.muted_sm),
                Button("🔍", cls=(ButtonT.ghost, "text-xs"),
                       hx_post="/piece/focus",
                       hx_vals=json.dumps({"piece_id": p.id}),
                       hx_target="#center-panel") if on_board else "",
            ),
            cls=f"{'ring-1 ring-primary' if on_board else 'opacity-70'}",
        )
        cards.append(card)
    
    return Div(
        health_bar,
        *warnings,
        Grid(*cards, cols_sm=2, cols_md=3, cols_lg=4, cls="gap-3"),
        cls="space-y-3",
    )


#### charts

In [ ]:
CHART_COLS = [
    ('king_pantry',   '🍞 Pantry'),
    ('net_food',      '🌾 Net/turn'),
    ('total_harvest', '🌻 Harvest'),
    ('drain_taken',   '⚔️ Drain in'),
    ('drain_dealt',   '🗡️ Drain out'),
    ('pieces_placed', '♟ Pieces'),
    ('bottlenecks',   '🔗 Bottlenecks'),
    ('sight_coverage','👁️ Sight'),
    ('king_connected','🔗 Connected'),
    ('isolated',      '🏝️ Isolated'),
]

@rt("/toggle_chart_col", methods=["POST"])
def toggle_chart_col(col: str):
    if col in GS.active_chart_cols:
        if len(GS.active_chart_cols) > 1:  # always keep at least one
            GS.active_chart_cols.discard(col)
    else:
        GS.active_chart_cols.add(col)
    return panel_charts()

@rt("/toggle_chart_squad", methods=["POST"])
def toggle_chart_squad(squad: str):
    if squad in GS.active_chart_squads:
        GS.active_chart_squads.discard(squad)
    else:
        GS.active_chart_squads.add(squad)
    return panel_charts()


In [ ]:
def _chart_controls():
    """Two toggle rows: columns and squads."""
    # Row 1: metric columns
    col_toggles = DivLAligned(
        *[Button(label,
            cls=(ButtonT.primary if key in GS.active_chart_cols else ButtonT.ghost, "text-xs"),
            hx_post="/toggle_chart_col",
            hx_vals=json.dumps({"col": key}),
            hx_target="#chart-panel")
          for key, label in CHART_COLS],
        cls="gap-1 flex-wrap",
    )

    # Row 2: squads (flag color dots + name)
    squad_toggles = []
    all_squads = GS.parts.squads if GS.parts else []
    active = GS.active_chart_squads or {s.name for s in all_squads}  # empty = all shown

    for sq in all_squads:
        is_on = sq.name in active or not GS.active_chart_squads
        is_player = sq is GS.player_squad
        label = DivLAligned(
            Span("●", style=f"color:{sq.flag.primary}; font-size:1.1em"),
            Small(sq.name.split()[0], cls="font-medium text-xs" if is_player else "text-xs"),
        )
        squad_toggles.append(
            Button(label,
                cls=(ButtonT.primary if is_on else ButtonT.ghost, "text-xs px-2"),
                hx_post="/toggle_chart_squad",
                hx_vals=json.dumps({"squad": sq.name}),
                hx_target="#chart-panel")
        )

    return Div(
        DivLAligned(*squad_toggles, cls="gap-1 flex-wrap"),
        col_toggles,
        cls="space-y-1 pb-2 border-b",
    )


def panel_charts():
    """Time-series sparklines — controlled by active_chart_cols + active_chart_squads."""
    if GS.tp is None or GS.tp.record.df.empty:
        return Div(
            _chart_controls(),
            P("No history yet — complete a turn.", cls=TextPresets.muted_sm),
            id="chart-panel", cls="space-y-3",
        )

    df = GS.tp.record.df
    all_squads = GS.parts.squads
    
    # Which squads to plot
    active_squads = GS.active_chart_squads or {s.name for s in all_squads}
    colors = {sq.name: sq.flag.primary for sq in all_squads}
    
    cols = sorted(GS.active_chart_cols)
    n = len(cols)
    ncols = min(3, n)
    nrows = math.ceil(n / ncols)
    
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(4.5 * ncols, 3.2 * nrows),
                              facecolor='none', squeeze=False)
    fig.patch.set_alpha(0)

    col_labels = dict(CHART_COLS)

    for i, col in enumerate(cols):
        ax = axes[i // ncols][i % ncols]
        
        for sq in all_squads:
            if sq.name not in active_squads:
                continue
            sub = df[df.squad == sq.name].sort_values('turn')
            if sub.empty or col not in sub.columns:
                continue
            is_player = sq is GS.player_squad
            is_alive = sub.iloc[-1]['alive']
            color = colors[sq.name]
            ax.plot(sub['turn'], sub[col],
                    color=color,
                    lw=2.5 if is_player else 1.2,
                    ls='-' if is_alive else '--',
                    alpha=1.0 if is_player else (0.55 if is_alive else 0.2),
                    marker='o', markersize=3 if is_player else 1.5,
                    label=sq.name.split()[0])
            if is_player and not sub.empty:
                last = sub.iloc[-1]
                ax.annotate(f"{last[col]:.0f}",
                            xy=(last['turn'], last[col]),
                            fontsize=7, color=color,
                            xytext=(4, 0), textcoords='offset points')

        ax.set_title(col_labels.get(col, col), fontsize=10, fontweight='bold')
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.2, lw=0.5)
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_xlabel("Turn", fontsize=7)

    # Hide unused axes
    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].set_visible(False)

    # Shared legend
    handles, labels = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='lower center',
                   ncol=len(active_squads), fontsize=8,
                   framealpha=0.3, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout(rect=[0, 0.06, 1, 1])

    buf = io.BytesIO()
    fig.savefig(buf, format='svg', bbox_inches='tight', transparent=True)
    plt.close(fig)
    buf.seek(0)

    return Div(
        _chart_controls(),
        NotStr(buf.read().decode()),
        id="chart-panel",
        cls="space-y-2",
    )


#### Toggle Overlays

In [ ]:
def RotationTargetOverlay(**kw) -> OverlaySpec:
    """When in rotation mode, highlight 6 adjacent hexes with direction labels."""
    def render(ctx) -> str:
        if GS.rotationClickId is None:
            return ""
        
        grid = ctx.grid
        N = len(grid.hexes)
        parts_list = []
        
        src_idx = GS.rotationClickId
        src_hp = GS.parts.i2hp(src_idx)
        
        # Find the piece being rotated (for its current facing)
        piece = next((p for p in GS.player_squad.pieces
                     if p.place is not None
                     and GS.parts.hp2i(p.place.location) == src_idx), None)
        current_facing = piece.place.facing if piece else None
        
        for d in HexPosition.directions():
            neighbor_hp = src_hp + d
            neighbor_idx = GS.parts.hp2i(neighbor_hp)
            
            for fi in ctx.fine_indices(neighbor_idx):
                if 0 <= fi < N:
                    h = grid.hexes[fi]
                    c = h.center
                    pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
                    
                    # Highlight: brighter for current facing
                    is_current = (d == current_facing)
                    fill_color = "#FFD700" if is_current else "#4a90d9"
                    opacity = 0.5 if is_current else 0.3
                    
                    parts_list.append(
                        f'<polygon points="{pts}" '
                        f'fill="{fill_color}" fill-opacity="{opacity}" '
                        f'stroke="{fill_color}" stroke-width="2" stroke-opacity="0.7"/>')
                    
                    # Direction label
                    fs = max(10, grid.radius * 0.4)
                    parts_list.append(
                        f'<text x="{c.x:.1f}" y="{c.y:.1f}" '
                        f'text-anchor="middle" dominant-baseline="central" '
                        f'font-size="{fs:.0f}" font-weight="bold" '
                        f'fill="white" opacity="0.9">{d.label}</text>')
        
        return '\n'.join(parts_list)
    
    return OverlaySpec("rotation_targets", render, priority=92)


In [ ]:
@rt("/toggle_debug", methods=["POST"])
def toggle_debug():
    GS.debug_overlays = not GS.debug_overlays
    log.info(f"DEBUG overlays={'ON' if GS.debug_overlays else 'OFF'}")
    return panel_center()


In [ ]:
@rt("/zoom", methods=["POST"])
def zoom(radius: int):
    GS.radius = max(10, min(45, radius))  # ← is this being called?
    log.info(f"ZOOM radius={GS.radius}")
    return panel_center()

In [ ]:
@rt("/toggle_overlay", methods=["POST"])
def toggle_overlay(overlay: str):
    log.info(f"TOGGLE overlay={overlay} active={GS.active_overlays}")
    if overlay in GS.active_overlays:
        GS.active_overlays.discard(overlay)
    else:
        GS.active_overlays.add(overlay)
    return panel_center()

In [ ]:
@rt("/piece/focus", methods=["POST"])
def piece_focus(piece_id: str):
    piece = next((p for p in GS.player_squad.pieces if p.id == piece_id), None)
    log.info(f"/piece/focus piece_id={piece_id}")
    if piece and piece.place is not None:
        GS.focused_piece = piece
    return panel_center(view="map")


In [ ]:
@rt("/piece/unfocus", methods=["POST"])
def piece_unfocus():
    GS.focused_piece = None
    return panel_center(view="map")


In [ ]:
def HexNumberOverlay(font_size: int = 0, color: str = "#333", opacity: float = 0.5, **kw) -> OverlaySpec:
    """Label each hex with its grid index."""
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        r = grid.radius
        fs = font_size or max(6, int(r * 0.35))
        parts = []
        
        for i in range(N):
            h = grid.hexes[i]
            c = h.center
            # Map back to coarse index if zoomed
            coarse = ctx.coarse_index(i) if hasattr(ctx, 'coarse_index') else i
            parts.append(
                f'<text x="{c.x:.1f}" y="{c.y:.1f}" '
                f'text-anchor="middle" dominant-baseline="central" '
                f'font-size="{fs}" fill="{color}" opacity="{opacity}" '
                f'font-family="monospace">{coarse}</text>'
            )
        
        return '\n'.join(parts)
    
    return OverlaySpec("hex_numbers", render, priority=95)


In [ ]:
def _build_overlays():
    """Assemble overlay list from GS.active_overlays."""
    if 'terrain' in GS.active_overlays:
        overlays = [TerrainOverlay(), RiverOverlay(max_width=4)]
    else:
        overlays = [CreamOverlay(stylized=True), RiverOverlay(max_width=4)]
    
    OVERLAY_MAP = {
        'food':          FoodOverlay,
        'lanterns':      LanternOverlay,
        'network':       FoodNetworkOverlay,
        'optimal':       lambda: OptimalPlacementOverlay(top_n=3),
        'ocean':         OceanWaveOverlay,
        'number':         HexNumberOverlay,
        'soil':          SoilOverlay,
        'climate':       ClimateOverlay,
        'temperature':   TemperatureOverlay,
        'precipitation': PrecipitationOverlay,
    }
    for key, factory in OVERLAY_MAP.items():
        if key in GS.active_overlays:
            overlays.append(factory())
    
    overlays.append(FogOverlay())
    overlays.append(RotationTargetOverlay())
    overlays.append(PieceOverlay())
    if 'names' in GS.active_overlays:
        overlays.append(SquadNamesOverlay())
    
    return overlays


def _focus_bar():
    """Focus banner + region if a piece is focused, else (None, "")."""
    focused = GS.focused_piece
    if focused and focused.place is not None:
        region = focused.sight()
        bar = DivFullySpaced(
            DivLAligned(
                Span(focused.type.icon),
                Strong(focused.name, cls="text-sm"),
                Small(f"Hex {GS.parts.hp2i(focused.place.location)}",
                      cls=TextPresets.muted_sm),
                Small(f"👁️ {len(region.hexes)} hexes", cls=TextPresets.muted_sm),
            ),
            Button("✕ Full Map", cls=(ButtonT.ghost, "text-xs"),
                   hx_post="/piece/unfocus", hx_target="#center-panel"),
            cls="px-3 py-1 bg-primary/10 rounded-md mb-2",
        )
        return region, bar
    return None, ""


def _map_controls():
    """Zoom slider, overlay toggles, debug button."""
    gameplay_overlays = [
        ("terrain", "🏔️ Elevation"), 
        ("number", "🏔️ places"), ("food", "🌾 Food"),
        ("lanterns", "🔦 Sight"),     ("network", "🔗 Network"),
        ("optimal", "🎯 Optimal"),    ("names", "🏷️ Names"),
    ]
    detail_overlays = [
        ("ocean", "🌊 Ocean"),        ("soil", "🪨 Soil"),
        ("climate", "🌡️ Climate"),    ("temperature", "🌤️ Temp"),
        ("precipitation", "🌧️ Rain"),
    ]
    
    def overlay_row(items):
        return DivLAligned(
            *[Button(
                label,
                cls=(ButtonT.primary if key in GS.active_overlays else ButtonT.ghost, "text-xs"),
                hx_post="/toggle_overlay",
                hx_vals=json.dumps({"overlay": key}),
                hx_target="#center-panel",
            ) for key, label in items],
            cls="gap-1",
        )
    
    zoom_slider = Input(type="range", id="radius", name="radius",
        value=str(GS.radius), min="10", max="45", step="1",
        hx_post="/zoom", hx_trigger="change",
        hx_target="#center-panel", cls="uk-range flex-1")
    
    debug_button = Button(
        f"🐛 {'ON' if GS.debug_overlays else 'OFF'}",
        cls=(ButtonT.primary if GS.debug_overlays else ButtonT.ghost, "text-xs"),
        hx_post="/toggle_debug", hx_target="#center-panel",
    )
    
    return Div(
        DivLAligned(
            Small(f"🔍 {GS.radius}", cls=TextPresets.muted_sm),
            zoom_slider, debug_button, cls="flex-1 gap-2",
        ),
        overlay_row(gameplay_overlays),
        overlay_row(detail_overlays),
        cls="space-y-1 py-2",
    )


In [ ]:
def _placement_status():
    """Status line showing pending selection/placement."""
    if GS.rotationClickId is not None:
        piece = next((p for p in GS.player_squad.pieces
                     if p.place is not None
                     and GS.parts.hp2i(p.place.location) == GS.rotationClickId), None)
        name = f"{piece.type.icon} {piece.name}" if piece else "piece"
        return P(f"⟳ Click a direction to rotate {name}",
                 cls="text-primary font-medium text-center")
    # ... existing selected/pending logic ...
    if GS.selected_piece:
        p = GS.selected_piece
        return P(f"📍 Click map to place {p.type.icon} {p.name}",
                 cls="text-primary font-medium text-center")
    if GS.pending_placement:
        p = GS.pending_placement.piece
        return P(f"Pending: {p.type.icon} {p.name} → hex. Commit or click to undo.",
                 cls="text-amber-600 text-center text-sm")
    return ""


In [ ]:
def panel_map_content():
    """Map view — zoomed to focused piece or full map."""
    overlays = _build_overlays()
    region, focus_bar = _focus_bar()
    
    ctx = GS.parts.overlayContext(
        region=region, padding=2 if region else 0, radius=GS.radius)
    ctx.grid.builder.layers = []
    ctx.grid.adjustRadius(GS.radius)
    TerrainDisplay(*overlays, radius=GS.radius, ctx=ctx, debug=GS.debug_overlays)
    
    map_div = HexTouchMap(
        ctx.grid,
        on_click=HexWrapper.route("/hex_click", "#center-panel"),
        cls="w-full h-full",
    )
    
    return Div(focus_bar, map_div, _placement_status(), _map_controls())

### Panel Flow

In [ ]:
def panel_flow():
    """Flow network diagram — standalone tab."""
    squad = GS.player_squad
    placed = [p for p in squad.pieces if p.place is not None]
    if len(placed) < 2:
        return P("Need at least 2 placed pieces to show flow.", cls=TextPresets.muted_sm)
    
    return SquadFlowDiagram(
        squad=squad, parts=GS.parts,
        width=700, height=450,
        on_click=lambda p: {
            "hx-post": "/piece/focus",
            "hx-vals": json.dumps({"piece_id": p.id}),
            "hx-target": "#center-panel",
        },
    ).__ft__()

## log file

In [ ]:
print('\n'.join(game_log))


The zoom slider doesn't seem to call the zoom route. is it a paramter name thing?

In [ ]:
@rt
def game():
    if not GS.is_active:
        return RedirectResponse("/", status_code=303)
    
    collapsed = getattr(GS, 'panel_collapsed', False)
    
    return Title("HexMagic — Game"), Container(
        DivFullySpaced(
            H3("⬡ HexMagic", cls="uk-h3"),
            Div(
                Span(f"Turn {GS.turn}", cls=TextT.lg),
                Span(f" · {GS.player_squad.name}", cls=TextPresets.muted_sm),
            ),
            Button("🏠 Menu", cls=ButtonT.ghost, hx_get="/", hx_target="body"),
            cls="py-2",
        ),
        Divider(),
        # Use flex instead of grid so left column can collapse
        Div(
            Div(
                Div(hx_get="/panel/pieces", hx_trigger="load",
                    hx_target="#piece-panel", id="piece-panel") if not collapsed
                else Button("▶", cls=(ButtonT.ghost, "text-xs"),
                           hx_post="/toggle_panel", hx_target="#left-col"),
                id="left-col",
                cls="w-8 flex-shrink-0" if collapsed else "w-[250px] flex-shrink-0 overflow-y-auto max-h-[80vh]",
            ),
            Div(id="center-panel", hx_get="/panel/center", hx_trigger="load",
                hx_target="#center-panel", hx_swap="outerHTML",
                cls="flex-1 min-w-0"),
            Div(
                Div(id="history-panel", hx_get="/panel/history", hx_trigger="load",
                    hx_target="#history-panel"),
                debug_log(),
                cls="w-[280px] flex-shrink-0 overflow-y-auto max-h-[80vh] space-y-3",
            ),
            cls="flex gap-3",
        ),
    )


I want to be able to toggle the middle section for the map and something like the matplotlib to show how pieces are doing


In [ ]:
@rt("/debug/ping", methods=["POST"])
def ping():
    log.info("PING received")
    return P("pong", cls="text-green-500")


In [ ]:
webMe(Div(hx_get="/", hx_trigger="load", hx_target="this"))

I am getting errors in commit move

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

There are two bugs visible from the earlier tracebacks:

**1. `Squad` is unhashable — `_apply_drain` crashes on `drained_squads.add(enemy_sq)`**

`@dataclass` sets `__hash__ = None` by default when `eq` is not explicitly set. Fix by tracking names instead:

```python
@patch
def _apply_drain(self: TurnProcessor, squad: Squad, summary: dict):
    placed = [p for p in squad.pieces if p.place is not None]
    drained_squads: dict[str, Squad] = {}  # name → squad
    
    for p in placed:
        cone = p.sight().hexes
        for enemy_sq in self.squads:
            if enemy_sq is squad: continue
            for enemy in enemy_sq.pieces:
                if enemy.place is None: continue
                if self.parts.hp2i(enemy.place.location) in cone:
                    drain = int(p.food.drain)
                    enemy_king = self._king(enemy_sq)
                    if enemy_king:
                        enemy_king.place.pantry -= drain
                        summary['food_drain_dealt'] += drain
                        drained_squads[enemy_sq.name] = enemy_sq

    for enemy_sq in drained_squads.values():
        enemy_king = self._king(enemy_sq)
        if enemy_king and enemy_king.place.pantry <= 0:
            enemy_king.place.pantry = 0
            enemy_sq.kill()
            summary.setdefault('kills', []).append(enemy_sq.name)
```

**2. `setup_game` references `tp` before it exists**

```python
@patch
def setup_game(self: GameParts, n_players: int = 5, year: int = 1950,

In [ ]:
testSquad = GS.parts.squads[0]
for p in testSquad.pieces:
    print(f"{p.name} {p.id}")

In [ ]:
panel_pieces()

In [ ]:
diagram = SquadFlowDiagram(testSquad,GS.parts)

In [ ]:
show(diagram)

In [ ]:
panel_charts()

How did the food pantry drop so quickly?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at the flow diagram and data, a few things compound:

**1. King starts on tier 0** — The start hex has `tier=0`, so the king produces zero harvest but costs 2 diet/turn. For the first few turns, you're bleeding with no income.

**2. Every purchase drains the treasury** — You bought 8 pieces over 8 turns. Even at pawn cost (3), that's ~30 food just in purchase costs, plus the bishop at 9.

**3. Diet scales faster than harvest** — Each new piece adds diet *immediately*, but only earns harvest if it lands on a good tile. Early pawns placed on low-tier hexes are net drains.

**4. `resolve_flow` centralizes everything** — All piece pantries flush to king (those `pantry=0` values in the diagram), so king pantry = total treasury. The +25/turn you see now is the *current* state with 9 producing pieces — it wasn't that good early on.

Rough timeline:
- **Turns 0–3**: spending ~6–10/turn (purchases + growing diet), earning ~0–7/turn → net negative
- **Turns 4–6**: enough pawns on good tiles to break even, but still buying
- **Turn 7–8**: finally net positive (+25), but treasury already eaten down to 7

The fix is the overflow mechanic you mentioned earlier — pieces should keep a small reserve and funnel surplus to king each turn, plus you probably want `best_purchase` to be more conservative about the `RESERVE` threshold (currently 8, should maybe scale with piece count).

So the strategies are overaggressive. Each of the AI expands recklessly and does not take into account that it is starving or budgeting. we need to put into some better constraints

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're right — the AI just spams purchases every turn regardless of economics. We need a **budget gate** before the LP even runs, and strategy should influence spending discipline. Here's the approach:

**1. Add budget awareness to `BoardNeeds`:**

```python
@dataclass
class BoardNeeds:
    food_gap: float
    drain_need: float
    connect_gap: float
    frontier_need: float
    king_threat: float
    budget_health: float  # NEW: >1 means surplus, <1 means deficit

    @classmethod
    def assess(cls, squad: Squad, parts: GameParts, flow: SquadFlow) -> 'BoardNeeds':
        king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        if not king: return cls(0,0,0,0,0,0)
        
        placed = [p for p in squad.pieces if p.place is not None]
        
        income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
        upkeep = sum(p.food.diet for p in placed)
        food_gap = max(0, upkeep - income + 2)
        
        # Budget health: ratio of income to upkeep
        budget_health = income / max(1, upkeep)
        
        current_visible = set()
        for p in placed: current_visible |= p.sight().hexes
        enemy_in_sight = 0
        for sq in parts.squads:
            if sq is squad: continue
            for p in sq.pieces:
                if p.place and parts.hp2i(p.place.location) in current_visible:
                    enemy_in_sight += 1
        
        connect_gap = len(flow.bottlenecks) + len(squad.isolated_pieces())
        
        king_ring = set(parts.grid.indices_in_range(
            parts.hp2i(king.place.location), 6))
        land_ring = {i for i in king_ring if parts.terr.elevations[i] > 0}
        seen = land_ring & current_visible
        frontier_need = 1.0 - (len(seen) / max(1, len(land_ring)))
        
        king_idx = parts.hp2i(king.place.location)
        king_threat = 0.0
        for sq in parts.squads:
            if sq is squad: continue
            for p in sq.pieces:
                if p.place and king_idx in p.sight().hexes:
                    king_threat += p.food.drain
        
        return cls(food_gap, enemy_in_sight, connect_gap, 
                   frontier_need, king_threat, budget_health)
```

**2. Add spending discipline to `PlacementStrategy`:**

```python
@dataclass
class PlacementStrategy:
    name: str
    defend_threat: int = 2
    attack_pantry: int = 60
    attack_placed: int = 4
    knight_roi_mult: float = 1.0
    # NEW: spending discipline
    min_reserve_per_piece: int = 5    # pantry reserve per placed piece
    min_budget_ratio: float = 1.0     # won't buy unless income/upkeep >= this
    max_deficit_turns: int = 2        # skip buying if net negative for N+ turns
    buy_cooldown: int = 0             # min turns between purchases (0=every turn)
    
    def can_afford(self, squad: Squad, parts: GameParts, cost: int) -> bool:
        """Budget gate: should we spend at all?"""
        king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        if not king: return False
        
        placed = [p for p in squad.pieces if p.place is not None]
        pantry = king.place.pantry
        
        # Dynamic reserve: scales with army size
        reserve = len(placed) * self.min_reserve_per_piece
        if pantry - cost < reserve:
            return False
        
        # Budget ratio check: are we earning enough to sustain current army?
        income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
        upkeep = sum(p.food.diet for p in placed)
        if upkeep > 0 and income / upkeep < self.min_budget_ratio:
            return False
        
        # Net food check: don't buy while bleeding
        net = income - upkeep
        if net < 0 and pantry < reserve * 2:
            return False
        
        return True
    
    def assess(self, squad: Squad, parts: GameParts) -> str:
        # ... existing assess logic unchanged ...
        pass


# Updated presets with spending discipline
PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive", defend_threat=3, attack_pantry=80, attack_placed=6,
    min_reserve_per_piece=4, min_budget_ratio=0.8, buy_cooldown=0)

PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3,
    knight_roi_mult=0.7,
    min_reserve_per_piece=3, min_budget_ratio=0.6, buy_cooldown=0)

PlacementStrategy.BALANCED = PlacementStrategy(
    "balanced", defend_threat=2, attack_pantry=60, attack_placed=4,
    min_reserve_per_piece=5, min_budget_ratio=1.0, buy_cooldown=1)

PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle", defend_threat=1, attack_pantry=999, attack_placed=99,
    min_reserve_per_piece=8, min_budget_ratio=1.2, buy_cooldown=2)
```

**3. Wire the gate into `best_purchase`:**

```python
@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy_name: str = "balanced",
                  stance: str = "expand",
                  strategy: PlacementStrategy | None = None) -> Move | None:
    """Two-phase purchase with budget gate."""
    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None

    pantry = king.place.pantry
    
    # Find cheapest affordable type first to check gate
    min_cost = min(PURCHASE_COST.values())
    if strategy and not strategy.can_afford(self, parts, min_cost):
        return None  # budget gate says no
    
    affordable = {pt: cost for pt, cost in PURCHASE_COST.items()
                  if cost <= pantry - (strategy.min_reserve_per_piece * 
                     sum(1 for p in self.pieces if p.place is not None)
                     if strategy else 8)}
    if not affordable: return None

    # ... rest of LP + placement logic unchanged ...
```

**4. Update `next_move` to pass strategy through:**

```python
@patch
def next_move(self: PlacementStrategy, squad: Squad, parts: GameParts) -> Move | None:
    target = getattr(squad, '_target', None)
    if target is not None: target.tick()
    if target is None or target.is_expired:
        squad._target = squad.choose_target(parts, self)
        squad.optimize_facings(parts, max_iters=3)
    else:
        squad.orient_facings(parts)
    
    stance = self.assess(squad, parts)
    return squad.best_purchase(parts, strategy_name=self.name, 
                                stance=stance, strategy=self)
```

The key behavioral changes:

| Before | After |
|---|---|
| Flat `RESERVE = 8` | Scales with army size (`5 × placed`) |
| Buy every turn | Budget ratio gate blocks buying during deficits |
| No income awareness | Won't expand if `income/upkeep < 1.0` |
| Turtle buys freely | Turtle hoards (reserve=8/piece, ratio≥1.2, cooldown=2) |
| Aggressive burns cash | Still loose but has floor (ratio≥0.6) |

The turtle should now stabilize with 2-3 pawns and a fat treasury. The balanced player grows steadily. The aggressive player still pushes but won't buy itself into bankruptcy.

it feels like best_purchase should be in pyomo with constraints for budgets. Is there a model that makes sense. I know the coeffients were just guessed in the exiting function.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great instinct — this is a textbook mixed-integer program. The value of Pyomo isn't speed (you could enumerate 2400 options), it's that **constraints become data, not if-statements**, and you can tune coefficients without touching logic.

Here's the model:

```python
# !pip install pyomo
# Uses GLPK solver (free) — or CBC
# !apt-get install -y glpk-utils  # if needed

from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint,
    Binary, maximize, SolverFactory, value
)

@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy: PlacementStrategy = None,
                  stance: str = "expand") -> Move | None:
    """MIP purchase: Pyomo model with explicit budget constraints."""
    strategy = strategy or PlacementStrategy.BALANCED
    strategy_name = strategy.name
    
    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None
    
    pantry = king.place.pantry
    placed = [p for p in self.pieces if p.place is not None]
    n_placed = len(placed)
    
    # Precompute board state
    terr, tiers = parts.terr, parts.fy.tiers
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    king_hp = king.place.location
    
    current_visible = set()
    for ally in placed: current_visible |= ally.sight().hexes
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    valid_hexes = [idx for idx in current_visible
                   if idx in land_set and idx not in occupied]
    
    if not valid_hexes: return None
    
    # Current economics
    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)
    
    # LP weights from strategy × stance
    flow = SquadFlow.topology_only(self)
    needs = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy_name, PURCHASE_OBJ["balanced"])[stance]
    n_vec = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                      needs.frontier_need, needs.king_threat])
    w_vec = np.array(weights)
    
    # ── Precompute supply scores for every (type, hex, dir) ──
    affordable_types = [pt for pt, cost in PURCHASE_COST.items()
                        if cost <= pantry - 1]  # loose filter; Pyomo enforces budget
    if not affordable_types: return None
    
    options = {}  # (pt_val, hex_idx, dir_idx) -> {score, cost, diet, harvest_net, ...}
    
    for pt in affordable_types:
        fs = FOOD_TABLE[pt]
        cost = PURCHASE_COST[pt]
        lifespan = LIFESPAN.get(pt, 10)
        
        for idx in valid_hexes:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            tier = int(tiers[idx])
            eff_sight = max(2, loc.level) + fs.bonus_sight
            
            for di, d in enumerate(dirs):
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
                
                # Supply vector
                s_food = (fs.harvest * tier - fs.diet) * lifespan - cost
                s_drain = len(enemy_idxs & cone) * fs.drain
                s_connect = (len(ally_idxs & cone)
                             + (2 if parts.hp2i(king_hp) in cone else 0))
                s_frontier = len((cone & land_set) - current_visible)
                s_threat = fs.drain if parts.hp2i(king_hp) in cone else 0
                
                supply = np.array([s_food, s_drain, s_connect, s_frontier, s_threat])
                score = float(n_vec * w_vec @ supply)
                
                harvest_net = int(fs.harvest * tier) - fs.diet
                
                key = (pt.value, idx, di)
                options[key] = dict(
                    score=score,
                    cost=cost,
                    diet=fs.diet,
                    harvest_net=harvest_net,
                    pt=pt, idx=idx, di=di,
                )
    
    if not options: return None
    
    # ── Build Pyomo model ──
    m = ConcreteModel()
    
    m.OPTIONS = Set(initialize=list(options.keys()))
    m.score   = Param(m.OPTIONS, initialize={k: v['score'] for k, v in options.items()})
    m.cost    = Param(m.OPTIONS, initialize={k: v['cost'] for k, v in options.items()})
    m.diet    = Param(m.OPTIONS, initialize={k: v['diet'] for k, v in options.items()})
    m.h_net   = Param(m.OPTIONS, initialize={k: v['harvest_net'] for k, v in options.items()})
    
    # Decision: pick at most one option (or skip)
    m.x = Var(m.OPTIONS, domain=Binary)
    
    # ── Objective: maximize weighted score ──
    m.obj = Objective(
        expr=sum(m.x[k] * m.score[k] for k in m.OPTIONS),
        sense=maximize)
    
    # ── C1: At most one purchase ──
    m.one_purchase = Constraint(
        expr=sum(m.x[k] for k in m.OPTIONS) <= 1)
    
    # ── C2: Budget floor — keep dynamic reserve ──
    reserve = n_placed * strategy.min_reserve_per_piece
    m.budget = Constraint(
        expr=sum(m.x[k] * m.cost[k] for k in m.OPTIONS) <= pantry - reserve)
    
    # ── C3: Sustainability — post-purchase income/upkeep ratio ──
    # income + new_harvest >= min_ratio * (upkeep + new_diet)
    min_ratio = strategy.min_budget_ratio
    m.sustainability = Constraint(
        expr=(income + sum(m.x[k] * m.h_net[k] for k in m.OPTIONS)
              >= min_ratio * upkeep - income))
    # Rearranged: income + harvest_net >= min_ratio * upkeep - income
    # Actually cleaner as: (income + harvest_net) / (upkeep + diet) >= ratio
    # But that's nonlinear, so linearize:
    # income + sum(x*h_net) >= min_ratio * (upkeep + sum(x*diet))
    
    # Let me redo this properly:
    m.del_component(m.sustainability)
    m.sustainability = Constraint(
        expr=(income + sum(m.x[k] * m.h_net[k] for k in m.OPTIONS)
              + sum(m.x[k] * m.diet[k] for k in m.OPTIONS)
              >= min_ratio * (upkeep + sum(m.x[k] * m.diet[k] for k in m.OPTIONS))))
    
    # ── C4: Don't buy if already bleeding and low treasury ──
    if income < upkeep and pantry < reserve * 2:
        # Only allow net-positive placements
        for k in m.OPTIONS:
            if options[k]['harvest_net'] <= 0:
                m.x[k].fix(0)
    
    # ── Solve ──
    solver = SolverFactory('glpk')
    result = solver.solve(m, tee=False)
    
    # ── Extract solution ──
    chosen = None
    for k in m.OPTIONS:
        if value(m.x[k]) > 0.5:
            chosen = options[k]
            break
    
    if chosen is None:
        return None  # solver said skip (all constraints too tight)
    
    pt = chosen['pt']
    idx = chosen['idx']
    facing = dirs[chosen['di']]
    loc = parts.location_by_index(idx)
    
    return Move.from_squad(self, purchase=Purchase(pt, loc, facing))
```

The key constraints:

| Constraint | What it does | Strategy knob |
|---|---|---|
| `one_purchase` | Buy ≤ 1 per turn | — |
| `budget` | `cost ≤ pantry - reserve` | `min_reserve_per_piece` |
| `sustainability` | `(income+harvest)/(upkeep+diet) ≥ ratio` | `min_budget_ratio` |
| `bleeding gate` | Fix non-profitable options to 0 when low | Implicit |

**Why Pyomo over enumerate-and-filter:**
- Adding a new constraint (e.g., "max 3 pawns", "must see king", "min elevation") is one line
- Strategy coefficients and constraint RHS values are *data*, not branching logic
- You can dump `m.pprint()` to debug why a squad isn't buying
- Later you can extend to multi-purchase turns (just change `<= 1` to `<= budget`)

You'll need to install:

```bash
pip install pyomo
apt-get install -y glpk-utils
```

Or use CBC (`apt-get install coinor-cbc`, `SolverFactory('cbc')`), which is faster for larger models.

The sustainability constraint is the important one — it means a turtle with `min_budget_ratio=1.2` literally *cannot* buy a queen (diet=4) unless it already has enough harvest surplus to absorb it. That's the behavioral change you want.

Is the new pyomo method that I just put in going to prevent from over expanding? should I add anything else in?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

> **Note:** Some earlier messages were trimmed from context due to size, but I have full view of the Pyomo model you just added.

The Pyomo model is a big improvement over the enumerate-and-buy approach, but there are a few gaps that will still let AIs over-expand:

**1. The sustainability constraint has a subtle issue** — it's correct algebraically, but it only constrains `income + harvest ≥ ratio × (upkeep + diet)`. It doesn't account for **drain being taken**. A squad that's earning 10/turn but getting drained 8/turn looks "sustainable" but is actually dying. Add:

```python
# Estimate drain taken on current pieces
drain_taken = sum(
    ep.food.drain for sq in parts.squads if sq is not self
    for ep in sq.pieces if ep.place is not None
    for p in placed if parts.hp2i(p.place.location) in ep.sight().hexes)

# C3 revised: income - drain must sustain upkeep
m.sustainability = Constraint(
    expr=(income - drain_taken 
          + sum(m.x[k] * m.h_net[k] for k in m.OPTIONS)
          + sum(m.x[k] * m.diet[k] for k in m.OPTIONS)
          >= min_ratio * (upkeep + sum(m.x[k] * m.diet[k] for k in m.OPTIONS))))
```

**2. `buy_cooldown` is defined but never enforced.** You'd need to track last purchase turn:

```python
# C5: Cooldown — skip if bought too recently
last_buy = getattr(self, '_last_purchase_turn', -999)
current_turn = getattr(parts, '_turn', 0)
if current_turn - last_buy < strategy.buy_cooldown:
    return None
```

**3. No minimum score threshold** — if all options score ≤ 0, the solver will still pick the "least bad" one. Add:

```python
# C6: Minimum value — don't buy garbage
m.min_value = Constraint(
    expr=sum(m.x[k] * m.score[k] for k in m.OPTIONS) >= 0)
```

This means "buying nothing" (all x=0) is always feasible, and the solver will choose it if no option has positive value.

**4. No army size cap** — nothing stops buying 15 pawns. Add a soft cap:

```python
# C7: Army size — diminishing returns
max_pieces = 3 + king.place.pantry // 20  # scales with wealth
m.army_size = Constraint(
    expr=sum(m.x[k] for k in m.OPTIONS) <= max(0, max_pieces - n_placed))
```

When `n_placed >= max_pieces`, this forces all x=0.

**5. Missing: "must be net positive" for conservative strategies** — the bleeding gate only activates when `income < upkeep AND pantry < reserve*2`. A turtle should *always* require profitable purchases:

```python
# C8: Conservative strategies require net positive placements
if strategy.min_budget_ratio >= 1.0:
    for k in m.OPTIONS:
        if options[k]['harvest_net'] <= 0:
            m.x[k].fix(0)
```

Here's a summary of what to add:

| Gap | Fix | Effect |
|---|---|---|
| Ignores drain taken | Include in sustainability | Won't expand while being drained to death |
| Cooldown unused | Check before model | Turtle waits 2 turns between buys |
| Buys negative-value options | `min_value ≥ 0` constraint | "Buy nothing" always valid |
| Unlimited army | Cap based on pantry | Prevents pawn spam |
| Conservative can buy losers | Fix unprofitable to 0 | Turtle/balanced only buy earners |

The `min_value ≥ 0` constraint is probably the single biggest fix — right now even if every option is terrible, the solver picks one.

In [ ]:
oppSquad = GS.parts.squads[1]
#show(oppSquad)

In [ ]:
diagram = SquadFlowDiagram(oppSquad,GS.parts)

In [ ]:
show(diagram)

In [ ]:
# In SquadFlowDiagram._render, near the footer:
king = next((p for p in diagram.squad.pieces 
             if p.type == PieceType.KING and p.place is not None), None)
king_pantry = king.place.pantry if king else 0
# Should match flow.G.nodes[king_id]['pantry'] if the flow is fresh


In [ ]:
??panel_pieces

In [ ]:
king_pantry

In [ ]:

def panel_squad(squad: Squad = None, refresh: bool = False):
    if squad is None:
        squad = GS.player_squad
    king = next((p for p in squad.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    pantry = king.place.pantry if king else 0

    optimal = GS.compute_optimal() if 'optimal' in GS.active_overlays else {}
    commit_label, commit_cls = _commit_label()

    by_type = {}
    for p in squad.pieces:
        by_type.setdefault(p.type, []).append(p)

    TYPE_ORDER = [PieceType.KING, PieceType.QUEEN, PieceType.BISHOP,
                  PieceType.KNIGHT, PieceType.PAWN]
    sections = [_piece_type_section(pt, by_type[pt], optimal)
                for pt in TYPE_ORDER if pt in by_type]

    header_row = DivFullySpaced(
        DivLAligned(Small("", cls="w-6"), Small("Name", cls=TextPresets.muted_sm)),
        DivLAligned(
            Small("±", cls="w-6 text-right text-muted-foreground text-xs"),
            Small("Dir", cls="w-6 text-center text-muted-foreground text-xs"),
            Small("T", cls="w-5 text-right text-muted-foreground text-xs"),
            Small("🍞", cls="w-7 text-right text-xs"),
            cls="gap-1",
        ),
        cls="px-2 pb-1 border-b",
    )

    action_row = DivFullySpaced(
        Button("🛒 Buy", cls=(ButtonT.secondary, "text-sm flex-1"),
               hx_post="/purchase", hx_target="#piece-panel",
               disabled=pantry <= 8 or GS.pending_purchase is not None),
        Button(f"✅ {commit_label}", cls=(commit_cls, "text-sm flex-1"),
               hx_post="/commit", hx_target="body",
               disabled=commit_cls == ButtonT.ghost),
        cls="gap-2",
    )

    return Div(
        Div(
            DivFullySpaced(
                Strong(squad.name, cls="text-sm"),
                Button("◀", cls=(ButtonT.ghost, "text-xs px-1"),
                    hx_post="/toggle_panel", hx_target="#left-col"),
            ),
            _pantry_bar(pantry),
            cls="space-y-1",
        ),
        _purchase_preview(),
        action_row,
        _status_line(),
        header_row,
        Div(*sections, cls="space-y-2"),
        id="piece-panel",
        hx_target="#piece-panel",
        cls="space-y-1",
    )

In [ ]:
panel_squad(oppSquad)

why arent the ai adding pawns?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at the output, 3 of 5 squads are stuck at 1 piece (just king) while only Monstrous Monkeys and Eternal Elephants are buying. The issue is in `can_afford`:

```python
# Budget ratio check: are we earning enough to sustain current army?
income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
upkeep = sum(p.food.diet for p in placed)
if upkeep > 0 and income / upkeep < self.min_budget_ratio:
    return False
```

**When you only have a king on a tier-0 hex:**
- `income = 1.0 * 0 = 0`
- `upkeep = 2`
- `0 / 2 = 0.0 < 0.8` (even the loosest ratio)
- → `can_afford` returns `False` → no purchase ever

The king is stuck on a dead hex, can't earn, and the budget gate won't let it spend its 100 pantry to buy a pawn that *would* earn food.

**Fix: skip the ratio check when there's no income yet (bootstrapping):**

```python
def can_afford(self, squad: Squad, parts: GameParts, cost: int) -> bool:
    king = next((p for p in squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return False
    
    placed = [p for p in squad.pieces if p.place is not None]
    pantry = king.place.pantry
    
    # Dynamic reserve: scales with army size
    reserve = len(placed) * self.min_reserve_per_piece
    if pantry - cost < reserve:
        return False
    
    # Budget ratio check — but skip during bootstrap (only king placed)
    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)
    if len(placed) > 1 and upkeep > 0 and income / upkeep < self.min_budget_ratio:
        return False
    
    # Net food check: don't buy while bleeding (but allow if plenty of reserves)
    net = income - upkeep
    if net < 0 and pantry < reserve * 2 and len(placed) > 1:
        return False
    
    return True
```

The `len(placed) > 1` guard means: "if you only have a king, you're always allowed to buy your first pawn." Once you have 2+ pieces, the ratio kicks in.

Similarly in the Pyomo sustainability constraint, you'd want to relax it early:

```python
# Only enforce sustainability once we have income
if income > 0:
    m.sustainability = Constraint(
        expr=(income - drain_taken
              + sum(m.x[k] * m.h_net[k] for k in m.OPTIONS)
              + sum(m.x[k] * m.diet[k] for k in m.OPTIONS)
              >= min_ratio * (upkeep + sum(m.x[k] * m.diet[k] for k in m.OPTIONS))))
```

That's why only Monkeys and Elephants are buying — they happen to start on hexes with tier 5 and tier 2, so their king actually earns income and passes the ratio check. The other three kings land on tier-0 hexes and get permanently locked out.

we still don't seem to be adding pawns

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

⚠️ Note: the dialog is large and some earlier messages were trimmed from context, so I may be missing some code. Let's debug directly:

```python
# Check each squad's purchase decision step by step
for sq in GS.parts.squads:
    king = next((p for p in sq.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: print(f"{sq.name}: NO KING PLACED"); continue
    
    placed = [p for p in sq.pieces if p.place is not None]
    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)
    pantry = king.place.pantry
    
    strat = getattr(sq, 'strategy', None)
    reserve = len(placed) * (strat.min_reserve_per_piece if strat else 5)
    affordable = {pt: cost for pt, cost in PURCHASE_COST.items() 
                  if pt != PieceType.KING and cost <= pantry - reserve}
    can = strat.can_afford(sq, GS.parts, min(PURCHASE_COST.values())) if strat else "no strategy"
    
    print(f"{sq.name:25s}  pantry={pantry:4d}  income={income:3d}  upkeep={upkeep:3d}"
          f"  placed={len(placed)}  reserve={reserve}  affordable={list(affordable.keys())}  can_afford={can}")
```

And then:

```python
# Try calling best_purchase directly on each squad
for sq in GS.parts.squads:
    strat = getattr(sq, 'strategy', PlacementStrategy.BALANCED)
    move = sq.best_purchase(GS.parts, strategy=strat)
    print(f"{sq.name:25s}  → {move}")
```

This will tell us exactly where it's breaking — likely one of:

1. **`sq.strategy` is `None`** — squads never got a strategy assigned
2. **`valid_hexes` is empty** — the king's sight covers no unoccupied land
3. **Pyomo solver not found** — GLPK not installed, silent failure
4. **`can_afford` still returning False** — the `len(placed) > 1` fix wasn't picked up

Run those two blocks and share the output — we'll see the exact failure point.

In [ ]:
# Check each squad's purchase decision step by step
for sq in GS.parts.squads:
    king = next((p for p in sq.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: print(f"{sq.name}: NO KING PLACED"); continue
    
    placed = [p for p in sq.pieces if p.place is not None]
    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)
    pantry = king.place.pantry
    
    strat = getattr(sq, 'strategy', None)
    reserve = len(placed) * (strat.min_reserve_per_piece if strat else 5)
    affordable = {pt: cost for pt, cost in PURCHASE_COST.items() 
                  if pt != PieceType.KING and cost <= pantry - reserve}
    can = strat.can_afford(sq, GS.parts, min(PURCHASE_COST.values())) if strat else "no strategy"
    
    print(f"{sq.name:25s}  pantry={pantry:4d}  income={income:3d}  upkeep={upkeep:3d}"
          f"  placed={len(placed)}  reserve={reserve}  affordable={list(affordable.keys())}  can_afford={can}")


In [ ]:
# Try calling best_purchase directly on each squad
for sq in GS.parts.squads:
    strat = getattr(sq, 'strategy', PlacementStrategy.BALANCED)
    move = sq.best_purchase(GS.parts, strategy=strat)
    print(f"{sq.name:25s}  → {move}")


In [ ]:
# Try calling best_purchase directly on each squad
for sq in GS.parts.squads:
    strat = getattr(sq, 'strategy', PlacementStrategy.BALANCED)
    move = sq.best_purchase(GS.parts, strategy=strat)
    print(f"{sq.name:25s}  → {move}")


In [ ]:
# Check each squad's purchase decision step by step
for sq in GS.parts.squads:
    king = next((p for p in sq.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: print(f"{sq.name}: NO KING PLACED"); continue
    
    placed = [p for p in sq.pieces if p.place is not None]
    income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep = sum(p.food.diet for p in placed)
    pantry = king.place.pantry
    
    strat = getattr(sq, 'strategy', None)
    reserve = len(placed) * (strat.min_reserve_per_piece if strat else 5)
    affordable = {pt: cost for pt, cost in PURCHASE_COST.items() 
                  if pt != PieceType.KING and cost <= pantry - reserve}
    can = strat.can_afford(sq, GS.parts, min(PURCHASE_COST.values())) if strat else "no strategy"
    
    print(f"{sq.name:25s}  pantry={pantry:4d}  income={income:3d}  upkeep={upkeep:3d}"
          f"  placed={len(placed)}  reserve={reserve}  affordable={list(affordable.keys())}  can_afford={can}")


bugs?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Two clear bugs from the output:

**Bug 1: No squad ever has `.strategy` set** — every squad shows `can_afford=no strategy`. Strategies are never being assigned. You need to assign them at game setup, e.g.:

```python
STRATEGIES = [
    PlacementStrategy.BALANCED,
    PlacementStrategy.AGGRESSIVE,
    PlacementStrategy.TURTLE,
    PlacementStrategy.EXPANSIVE,
    PlacementStrategy.BALANCED,
]

for sq, strat in zip(GS.parts.squads, STRATEGIES):
    sq.strategy = strat
    print(f"{sq.name} → {strat.name}")
```

**Bug 2: Winter Whales and Crimson Crabs return `None`** even with BALANCED strategy — likely `valid_hexes` is empty. Debug:

```python
from contextlib import redirect_stdout
import io

sq = next(s for s in GS.parts.squads if s.name == "Winter Whales")
king = next(p for p in sq.pieces if p.type == PieceType.KING and p.place is not None)
placed = [p for p in sq.pieces if p.place is not None]

current_visible = set()
for p in placed: current_visible |= p.sight().hexes

occupied = {GS.parts.hp2i(p.place.location)
            for s in GS.parts.squads for p in s.pieces if p.place is not None}
land_set = {i for i in range(len(GS.parts.terr.elevations)) 
            if GS.parts.terr.elevations[i] > 0}

valid_hexes = [i for i in current_visible if i in land_set and i not in occupied]

print(f"visible={len(current_visible)}  land={len(land_set)}  occupied={len(occupied)}  valid={len(valid_hexes)}")
```

My bet: the king on Winter Whales has **very short sight** (level=1, no bonus) so it only sees 2-3 hexes, all of which are either water or already occupied. The fix would be to guarantee kings always see at least a ring of 6 valid land hexes, or seed kings on better tiles.

In [ ]:
from contextlib import redirect_stdout
import io

sq = next(s for s in GS.parts.squads if s.name == "Winter Whales")
king = next(p for p in sq.pieces if p.type == PieceType.KING and p.place is not None)
placed = [p for p in sq.pieces if p.place is not None]

current_visible = set()
for p in placed: current_visible |= p.sight().hexes

occupied = {GS.parts.hp2i(p.place.location)
            for s in GS.parts.squads for p in s.pieces if p.place is not None}
land_set = {i for i in range(len(GS.parts.terr.elevations)) 
            if GS.parts.terr.elevations[i] > 0}

valid_hexes = [i for i in current_visible if i in land_set and i not in occupied]

print(f"visible={len(current_visible)}  land={len(land_set)}  occupied={len(occupied)}  valid={len(valid_hexes)}")


it has visible

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good — 12 valid hexes, so it's not a visibility issue. The failure is **inside `best_purchase`** after that point. Most likely the Pyomo solver is failing silently. Add a print trace:

```python
sq = next(s for s in GS.parts.squads if s.name == "Winter Whales")

# Monkey-patch to trace the failure point
king = next(p for p in sq.pieces if p.type == PieceType.KING and p.place is not None)
pantry = king.place.pantry
placed = [p for p in sq.pieces if p.place is not None]

affordable_types = [pt for pt, cost in PURCHASE_COST.items()
                    if pt != PieceType.KING and cost <= pantry - 1]
print(f"affordable_types: {affordable_types}")
print(f"pantry: {pantry}")

# Check if options dict gets populated
# Check if solver is available
from pyomo.environ import SolverFactory
solver = SolverFactory('glpk')
print(f"GLPK available: {solver.available()}")

solver2 = SolverFactory('cbc')
print(f"CBC available: {solver2.available()}")
```

My strong suspicion: **GLPK/CBC aren't installed**, the solver returns an infeasible/error status, and the code falls through to `return None`. 

If that's the case, quickest fix is to use the HiGHS solver which ships with scipy:

```python
# No external solver needed — use scipy.optimize.milp directly
from scipy.optimize import milp, LinearConstraint, Bounds
import numpy as np
```

Or install it:

```bash
!pip install highspy
```

```python
solver = SolverFactory('appsi_highs')
print(f"HiGHS available: {solver.available()}")
```

HiGHS is fast, free, and has Python wheels — no `apt-get` needed.

but is best_purchase still using pyomo?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

⚠️ Note: the dialog was truncated (250 → 52 messages), so I can't see the current implementation of `best_purchase`. Let's just check directly:

```python
??sq.best_purchase
```

Or if that doesn't work:

```python
import inspect
print(inspect.getsource(sq.best_purchase))
```

That'll tell us immediately whether it's the Pyomo version, the old enumerate-and-filter version, or something in between. The fix depends entirely on which one is actually running.

In [ ]:
import inspect
print(inspect.getsource(sq.best_purchase))

wat is wrong?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

It's **not using Pyomo** — it's the old enumerate-and-score version. There are two filters killing all options for Winter Whales:

**Problem 1: Sustainability gate with income already negative**

```python
min_ratio = strategy.min_budget_ratio if income > 0 else 0.0
# Winter Whales: income=1, upkeep=2 → min_ratio = 1.0

# For a pawn on tier-1: harvest_net = 2*1 - 1 = 1
# post_upkeep = 2 + 1 = 3
# income + harvest_net = 1 + 1 = 2 < 1.0 * 3 = 3 → SKIP
```

Since the king is already losing, almost no pawn passes the sustainability gate. It needs a tier-2+ hex to earn enough. Fix:

```python
# Only apply sustainability if currently sustainable
if min_ratio > 0 and upkeep > 0 and income >= upkeep:
    post_upkeep = upkeep + fs.diet
    if income + harvest_net < min_ratio * post_upkeep:
        continue
```

**Problem 2: King threat filter nukes all pawns**

```python
if needs.king_threat > 2.0 and fs.drain == 0:
    continue  # pawns have drain=0 → ALL pawns skipped when threatened
```

Pawns are actually your *best* early defense (they're cheap and block). Fix:

```python
# Only skip non-draining pieces if we already have some pieces placed
if needs.king_threat > 2.0 and fs.drain == 0 and n_placed > 3:
    continue
```

**Also — squads still have no `.strategy`**, so `next_move` always passes `None` → defaults to BALANCED. That's fine for now but means cooldown/etc. never apply. After fixing the two filters above, assign strategies:

```python
for sq, strat in zip(GS.parts.squads, [
    PlacementStrategy.BALANCED,
    PlacementStrategy.AGGRESSIVE, 
    PlacementStrategy.TURTLE,
    PlacementStrategy.EXPANSIVE,
    PlacementStrategy.BALANCED,
]):
    sq.strategy = strat
```

The two filter fixes should unblock Winter Whales and Crimson Crabs immediately.

so how would you write best_purchase?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

⚠️ Note: the dialog was truncated (253 → 52 messages), so I may be missing some context about your data structures.

Based on everything we've debugged, here's a clean rewrite:

```python
@patch
def best_purchase(self: Squad, parts: GameParts,
                  strategy: 'PlacementStrategy' = None,
                  stance: str = "expand") -> Move | None:
    """Purchase: score all (type, hex, facing) options, with sane budget gates."""
    strategy = strategy or PlacementStrategy.BALANCED

    king = next((p for p in self.pieces
                 if p.type == PieceType.KING and p.place is not None), None)
    if not king: return None

    placed      = [p for p in self.pieces if p.place is not None]
    n_placed    = len(placed)
    pantry      = king.place.pantry
    income      = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    upkeep      = sum(p.food.diet for p in placed)
    net         = income - upkeep

    # ── Budget gate ──────────────────────────────────────────────────────────
    reserve = n_placed * strategy.min_reserve_per_piece
    if pantry <= reserve:
        return None  # can't afford anything safely

    # Cooldown gate
    last_buy = getattr(self, '_last_purchase_turn', -999)
    current_turn = getattr(parts, 'turn', 0)
    if current_turn - last_buy < strategy.buy_cooldown:
        return None

    # ── Board state ──────────────────────────────────────────────────────────
    terr, tiers = parts.terr, parts.fy.tiers
    dirs        = HexPosition.directions()
    occupied    = {parts.hp2i(p.place.location)
                   for sq in parts.squads for p in sq.pieces if p.place is not None}
    enemy_idxs  = {parts.hp2i(p.place.location)
                   for sq in parts.squads if sq is not self
                   for p in sq.pieces if p.place is not None}
    ally_idxs   = {parts.hp2i(p.place.location) for p in placed}
    king_idx    = parts.hp2i(king.place.location)

    current_visible: set[int] = set()
    ally_sight_map:  dict[int, int] = {}   # hex → count of allies that see it
    for ally in placed:
        for h in ally.sight().hexes:
            current_visible.add(h)
            ally_sight_map[h] = ally_sight_map.get(h, 0) + 1

    land_set    = {i for i, e in enumerate(terr.elevations) if e > 0}
    valid_hexes = [i for i in current_visible if i in land_set and i not in occupied]
    if not valid_hexes: return None

    # ── Needs vector ─────────────────────────────────────────────────────────
    flow   = SquadFlow.topology_only(self)
    needs  = BoardNeeds.assess(self, parts, flow)
    weights = PURCHASE_OBJ.get(strategy.name, PURCHASE_OBJ["balanced"])[stance]
    n_vec  = np.array([needs.food_gap, needs.drain_need, needs.connect_gap,
                       needs.frontier_need, needs.king_threat])
    w_vec  = np.array(weights)

    # ── Sustainability threshold ──────────────────────────────────────────────
    # Only enforce if we're currently sustainable — don't lock a losing squad out
    # of the one purchase that could save it
    enforce_sustainability = (income >= upkeep) and (n_placed > 1)
    min_ratio = strategy.min_budget_ratio if enforce_sustainability else 0.0

    # ── Score all options ─────────────────────────────────────────────────────
    best_score  = -1e9  # must beat this — ensures "buy nothing" wins if all < 0
    best_option = None

    for pt, fs in FOOD_TABLE.items():
        if pt == PieceType.KING: continue
        cost = PURCHASE_COST[pt]
        if cost > pantry - reserve: continue            # hard budget
        lifespan = LIFESPAN.get(pt, 10)

        for idx in valid_hexes:
            hp   = parts.i2hp(idx)
            loc  = parts.location_by_index(idx)
            tier = int(tiers[idx])
            eff_sight = max(2, loc.level) + fs.bonus_sight
            harvest_net = int(fs.harvest * tier) - fs.diet

            # Sustainability filter — skip if it makes economics worse
            if min_ratio > 0 and upkeep > 0:
                if income + harvest_net < min_ratio * (upkeep + fs.diet):
                    continue

            # Conservative strategies: only buy net-positive pieces
            if strategy.min_budget_ratio >= 1.0 and harvest_net <= 0:
                continue

            # Threat filter: only skip drain=0 if we already have defenders
            if needs.king_threat > 2.0 and fs.drain == 0 and n_placed > 3:
                continue

            for di, d in enumerate(dirs):
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))

                s_food     = (fs.harvest * tier - fs.diet) * lifespan - cost
                s_drain    = len(enemy_idxs & cone) * fs.drain
                s_connect  = (len(ally_idxs & cone)
                              + ally_sight_map.get(idx, 0)
                              + (2 if king_idx in cone else 0))
                s_frontier = len((cone & land_set) - current_visible)
                s_threat   = float(fs.drain if king_idx in cone else 0)

                supply = np.array([s_food, s_drain, s_connect, s_frontier, s_threat])
                score  = float(n_vec * w_vec @ supply)

                if score > best_score:
                    best_score  = score
                    best_option = (pt, idx, di, cost)

    # Require positive value — "buy nothing" always beats a bad purchase
    if best_option is None or best_score <= 0:
        return None

    pt, idx, di, _ = best_option
    self._last_purchase_turn = current_turn
    return Move.from_squad(
        self, purchase=Purchase(pt, parts.location_by_index(idx), dirs[di]))
```

Key differences from what you had:

| Old | New |
|---|---|
| Builds full `options` dict then scans again | Single pass, track `best_score` inline |
| `can_afford` called twice (pre-filter + final) | One clean budget check at top |
| Sustainability blocks losing squads from buying | Only enforces when already sustainable |
| Threat filter kills all pawns early game | Only applies when `n_placed > 3` |
| No minimum score — always buys something | `best_score <= 0` → returns `None` |
| Cooldown defined but never checked | Checked at top |
| `_last_purchase_turn` never set | Set on successful purchase |